# Ozone and temperature in 2025

We use EPA ozone observations and NOAA temperature reports to examine elevated-ozone periods, nearby temperature conditions and gaps in the data. The selected monitor–station pairs are in Alabama and Georgia.

This notebook preserves source files in Bronze, cleans Alabama's measurements and reference metadata, and publishes the source Silver tables as Parquet. It then prepares hourly ozone–temperature matches, calendar coverage and gap summaries. Georgia inputs provide an initial check that the processing functions can be reused.

Period detection and analytical outputs belong to the subsequent Gold processing.

The source formats are described in the [EPA AirData documentation](https://aqs.epa.gov/aqsweb/airdata/FileFormats.html) and the [NOAA GHCNh documentation](https://www.ncei.noaa.gov/oa/global-historical-climatology-network/hourly/doc/ghcnh_DOCUMENTATION.pdf).

## 1. Setup

Run the notebook on the VM with the `spark-env` kernel. It uses Python 3.11 and the Spark 3.5.0 libraries installed under `/opt/spark`.

Sources are listed in `config/sources.json` and storage settings in `config/environments.json`. The defaults are target `vm` and environment `dev`. If several versions of a source are available, select its checksum in `source_versions`.

The setup cell explicitly applies these settings on every run:

- `PIPELINE_REFRESH_SOURCES=false` reuses saved source versions and downloads missing inputs.
- `PIPELINE_RUN_FIXTURES=true` enables the verification tests in `silver_checks.ipynb`.

To change either setting, edit its assignment in the setup cell. These assignments override values inherited from the environment. The companion notebook must be in the same notebook directory when fixtures are enabled.

`PIPELINE_SHOW_DIAGNOSTICS` is read from the environment and defaults to `false`. Setting it to `true` displays additional sample records.

In [1]:
import calendar
import csv
import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time
import shlex
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import Request, urlopen
from urllib.parse import urlsplit
from zipfile import ZipFile

os.environ["PIPELINE_REFRESH_SOURCES"] = "false"
os.environ["PIPELINE_RUN_FIXTURES"] = "true"

# Find the project whether Jupyter starts here or in the notebooks folder.
for folder in (Path.cwd(), *Path.cwd().parents):
    project_dir = folder if folder.name == "final_project" else folder / "final_project"
    if (project_dir / "config" / "sources.json").is_file():
        project_dir = project_dir.resolve()
        break
else:
    raise FileNotFoundError("Start Jupyter from inside the project checkout.")

sys.path.insert(0, str(project_dir / "src"))
from run_context import create_run_context

target = os.environ.get("PIPELINE_TARGET", "vm")
environment = os.environ.get("PIPELINE_ENVIRONMENT", "dev")
refresh_setting = os.environ.get("PIPELINE_REFRESH_SOURCES", "false").lower()
if target not in {"local", "vm"} or refresh_setting not in {"true", "false"}:
    raise ValueError("Use target local/vm and refresh true/false.")
refresh_sources = refresh_setting == "true"

config_name = "environments.local.json" if target == "local" else "environments.json"
environments_file = project_dir / "config" / config_name
sources = json.loads((project_dir / "config" / "sources.json").read_text(encoding="utf-8"))

# Fill this in only if a source has more than one saved version.
source_versions = {}
# Example: source_versions["epa/hourly_ozone_2025"] = "<original SHA-256>"

# Require VM execution for Spark processing.
if target != "vm":
    raise RuntimeError("Run this notebook on the VM with PIPELINE_TARGET=vm.")

if (
    sys.version_info[:2] != (3, 11)
    or Path(sys.prefix) != Path("/home/ubuntu/spark-env")
):
    raise RuntimeError("Select the VM's spark-env kernel, then restart it.")

# Select the installed Spark distribution, Java and Python worker interpreter.
spark_home = Path("/opt/spark")
os.environ["SPARK_HOME"] = str(spark_home)
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["PYSPARK_PYTHON"] = "/home/ubuntu/spark-env/bin/python"

# Search Spark's bundled libraries before the environment's site-packages.
spark_archives = sorted((spark_home / "python" / "lib").glob("*.zip"))
if not spark_archives:
    raise RuntimeError("The installed Spark Python libraries were not found.")

sys.path[:0] = [str(path) for path in spark_archives]

# These imports must come after the library-path setup.
import pyspark
from pyspark.sql import SparkSession, Window, functions as F

if (
    pyspark.__version__ != "3.5.0"
    or not pyspark.__file__.startswith("/opt/spark/")
):
    raise RuntimeError(
        "Expected bundled PySpark 3.5.0. Restart the kernel and run Setup again."
    )

print(f"Target: {target}; environment: {environment}; refresh: {refresh_sources}")
print("Python:", sys.executable, sys.version.split()[0])
print("PySpark:", pyspark.__version__)
print("PySpark location:", pyspark.__file__)

# Enable detailed sample tables when inspecting source results.
diagnostic_setting = os.environ.get("PIPELINE_SHOW_DIAGNOSTICS", "false").lower()
if diagnostic_setting not in {"true", "false"}:
    raise ValueError("PIPELINE_SHOW_DIAGNOSTICS must be true or false.")
show_diagnostics = diagnostic_setting == "true"


Target: vm; environment: dev; refresh: False
Python: /home/ubuntu/spark-env/bin/python 3.11.15
PySpark: 3.5.0
PySpark location: /opt/spark/python/lib/pyspark.zip/pyspark/__init__.py


### Start Spark

Spark uses the configured master and resource settings, UTC timestamps and a warehouse directory outside the repository. The startup check verifies the engine and library versions, the master address, and the result of a small calculation.


In [2]:
# Load the configured Spark master, Python and resource settings.
spark_properties = project_dir / "config" / "spark.properties"
os.environ["PYSPARK_SUBMIT_ARGS"] = shlex.join([
    "--properties-file", str(spark_properties),
    "pyspark-shell",
])

# Give this Silver run an identity and an external staging location.
silver_context = create_run_context(
    environments_file,
    environment,
    test_run_id=os.environ.get("PIPELINE_TEST_RUN_ID"),
)

spark = (
    SparkSession.builder
    .appName(f"DAT535-Silver-{environment}")
    .config("spark.sql.session.timeZone", "UTC")
    .config(
        "spark.sql.warehouse.dir",
        str(silver_context.staging_root / "spark_warehouse"),
    )
    .getOrCreate()
)

try:
    if spark.version != pyspark.__version__:
        raise RuntimeError("Spark engine and PySpark versions differ.")

    if spark.sparkContext.master != "spark://sparkvm-test:7077":
        raise RuntimeError("Spark is not using the intended standalone master.")

    # Independent expectation: 0 + 1 + 2 + 3 + 4 = 10.
    total = spark.range(5).selectExpr("sum(id) AS total").first()["total"]
    if total != 10:
        raise RuntimeError(f"Startup check expected 10, received {total}.")

except Exception:
    spark.stop()
    raise

print("Spark engine:", spark.version)
print("Master:", spark.sparkContext.master)
print("Session timezone:", spark.conf.get("spark.sql.session.timeZone"))
print("Verification total:", total)
print("Silver environment root:", silver_context.environment_root)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 23:11:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark engine: 3.5.0
Master: spark://sparkvm-test:7077
Session timezone: UTC
Verification total: 10
Silver environment root: /home/ubuntu/dat535-final-project-data/dev


### Required source columns

Each source must contain the fields used by the pipeline. Additional columns are allowed and retained. Values are checked during Silver processing.


In [3]:
# Extra columns are fine; these are the ones we need later.
required_columns = {
    ("epa", "sites"): [
        "State Code", "County Code", "Site Number",
        "Latitude", "Longitude", "Datum",
    ],
    ("epa", "monitors"): [
        "State Code", "County Code", "Site Number",
        "Parameter Code", "POC",
        "First Year of Data", "Last Sample Date",
    ],
    ("epa", "hourly_ozone"): [
        "State Code", "County Code", "Site Num", "Parameter Code", "POC",
        "Date GMT", "Time GMT", "Sample Measurement", "Units of Measure",
        "Qualifier", "MDL", "Method Type", "Method Code",
    ],
    ("noaa", "stations"): [
        "GHCN_ID", "LATITUDE", "LONGITUDE", "ELEVATION", "ISO_CODE",
    ],
    ("noaa", "hourly_weather"): [
        "STATION", "DATE", "temperature",
        "temperature_Measurement_Code", "temperature_Quality_Code",
        "temperature_Report_Type", "temperature_Source_Code",
        "temperature_Source_Station_ID",
    ],
}

## 2. Bronze: acquire and preserve inputs

We keep the original source files so that results can be traced back to them. EPA ZIP files are stored with their extracted CSVs. A manifest records the source URL, retrieval details, file sizes, checksums and required-column checks.

### 2.1 File details and column checks

These functions calculate SHA-256 checksums and describe each file for its manifest. Large files are read in chunks to limit memory use.


In [4]:
def file_sha256(path):
    # The ozone CSV is large, so read it a chunk at a time.
    checksum = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(1024 * 1024)
            if not chunk:
                break
            checksum.update(chunk)
    return checksum.hexdigest()


def write_json(path, value):
    """Save to a temporary file first, then replace the JSON file."""
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(value, indent=2) + "\n", encoding="utf-8")
    temporary.replace(path)


def source_name(source):
    name = source["dataset"]
    for field in ("year", "station_id"):
        if field in source:
            name += "_" + str(source[field])
    return name


def check_columns(source, columns):
    required = required_columns[(source["source"], source["dataset"])]
    missing = sorted(set(required) - set(columns))
    return {
        "contract_version": "bronze-columns-v2",
        "observed_columns": columns,
        "status": "blocked" if missing else "compatible",
        "findings": [f"Missing required columns: {missing}"] if missing else [],
    }


def file_record(path, checksum, root):
    """File details for the source manifest."""
    return {
        "path": path.relative_to(root).as_posix(),
        "size_bytes": path.stat().st_size,
        "sha256": checksum,
    }


### Run provenance

Each run records its configuration and, when available, the Git revision and working-tree status. Files run outside a Git checkout are labelled `manual_copy`, with Git information left unknown.


In [5]:
def capture_provenance(context):
    config_path = context.reports_dir / "effective_config.json"
    write_json(config_path, {
        "sources": sources, "target": target, "environment": context.environment,
        "environment_root": str(context.environment_root),
        "refresh_sources": refresh_sources, "source_versions": source_versions,
    })

    # A copied VM folder has no Git history to read, so these values stay unknown.
    provenance = {
        "code_source": "manual_copy",
        "code_revision": None,
        "working_tree_dirty": None,
        "config_sha256": file_sha256(config_path),
        "evidence_path": context.reports_dir.relative_to(context.environment_root).as_posix(),
    }
    in_git_checkout = any(
        (folder / ".git").exists() for folder in (project_dir, *project_dir.parents)
    )
    if in_git_checkout:
        changes = subprocess.check_output(
            ["git", "status", "--porcelain"], cwd=project_dir, text=True
        ).strip()
        provenance.update({
            "code_source": "git",
            "code_revision": subprocess.check_output(
                ["git", "rev-parse", "HEAD"], cwd=project_dir, text=True
            ).strip(),
            "working_tree_dirty": bool(changes),
        })
    return provenance


### 2.2 Download and extract files

Downloads are retried up to three times. Files must be nonempty and match the server's reported size when one is supplied. EPA ZIPs are extracted; NOAA CSV and PSV files are read directly.

Before reusing a saved source, the acquisition step checks the presence, size and SHA-256 checksum of its original file and any extracted files against the manifest. A mismatch stops processing. Saved column checks are reused after file verification.

In [6]:
def download_file(url, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    # Rename the partial file only after the download checks pass.
    partial = destination.with_name(destination.name + ".part")
    for attempt in range(1, 4):
        try:
            request = Request(url, headers={"Accept-Encoding": "identity"})
            with urlopen(request, timeout=60) as response:
                expected_size = response.headers.get("Content-Length")
                resolved_url = response.geturl()
                with partial.open("wb") as handle:
                    shutil.copyfileobj(response, handle, length=1024 * 1024)
            size = partial.stat().st_size
            if size == 0 or (expected_size is not None and size != int(expected_size)):
                raise ValueError("Download is empty or has an unexpected size.")
            partial.rename(destination)
            return {
                "requested_url": url, "resolved_url": resolved_url,
                "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
            }
        except (OSError, ValueError) as error:
            if attempt == 3:
                raise
            print(f"Retry {attempt}: {error}")
            time.sleep(2)


def read_header(path):
    delimiter = "|" if path.suffix == ".psv" else ","
    with path.open(newline="", encoding="utf-8-sig") as handle:
        return next(csv.reader(handle, delimiter=delimiter), [])


def prepare_file(path):
    """Extract the CSV from an EPA ZIP, or return the plain NOAA file."""
    if path.suffix != ".zip":
        return path
    csv_path = path.with_suffix(".csv")
    with ZipFile(path) as archive:
        with archive.open(csv_path.name) as source, csv_path.open("xb") as destination:
            shutil.copyfileobj(source, destination)
    return csv_path


def stored_data_path(manifest, context, verify_hashes=False):
    """Find the saved data file and check that its files are still present."""
    for record in [manifest["original_file"], *manifest["extracted_files"]]:
        path = context.environment_root / record["path"]
        if not path.is_file() or path.stat().st_size != record["size_bytes"]:
            raise ValueError(f"Stored file is missing or its size changed: {path}")
        if verify_hashes and file_sha256(path) != record["sha256"]:
            raise ValueError(f"Stored file checksum differs: {path}")
    if manifest["extracted_files"]:
        data_file = manifest["extracted_files"][0]
    else:
        data_file = manifest["original_file"]
    return context.environment_root / data_file["path"]

### 2.3 Download or reuse a source

A saved version is reused unless refresh is enabled. Its files must match the sizes and checksums recorded in the manifest. New downloads are checked and stored under the original file's checksum.

If the downloaded version already exists, its original manifest is preserved and the extra download is removed after verification. Complete files with missing required columns remain in Bronze but are blocked from further processing.

In [7]:
def acquire_source(source, context, provenance):
    """Find a saved source version or download it, then return its details."""
    name = source_name(source)
    key = source["source"] + "/" + name
    published = context.published_root / "bronze" / source["source"] / name
    version = source_versions.get(key)
    manifest_path = None
    identity = {"provider": source["source"], "dataset": source["dataset"]}
    for field in ("year", "station_id"):
        if field in source:
            identity[field] = source[field]

    if not refresh_sources:
        if version is not None:
            manifest_path = published / version / "manifest.json"
            if not manifest_path.is_file():
                raise FileNotFoundError(
                    f"Selected source version is missing: {manifest_path}"
                )
        else:
            candidates = sorted(published.glob("*/manifest.json"))
            if len(candidates) > 1:
                raise ValueError(
                    f"Select a checksum in source_versions for {key}."
                )
            manifest_path = candidates[0] if candidates else None

    downloaded = manifest_path is None
    acquisition = None
    if not downloaded:
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        data_path = stored_data_path(
            manifest, context, verify_hashes=True
        )
        outcome = "reused"
    else:
        staging = context.staging_root / "bronze" / source["source"] / name
        filename = Path(urlsplit(source["url"]).path).name
        original = staging / filename
        acquisition = download_file(source["url"], original)
        acquisition["run_id"] = context.run_id
        data = prepare_file(original)
        schema = check_columns(source, read_header(data))
        checksum = file_sha256(original)
        data_checksum = checksum if data == original else file_sha256(data)
        snapshot = (
            context.data_root / "bronze" / source["source"] / name / checksum
        )
        reused = snapshot.exists()
        if not reused:
            snapshot.parent.mkdir(parents=True, exist_ok=True)
            staging.rename(snapshot)

        # The manifest points to the saved files, not the staging folder.
        original_record = file_record(
            snapshot / original.name,
            checksum,
            context.environment_root,
        )
        extracted = []
        if data != original:
            extracted_record = file_record(
                snapshot / data.name,
                data_checksum,
                context.environment_root,
            )
            extracted_record["archive_member"] = data.name
            extracted.append(extracted_record)

        candidate = {
            "manifest_version": 1,
            "source": identity,
            "acquisition": acquisition,
            "original_file": original_record,
            "extracted_files": extracted,
            "schema": schema,
            "provenance": provenance,
        }
        if reused:
            stored_data_path(candidate, context, verify_hashes=True)

        manifest_path = published / checksum / "manifest.json"
        if manifest_path.exists():
            # Preserve the original acquisition details.
            manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        else:
            manifest = candidate
            write_json(manifest_path, manifest)

        data_path = snapshot / data.name
        outcome = "reused" if reused else "acquired"

        # Remove this run's extra copies after verifying the saved version.
        if reused:
            if not staging.resolve().is_relative_to(
                context.staging_root.resolve()
            ):
                raise ValueError(
                    "Duplicate cleanup must stay inside this run's staging."
                )
            for path in {original, data}:
                path.unlink()
            if not any(staging.iterdir()):
                staging.rmdir()

    return {
        "request": source,
        "manifest_path": manifest_path,
        "data_path": data_path,
        "schema": manifest["schema"],
        "outcome": outcome,
        "downloaded": downloaded,
        "acquisition": acquisition,
    }

### 2.4 Acquisition and reporting

Sources are processed one at a time. An environment lock prevents concurrent acquisition runs from writing to the same location. The run report records the outcome for each source; a failure stops the notebook.


In [8]:
def run_bronze():
    context = create_run_context(
        environments_file, environment,
        test_run_id=os.environ.get("PIPELINE_TEST_RUN_ID"),
    )
    context.reports_dir.mkdir(parents=True, exist_ok=True)
    started = time.perf_counter()
    report = {
        "run_id": context.run_id, "environment": context.environment,
        "status": "failure", "started_at_utc": datetime.now(timezone.utc).isoformat(),
        "refresh_sources": refresh_sources, "results": [], "blocked_sources": [],
    }
    downloads = []
    lock = context.environment_root / ".bronze-publication.lock"
    locked = False
    print(f"Target: {target}; environment: {environment}; refresh: {refresh_sources}")
    print(f"Run: {context.run_id}")
    try:
        provenance = capture_provenance(context)
        report["provenance"] = provenance
        # Allow only one acquisition run to write to this environment at a time.
        lock.mkdir()
        locked = True
        for source in sources:
            try:
                item = acquire_source(source, context, provenance)
            except Exception as error:
                report["results"].append({
                    "request": source, "outcome": "failed", "error": str(error),
                })
                raise
            downloads.append(item)
            report["results"].append({
                "request": source, "outcome": item["outcome"],
                "downloaded": item["downloaded"],
                "manifest_path": item["manifest_path"].relative_to(
                    context.environment_root
                ).as_posix(),
                "acquisition": item["acquisition"], "error": None,
            })
            if item["schema"]["status"] == "blocked":
                report["blocked_sources"].append(source)
            action = "Downloaded" if item["downloaded"] else "Reused"
            print(f"{action}: {source['source']}/{source_name(source)} "
                  f"| {item['schema']['status']} | {item['data_path']}")
        report["status"] = "success"
    except Exception as error:
        report["error"] = str(error)
        raise
    finally:
        # Save a report even if one of the sources failed.
        if locked:
            lock.rmdir()
        report["finished_at_utc"] = datetime.now(timezone.utc).isoformat()
        report["elapsed_seconds"] = round(time.perf_counter() - started, 3)
        report_path = context.reports_dir / "run.json"
        write_json(report_path, report)
        print(f"Run report: {report_path}")
    return downloads

### 2.5 Acquire selected sources

This call reuses saved inputs and downloads any that are missing. With refresh enabled, it checks the remote sources for new versions. The Bronze report describes acquisition; it does not establish that the later Silver processing succeeded.


In [9]:
downloads = run_bronze()

Target: vm; environment: dev; refresh: False
Run: 11084be819ee4d7081c2db7c0a6417e6
Reused: epa/sites | compatible | /home/ubuntu/dat535-final-project-data/dev/data/bronze/epa/sites/56b322f6a3d5368d055bce53d0988275c0b27fbcced3d57a6ce8f2e2b539e15a/aqs_sites.csv
Reused: epa/monitors | compatible | /home/ubuntu/dat535-final-project-data/dev/data/bronze/epa/monitors/e8325216d4ec5df462a23c201e6ffa1176b2fce363c9f6b0e3f18ce6ad958525/aqs_monitors.csv
Reused: epa/hourly_ozone_2025 | compatible | /home/ubuntu/dat535-final-project-data/dev/data/bronze/epa/hourly_ozone_2025/2b7b9e03b43a2bf20d3c3e78fc8c98b6b349fe2c3c4778382a2997144e7b2fd7/hourly_44201_2025.csv
Reused: noaa/stations | compatible | /home/ubuntu/dat535-final-project-data/dev/data/bronze/noaa/stations/812ea50bd5efd1b9cc09d5800c55825b3d43b3c5aa75ab17b2848f43b1bfaaef/ghcnh-station-list.csv
Reused: noaa/hourly_weather_2025_USW00053864 | compatible | /home/ubuntu/dat535-final-project-data/dev/data/bronze/noaa/hourly_weather_2025_USW00053864

## Input availability

This optional function counts numeric ozone hours and temperature hours that pass the trial source-quality rules, month by month. A report at 14:56 counts towards the 14:00 UTC hour.

The counts include ozone records with excluded qualifiers and do not select an hourly temperature value. They describe preliminary availability. Each call scans the annual ozone CSV.


In [10]:
def check_region(profile, downloads):
    """Count available hours for one monitor and weather station."""
    year = profile["year"]
    monitor = profile["ozone_monitor"]
    monitor_id = "-".join(monitor[field] for field in (
        "state_code", "county_code", "site_number", "parameter_code", "poc"
    ))
    station = profile["weather_mapping"]["noaa_station_id"]
    trial_codes = {
        tuple(pair) for pair in profile["temperature_policy"]["source_quality_pairs"]
    }

    # Find the two inputs selected by this profile.
    ozone = next(
        item for item in downloads
        if item["request"]["source"] == "epa"
        and item["request"]["dataset"] == "hourly_ozone"
        and item["request"].get("year") == year
    )
    weather = next(
        item for item in downloads
        if item["request"]["source"] == "noaa"
        and item["request"]["dataset"] == "hourly_weather"
        and item["request"].get("year") == year
        and item["request"].get("station_id") == station
    )
    for item in (ozone, weather):
        if item["schema"]["status"] != "compatible":
            raise ValueError(f"Source is blocked: {item['data_path']}")

    # Sets stop several reports in the same hour being counted more than once.
    ozone_hours = {month: set() for month in range(1, 13)}
    numeric_temp_hours = {month: set() for month in range(1, 13)}
    trial_temp_hours = {month: set() for month in range(1, 13)}
    temperature_codes = Counter()
    monitor_fields = ["State Code", "County Code", "Site Num", "Parameter Code", "POC"]

    with ozone["data_path"].open(newline="", encoding="utf-8-sig") as handle:
        for row in csv.DictReader(handle):
            if "-".join(row[field].strip() for field in monitor_fields) != monitor_id:
                continue
            stamp = datetime.fromisoformat(row["Date GMT"] + "T" + row["Time GMT"])
            if stamp.year != year:
                continue
            try:
                value = float(row["Sample Measurement"])
            except ValueError:
                continue
            if math.isfinite(value):
                ozone_hours[stamp.month].add(stamp)

    with weather["data_path"].open(newline="", encoding="utf-8-sig") as handle:
        for row in csv.DictReader(handle, delimiter="|"):
            if row["STATION"].strip() != station:
                continue
            stamp = datetime.fromisoformat(row["DATE"])
            if stamp.year != year:
                continue
            source = row["temperature_Source_Code"].strip()
            quality = row["temperature_Quality_Code"].strip()
            report_type = row["temperature_Report_Type"].strip()
            # Show these codes even when the temperature value is missing.
            temperature_codes[(source, quality, report_type)] += 1
            try:
                value = float(row["temperature"])
            except ValueError:
                continue
            if not math.isfinite(value):
                continue
            # Group by clock hour for availability; no temperature value is chosen here.
            hour = stamp.replace(minute=0, second=0, microsecond=0)
            numeric_temp_hours[stamp.month].add(hour)
            if (source, quality) in trial_codes:
                trial_temp_hours[stamp.month].add(hour)

    # Include months without observations, so gaps stay visible.
    monthly = []
    for month in range(1, 13):
        ozone_count = len(ozone_hours[month])
        shared = len(ozone_hours[month] & trial_temp_hours[month])
        monthly.append({
            "month": month,
            "calendar_hours": calendar.monthrange(year, month)[1] * 24,
            "ozone_hours": ozone_count,
            "numeric_temperature_hours": len(numeric_temp_hours[month]),
            "trial_temperature_hours": len(trial_temp_hours[month]),
            "shared_hours": shared,
            "ozone_without_temperature": ozone_count - shared,
            "overlap_percent": 100 * shared / ozone_count if ozone_count else None,
        })

    print(f"{profile['region']} {year} ({profile['status']})")
    print(f"Ozone monitor: {monitor_id}; weather station: {station}")
    print("\nTemperature source | Quality | Report type | Reports")
    for codes, count in temperature_codes.most_common():
        print(" | ".join(code or "<blank>" for code in codes), "|", count)

    print("\nMonth | Calendar | Ozone | Numeric temp | Trial temp | Shared | Without temp | Overlap")
    for row in monthly:
        percent = row["overlap_percent"]
        percent_text = f"{percent:.1f}%" if percent is not None else "n/a"
        print(f"{row['month']:02d} | {row['calendar_hours']} | {row['ozone_hours']} | "
              f"{row['numeric_temperature_hours']} | {row['trial_temperature_hours']} | "
              f"{row['shared_hours']} | {row['ozone_without_temperature']} | {percent_text}")
    total_ozone = sum(row["ozone_hours"] for row in monthly)
    total_shared = sum(row["shared_hours"] for row in monthly)
    total_missing = sum(row["ozone_without_temperature"] for row in monthly)
    print(f"\nTotals: {total_ozone} ozone hours; {total_shared} shared hours; "
          f"{total_missing} without temperature")

    return {
        "monthly": monthly,
        "temperature_codes": dict(temperature_codes),
        "source_manifests": {
            "ozone": ozone["manifest_path"], "weather": weather["manifest_path"],
        },
    }


## 3. Alabama: McAdory and Alabaster

The McAdory ozone monitor is paired with the Alabaster weather station, approximately 26.7 km away. The station provides nearby weather context. The profile records the initial provisional selection; section 9.10 documents the subsequent suitability decision using cleaned observations and coverage results.

The inspected 2025 input contained 5,166 numeric ozone hours, of which 4,931 had temperature passing the trial quality rules. This included 95 ozone records with qualifier `6` in February and March. January and December had no ozone observations in that file. The cleaning results below account for the qualifier exclusions.

### 3.1 Load the profile

The profile specifies the monitor, weather station, year and temperature quality policy.

In [11]:
alabama_profile = json.loads(
    (project_dir / "config" / "alabama_profile.json").read_text(encoding="utf-8")
)


## 4. Georgia: South DeKalb and Atlanta

The South DeKalb ozone monitor is paired with Atlanta Hartsfield-Jackson weather station, approximately 15.5 km away. The pairing remains provisional.

The inspected 2025 input contained 8,526 numeric ozone hours and 8,404 shared trial temperature hours. Most of the 122 unmatched hours were in August and September. FM12 reports were present, and quality code `7` was excluded by the trial rules. These preliminary counts do not establish cleaned Silver coverage.

### 4.1 Load the profile

The Georgia profile has the same structure as Alabama's, with different monitor and station identifiers.


In [12]:
georgia_profile = json.loads(
    (project_dir / "config" / "georgia_profile.json").read_text(encoding="utf-8")
)


## 5. Silver: ozone observations

### 5.1 Read source files and check fields

The reader checks headers, field counts and quotation handling, requiring one record per physical line. It initially keeps all columns as strings. The accompanying functions identify missing strings and finite numbers.


In [13]:
def read_validated_delimited_file(path, delimiter):
    """Check delimited text and return string columns with scan metadata."""
    path = Path(path).resolve(strict=True)
    record_count = 0

    with path.open(encoding="utf-8-sig", newline="") as handle:
        reader = csv.reader(
            handle,
            delimiter=delimiter,
            quotechar='"',
            doublequote=True,
            escapechar=None,
            skipinitialspace=False,
            strict=True,
        )

        try:
            columns = next(reader, None)

            if not columns or any(
                not name or name != name.strip()
                for name in columns
            ):
                raise ValueError(
                    f"{path.name}: missing, blank or untrimmed column names"
                )

            if len({name.casefold() for name in columns}) != len(columns):
                raise ValueError(f"{path.name}: duplicate column names")

            if reader.line_num != 1:
                raise ValueError(
                    f"{path.name}: multiline headers are unsupported"
                )

            previous_line = reader.line_num

            for values in reader:
                if reader.line_num != previous_line + 1:
                    raise ValueError(
                        f"{path.name}: multiline record ending at "
                        f"line {reader.line_num} is unsupported"
                    )

                if len(values) != len(columns):
                    raise ValueError(
                        f"{path.name}: line {reader.line_num}: "
                        f"expected {len(columns)} fields, "
                        f"found {len(values)}"
                    )

                record_count += 1
                previous_line = reader.line_num

        except csv.Error as error:
            raise ValueError(
                f"{path.name}: CSV parsing failed near "
                f"line {reader.line_num}: {error}"
            ) from error

    # Preserve source fields as strings until explicit type conversion.
    schema = pyspark.sql.types.StructType()
    for name in columns:
        schema.add(name, "string", nullable=True)

    records = (
        spark.read
        .schema(schema)
        .option("header", True)
        .option("enforceSchema", False)
        .option("encoding", "UTF-8")
        .option("sep", delimiter)
        .option("quote", '"')
        .option("escape", '"')
        .option("unescapedQuoteHandling", "RAISE_ERROR")
        .option("multiLine", False)
        .option("ignoreLeadingWhiteSpace", False)
        .option("ignoreTrailingWhiteSpace", False)
        .option("mode", "FAILFAST")
        .csv(str(path))
    )

    return records, {
        "columns": columns,
        "records": record_count,
    }


def read_validated_ozone_csv(path):
    """Read ozone CSV using the shared structural checks."""
    return read_validated_delimited_file(path, delimiter=",")


def missing_string(column_name):
    """Identify null, empty or space-only source values."""
    value = F.col(column_name)
    return value.isNull() | (F.trim(value) == "")


def finite_number(column_name):
    """Identify numeric values excluding null, NaN and infinity."""
    value = F.col(column_name)
    return (
        value.isNotNull()
        & ~F.isnan(value)
        & (F.abs(value) != float("inf"))
    )


### 5.2 Parse and classify ozone fields

GMT date and time fields become UTC timestamps. Ozone and the reported detection limit, MDL, become numeric values. Original fields are retained, and each record is checked against the identifier, time, unit, measurement and qualifier rules.


In [14]:
def type_ozone_fields(records):
    """Add typed fields while retaining the source columns."""
    if records.sparkSession.conf.get("spark.sql.session.timeZone") != "UTC":
        raise ValueError("Ozone timestamp parsing requires a UTC Spark session.")

    sample_time = F.concat(
        F.col("Date GMT"), F.lit(" "), F.col("Time GMT")
    )

    return (
        records
        .withColumn(
            "timestamp_utc",
            F.try_to_timestamp(sample_time, F.lit("yyyy-MM-dd HH:mm")),
        )
        .withColumn(
            "ozone_value",
            F.expr("try_cast(`Sample Measurement` AS DOUBLE)"),
        )
        .withColumn(
            "mdl_value",
            F.expr("try_cast(`MDL` AS DOUBLE)"),
        )
    )


missing_identifier = F.lit(False)
for column_name in (
    "State Code", "County Code", "Site Num", "Parameter Code", "POC"
):
    missing_identifier = missing_identifier | missing_string(column_name)

missing_time = missing_string("Date GMT") | missing_string("Time GMT")
missing_units = missing_string("Units of Measure")

ozone_field_checks = {
    "missing_identifier": missing_identifier,
    "missing_timestamp": missing_time,
    "invalid_timestamp": ~missing_time & F.col("timestamp_utc").isNull(),
    "missing_units": missing_units,
    "unsupported_units": (
        ~missing_units
        & (F.col("Units of Measure") != "Parts per million")
    ),
}

for label, source_column, typed_column in (
    ("ozone", "Sample Measurement", "ozone_value"),
    ("mdl", "MDL", "mdl_value"),
):
    missing = missing_string(source_column)
    value = F.col(typed_column)
    finite = finite_number(typed_column)

    ozone_field_checks.update({
        f"missing_{label}": missing,
        f"invalid_{label}": ~missing & value.isNull(),
        f"nonfinite_{label}": value.isNotNull() & ~finite,
        f"negative_{label}": finite & (value < 0),
    })


def classify_ozone_fields(records, year):
    """Classify checked ozone records using ordered field rules."""
    allowed_qualifiers = ["", "XS", "MD", "1V", "V"]
    qualifier = F.coalesce(F.trim(F.col("Qualifier")), F.lit(""))
    timestamp = F.col("timestamp_utc")
    value = F.col("ozone_value")

    rules = [
        ("missing_identifier", F.col("missing_identifier")),
        ("unusable_timestamp", timestamp.isNull()),
        (
            "outside_year",
            timestamp.isNotNull() & (F.year(timestamp) != year),
        ),
        (
            "off_hour",
            timestamp.isNotNull()
            & (timestamp != F.date_trunc("hour", timestamp)),
        ),
        (
            "unusable_units",
            missing_string("Units of Measure")
            | (F.col("Units of Measure") != "Parts per million"),
        ),
        ("unusable_ozone", ~finite_number("ozone_value")),
        (
            "below_minimum",
            finite_number("ozone_value") & (value < -0.004),
        ),
        ("qualifier_6", qualifier == "6"),
        (
            "unsupported_qualifier",
            ~qualifier.isin(allowed_qualifiers + ["6"]),
        ),
    ]

    reasons = F.filter(
        F.array(*[
            F.when(condition, F.lit(reason))
            for reason, condition in rules
        ]),
        lambda reason: reason.isNotNull(),
    )

    return (
        records
        .withColumn("qualifier_code", qualifier)
        .withColumn("field_reasons", reasons)
        .withColumn("field_eligible", F.size("field_reasons") == 0)
        .withColumn(
            "field_disposition",
            F.when(F.col("field_eligible"), F.lit("eligible"))
            .otherwise(F.element_at("field_reasons", 1)),
        )
    )


### 5.3 Track source records and resolve duplicate keys

Each record receives its Bronze version, file checksum and a fingerprint of its original parsed fields. Exact copies are counted. Differing rows with the same usable monitor-timestamp key are quarantined within their source file and version.


In [15]:
ozone_key_columns = [
    "State Code",
    "County Code",
    "Site Num",
    "Parameter Code",
    "POC",
    "timestamp_utc",
]


def attach_bronze_lineage(records, source, environment_root, source_columns):
    """Attach manifest references and a source-row content fingerprint."""
    root = Path(environment_root).resolve()
    manifest_path = Path(source["manifest_path"]).resolve()
    data_path = Path(source["data_path"]).resolve()
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))

    if manifest["manifest_version"] != 1:
        raise ValueError("Unsupported Bronze manifest version.")

    relative_data_path = data_path.relative_to(root).as_posix()
    file_entries = [
        manifest["original_file"],
        *manifest["extracted_files"],
    ]
    matching_files = [
        entry for entry in file_entries
        if entry["path"] == relative_data_path
    ]

    if len(matching_files) != 1:
        raise ValueError("Selected data file must match one manifest entry.")

    source_version = manifest["original_file"]["sha256"]
    if manifest_path.parent.name != source_version:
        raise ValueError("Manifest directory does not match its source version.")

    metadata = {
        "source_provider": manifest["source"]["provider"],
        "source_dataset": manifest["source"]["dataset"],
        "source_version_sha256": source_version,
        "source_file_path": relative_data_path,
        "source_file_sha256": matching_files[0]["sha256"],
        "source_manifest_path": manifest_path.relative_to(root).as_posix(),
        "source_record_hash_scheme": "sha256-json-raw-fields-v1",
    }

    # Sort field names and include nulls for consistent serialization.
    source_record_json = F.to_json(
        F.struct(*[F.col(name) for name in sorted(source_columns)]),
        options={"ignoreNullFields": "false"},
    )

    return records.select(
        "*",
        *[F.lit(value).alias(name) for name, value in metadata.items()],
        F.sha2(source_record_json, 256).alias("source_record_sha256"),
    )


def resolve_source_keys(records, key_columns, key_is_usable):
    """Count exact copies and quarantine conflicting source-key variants."""
    source_scope = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
    ]

    # Input columns contain source fields, stable lineage and field results.
    distinct_records = records.groupBy(*records.columns).agg(
        F.count("*").alias("source_occurrences")
    )

    key_window = Window.partitionBy(*(source_scope + key_columns))
    usable_key = F.coalesce(key_is_usable, F.lit(False))

    return (
        distinct_records
        .withColumn(
            "duplicate_extra_records",
            F.col("source_occurrences") - 1,
        )
        .withColumn(
            "key_variant_count",
            F.when(
                usable_key,
                F.count(F.lit(1)).over(key_window),
            ),
        )
        .withColumn(
            "key_conflict",
            F.coalesce(F.col("key_variant_count") > 1, F.lit(False)),
        )
        .withColumn(
            "record_disposition",
            F.when(F.col("key_conflict"), F.lit("conflicting_key"))
            .otherwise(F.col("field_disposition")),
        )
        .withColumn(
            "record_reasons",
            F.when(
                F.col("key_conflict"),
                F.concat(
                    F.array(F.lit("conflicting_key")),
                    F.col("field_reasons"),
                ),
            ).otherwise(F.col("field_reasons")),
        )
        .withColumn(
            "record_eligible",
            F.col("record_disposition") == "eligible",
        )
    )


### 5.4 Ozone observation table

The observation table contains eligible ozone measurements, their original values and codes, quality flags and source references. Unusable optional MDL values become null, with their raw values and flags retained.


In [16]:
def build_ozone_observations(records):
    """Build the eligible ozone observation schema."""
    identifier_columns = {
        "State Code": "state_code",
        "County Code": "county_code",
        "Site Num": "site_number",
        "Parameter Code": "parameter_code",
        "POC": "poc",
    }

    mdl_usable = (
        finite_number("mdl_value")
        & (F.col("mdl_value") >= 0)
    )

    flag_rules = {
        "negative_ozone": F.col("negative_ozone"),
        "below_reported_mdl": (
            mdl_usable & (F.col("ozone_value") < F.col("mdl_value"))
        ),
        "source_qualifier_XS": F.col("qualifier_code") == "XS",
        "source_qualifier_MD": F.col("qualifier_code") == "MD",
        "source_validation_qualifier": (
            F.col("qualifier_code").isin("1V", "V")
        ),
        "missing_mdl": F.col("missing_mdl"),
        "invalid_mdl": F.col("invalid_mdl"),
        "nonfinite_mdl": F.col("nonfinite_mdl"),
        "negative_mdl": F.col("negative_mdl"),
    }

    quality_flags = F.filter(
        F.array(*[
            F.when(condition, F.lit(name))
            for name, condition in flag_rules.items()
        ]),
        lambda flag: flag.isNotNull(),
    )

    lineage_columns = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
        "source_manifest_path",
        "source_record_hash_scheme",
        "source_record_sha256",
    ]

    return records.filter(F.col("record_eligible")).select(
        *[
            F.col(source_name).alias(output_name)
            for source_name, output_name in identifier_columns.items()
        ],
        F.concat_ws(
            "-", *[F.col(name) for name in identifier_columns]
        ).alias("monitor_id"),
        "timestamp_utc",
        F.col("ozone_value").alias("ozone_ppm"),
        F.col("Sample Measurement").alias("ozone_raw"),
        F.col("Units of Measure").alias("source_unit"),
        F.when(mdl_usable, F.col("mdl_value")).alias("mdl_ppm"),
        F.col("MDL").alias("mdl_raw"),
        "qualifier_code",
        F.col("Qualifier").alias("qualifier_raw"),
        F.col("Method Type").alias("method_type"),
        F.col("Method Code").alias("method_code"),
        F.col("Method Name").alias("method_name"),
        quality_flags.alias("quality_flags"),
        "source_occurrences",
        *lineage_columns,
    )


### 5.5 Ozone audit table

The audit explains what happened to every source record, including excluded records, conflicting rows and extra copies. Each entry has one primary outcome, or disposition, and may contain several reasons.


In [17]:
def named_column_map(column_names):
    """Map column names to their values."""
    entries = []
    for name in column_names:
        entries.extend([F.lit(name), F.col(name)])
    return F.create_map(*entries)


def build_ozone_audit(records, source_columns, diagnostic_columns):
    """Build counted audit entries for representatives and duplicate copies."""
    monitor_columns = [
        "State Code", "County Code", "Site Num", "Parameter Code", "POC"
    ]
    lineage_columns = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
        "source_manifest_path",
        "source_record_hash_scheme",
        "source_record_sha256",
    ]

    base = (
        records
        .withColumn(
            "monitor_id",
            F.when(
                ~F.col("missing_identifier"),
                F.concat_ws("-", *[F.col(name) for name in monitor_columns]),
            ),
        )
        .withColumn("raw_fields", named_column_map(source_columns))
        .withColumn("field_flags", named_column_map(diagnostic_columns))
    )

    common_columns = [
        "monitor_id",
        "timestamp_utc",
        "raw_fields",
        "field_flags",
        "field_disposition",
        "field_reasons",
        "key_variant_count",
        "key_conflict",
        *lineage_columns,
    ]

    representatives = base.select(
        *common_columns,
        F.col("record_disposition").alias("disposition"),
        F.col("record_reasons").alias("reasons"),
        F.lit(1).cast("long").alias("record_count"),
    )

    duplicate_copies = (
        base
        .filter(F.col("duplicate_extra_records") > 0)
        .select(
            *common_columns,
            F.lit("exact_duplicate").alias("disposition"),
            F.concat(
                F.array(F.lit("exact_duplicate")),
                F.col("record_reasons"),
            ).alias("reasons"),
            F.col("duplicate_extra_records").alias("record_count"),
        )
    )

    return representatives.unionByName(duplicate_copies)


### 5.6 Quality summary

The summary compares record counts, value ranges, missing MDL values and time bounds before and after cleaning. Primary dispositions account for source records. Quality flags can overlap and are reported separately.


In [18]:
def summarize_ozone_values(records, ozone_column, mdl_column):
    ozone = F.col(ozone_column)
    ozone_is_finite = finite_number(ozone_column)
    mdl_is_usable = (
        finite_number(mdl_column)
        & (F.col(mdl_column) >= 0)
    )

    return records.agg(
        F.count("*").alias("records"),
        F.count(
            F.when(ozone_is_finite & (ozone < 0), 1)
        ).alias("negative_ozone"),
        F.count(
            F.when(~mdl_is_usable, 1)
        ).alias("missing_or_unusable_mdl"),
        F.min(
            F.when(ozone_is_finite, ozone)
        ).alias("minimum_ozone_ppm"),
        F.max(
            F.when(ozone_is_finite, ozone)
        ).alias("maximum_ozone_ppm"),
        F.date_format(
            F.min("timestamp_utc"), "yyyy-MM-dd HH:mm:ss"
        ).alias("earliest_timestamp_utc"),
        F.date_format(
            F.max("timestamp_utc"), "yyyy-MM-dd HH:mm:ss"
        ).alias("latest_timestamp_utc"),
    ).first().asDict()


def show_source_summary(summary, metrics):
    """Display existing aggregate results without reprocessing source records."""
    rows = [
        (
            label,
            str(summary["before"][key])
            if summary["before"][key] is not None else "N/A",
            str(summary["after"][key])
            if summary["after"][key] is not None else "N/A",
        )
        for label, key in metrics
    ]
    spark.createDataFrame(
        rows,
        schema="metric STRING, before_cleaning STRING, after_cleaning STRING",
    ).show(truncate=False)
    print("Exclusive source-record dispositions:")
    for disposition, count in sorted(summary["dispositions"].items()):
        print(f"  {disposition}: {count}")
    print("Reconciled source records:", sum(summary["dispositions"].values()))
    print("Overlapping quality flags on retained observations:")
    if summary["quality_flags"]:
        for flag, count in sorted(summary["quality_flags"].items()):
            print(f"  {flag}: {count}")
    else:
        print("  None")


### 5.7 Ozone processing function

This function checks the selected source against the profile, applies the cleaning rules, and returns observations, audit records and a summary. It verifies that the audit accounts for all selected records and agrees with the eligible-observation count.


In [19]:
def process_ozone_source(source, profile, environment_root):
    """Build ozone observations, audit records and quality summaries."""
    year = profile["year"]
    monitor = profile["ozone_monitor"]
    monitor_fields = {
        "State Code": "state_code",
        "County Code": "county_code",
        "Site Num": "site_number",
        "Parameter Code": "parameter_code",
        "POC": "poc",
    }

    if type(year) is not int:
        raise ValueError("The profile year must be an integer")

    monitor_values = {
        column: monitor.get(profile_field)
        for column, profile_field in monitor_fields.items()
    }
    if any(
        not isinstance(value, str)
        or not value
        or value != value.strip()
        for value in monitor_values.values()
    ):
        raise ValueError("Monitor identifiers must be nonempty, trimmed strings")

    if monitor_values["Parameter Code"] != "44201":
        raise ValueError("The ozone entry point requires parameter code 44201")

    request = source["request"]
    if (
        request.get("source"),
        request.get("dataset"),
        request.get("year"),
    ) != ("epa", "hourly_ozone", year):
        raise ValueError("The Bronze source does not match the ozone profile")

    root = Path(environment_root).resolve()
    data_path = Path(source["data_path"]).resolve()
    manifest_path = Path(source["manifest_path"]).resolve()

    for path in (data_path, manifest_path):
        if not path.is_relative_to(root):
            raise ValueError(f"Source path is outside the environment root: {path}")

    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    expected_identity = {
        "provider": "epa",
        "dataset": "hourly_ozone",
        "year": year,
    }
    if any(
        manifest["source"].get(name) != value
        for name, value in expected_identity.items()
    ):
        raise ValueError("The Bronze manifest does not match the ozone profile")

    if (
        source["schema"]["status"] != "compatible"
        or manifest["schema"]["status"] != "compatible"
    ):
        raise ValueError("The Bronze ozone schema is not compatible")

    raw, _ = read_validated_ozone_csv(data_path)

    required_columns = set(monitor_fields) | {
        "Date GMT",
        "Time GMT",
        "Sample Measurement",
        "Units of Measure",
        "MDL",
        "Qualifier",
        "Method Type",
        "Method Code",
        "Method Name",
    }
    missing_columns = required_columns - set(raw.columns)
    if missing_columns:
        raise ValueError(f"Missing ozone columns: {sorted(missing_columns)}")

    source_columns = raw.columns
    monitor_filter = F.lit(True)
    for column, value in monitor_values.items():
        monitor_filter = monitor_filter & (F.col(column) == F.lit(value))

    selected = raw.filter(monitor_filter)
    typed = type_ozone_fields(selected)
    checked = typed.select(
        "*",
        *[
            condition.alias(name)
            for name, condition in ozone_field_checks.items()
        ],
    )
    classified = classify_ozone_fields(checked, year=year)
    with_lineage = attach_bronze_lineage(
        classified,
        source,
        root,
        source_columns,
    )
    resolved = resolve_source_keys(
        with_lineage,
        key_columns=[*monitor_fields, "timestamp_utc"],
        key_is_usable=(
            ~F.col("missing_identifier")
            & F.col("timestamp_utc").isNotNull()
        ),
    )

    observations = build_ozone_observations(resolved)
    audit = build_ozone_audit(
        resolved,
        source_columns,
        list(ozone_field_checks),
    )

    before = summarize_ozone_values(
        checked,
        ozone_column="ozone_value",
        mdl_column="mdl_value",
    )
    after = summarize_ozone_values(
        observations,
        ozone_column="ozone_ppm",
        mdl_column="mdl_ppm",
    )

    disposition_rows = (
        audit
        .groupBy("disposition")
        .agg(F.sum("record_count").alias("source_records"))
        .orderBy("disposition")
        .collect()
    )
    dispositions = {
        row["disposition"]: row["source_records"]
        for row in disposition_rows
    }

    if sum(dispositions.values()) != before["records"]:
        raise ValueError("Ozone audit counts do not reconcile with the input")

    if dispositions.get("eligible", 0) != after["records"]:
        raise ValueError("Ozone observation and eligible audit counts differ")

    flag_rows = (
        observations
        .select(F.explode("quality_flags").alias("quality_flag"))
        .groupBy("quality_flag")
        .agg(F.count("*").alias("observations"))
        .orderBy("quality_flag")
        .collect()
    )
    quality_flags = {
        row["quality_flag"]: row["observations"]
        for row in flag_rows
    }

    return {
        "observations": observations,
        "audit": audit,
        "summary": {
            "monitor_id": "-".join(monitor_values.values()),
            "year": year,
            "source_version_sha256": manifest["original_file"]["sha256"],
            "before": before,
            "after": after,
            "dispositions": dispositions,
            "quality_flags": quality_flags,
        },
    }


### 5.8 Process Alabama ozone

The selected Alabama monitor is processed using the function above. The output summarizes retained and excluded records and the quality flags on retained observations.


In [20]:
ozone_candidates = [
    item for item in downloads
    if item["request"]["source"] == "epa"
    and item["request"]["dataset"] == "hourly_ozone"
    and item["request"]["year"] == alabama_profile["year"]
]

if len(ozone_candidates) != 1:
    raise ValueError("Expected one ozone input for the selected year.")

ozone_source = ozone_candidates[0]
if ozone_source["schema"]["status"] != "compatible":
    raise ValueError("The selected ozone input has a blocked schema.")

ozone_result = process_ozone_source(
    source=ozone_source,
    profile=alabama_profile,
    environment_root=silver_context.environment_root,
)
ozone_observations = ozone_result["observations"]
ozone_audit = ozone_result["audit"]
ozone_result_summary = ozone_result["summary"]

print("Ozone source:", ozone_source["data_path"])
print("Source version:", ozone_result_summary["source_version_sha256"])
ozone_summary_metrics = [
    ("Records", "records"),
    ("Negative ozone values", "negative_ozone"),
    ("Missing or unusable MDL", "missing_or_unusable_mdl"),
    ("Minimum finite ozone (ppm)", "minimum_ozone_ppm"),
    ("Maximum finite ozone (ppm)", "maximum_ozone_ppm"),
    ("Earliest timestamp (UTC)", "earliest_timestamp_utc"),
    ("Latest timestamp (UTC)", "latest_timestamp_utc"),
]
show_source_summary(ozone_result_summary, ozone_summary_metrics)

if show_diagnostics:
    ozone_observations.select(
        "monitor_id", "timestamp_utc", "ozone_ppm", "mdl_ppm", "quality_flags"
    ).orderBy("timestamp_utc").show(5, truncate=False)


26/11/09 23:11:59 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Ozone source: /home/ubuntu/dat535-final-project-data/dev/data/bronze/epa/hourly_ozone_2025/2b7b9e03b43a2bf20d3c3e78fc8c98b6b349fe2c3c4778382a2997144e7b2fd7/hourly_44201_2025.csv
Source version: 2b7b9e03b43a2bf20d3c3e78fc8c98b6b349fe2c3c4778382a2997144e7b2fd7


+--------------------------+-------------------+-------------------+
|metric                    |before_cleaning    |after_cleaning     |
+--------------------------+-------------------+-------------------+
|Records                   |5166               |5071               |
|Negative ozone values     |52                 |52                 |
|Missing or unusable MDL   |0                  |0                  |
|Minimum finite ozone (ppm)|-0.001             |-0.001             |
|Maximum finite ozone (ppm)|0.072              |0.072              |
|Earliest timestamp (UTC)  |2025-02-25 01:00:00|2025-04-01 06:00:00|
|Latest timestamp (UTC)    |2025-11-03 13:00:00|2025-11-03 13:00:00|
+--------------------------+-------------------+-------------------+

Exclusive source-record dispositions:
  eligible: 5071
  qualifier_6: 95
Reconciled source records: 5166
Overlapping quality flags on retained observations:
  below_reported_mdl: 922
  negative_ozone: 52


### 5.9 Ozone results and table definition

Source version: `2b7b9e03b43a2bf20d3c3e78fc8c98b6b349fe2c3c4778382a2997144e7b2fd7`.

This input contained 5,166 records. Cleaning retained 5,071 observations and excluded 95 records with qualifier `6`. No exact duplicates or conflicting keys were found, and the audit accounted for every source record.

All 52 negative measurements were retained. They are included in the 922 observations below the reported MDL, so the two flag counts overlap. Retained values ranged from -0.001 to 0.072 ppm.

The earliest timestamp changed from 2025-02-25 01:00 UTC to 2025-04-01 06:00 UTC after cleaning. The latest remained 2025-11-03 13:00 UTC. These bounds describe observed records; they do not establish continuous coverage or the monitor's operating schedule.

The observation table follows these rules:

- One eligible row per monitor and UTC timestamp within a source file and version. Identifier strings retain leading zeros.
- Timestamps fall on hourly boundaries within the selected year.
- Ozone values are finite, measured in ppm and at least -0.004 under the project's quality policy. Accepted values are unchanged.
- Blank, `XS`, `MD`, `1V` and `V` qualifiers are accepted subject to the other checks. Qualifier `6` and unsupported qualifiers are excluded.
- Missing, malformed, nonfinite or negative MDL values become null. Original values and flags are retained.
- Exact copies collapse with their count stored in `source_occurrences`. Differing rows at a usable key are quarantined.
- Original measurement values, relevant source codes and Bronze references remain available.

Audit entries can account for several identical copies. Summing `record_count` gives the source-record total.


## 6. Silver: temperature reports

### 6.1 Parse temperature fields

NOAA temperatures are already in degrees Celsius. The code parses them as numeric values and treats report times as UTC. Original timestamps are preserved, including reports between clock-hour boundaries.


In [21]:
temperature_inspection_columns = [
    "STATION",
    "DATE",
    "temperature",
    "temperature_Measurement_Code",
    "temperature_Quality_Code",
    "temperature_Report_Type",
    "temperature_Source_Code",
    "temperature_Source_Station_ID",
]


def type_temperature_fields(records):
    """Add typed UTC timestamps and temperatures, retaining source fields."""
    if records.sparkSession.conf.get("spark.sql.session.timeZone") != "UTC":
        raise ValueError(
            "Temperature timestamp parsing requires a UTC Spark session."
        )

    return (
        records
        .withColumn(
            "timestamp_utc",
            F.try_to_timestamp(
                F.col("DATE"),
                F.lit("yyyy-MM-dd'T'HH:mm:ss"),
            ),
        )
        .withColumn(
            "temperature_value",
            F.expr("try_cast(`temperature` AS DOUBLE)"),
        )
    )


temperature_missing_time = missing_string("DATE")
temperature_missing_value = missing_string("temperature")

temperature_field_checks = {
    "missing_station": missing_string("STATION"),
    "missing_timestamp": temperature_missing_time,
    "invalid_timestamp": (
        ~temperature_missing_time
        & F.col("timestamp_utc").isNull()
    ),
    "missing_temperature": temperature_missing_value,
    "invalid_temperature": (
        ~temperature_missing_value
        & F.col("temperature_value").isNull()
    ),
    "nonfinite_temperature": (
        F.col("temperature_value").isNotNull()
        & ~finite_number("temperature_value")
    ),
}


### 6.2 Classify temperature fields

Eligibility depends on station and year, a finite temperature within the allowed range, and the source, quality, report and measurement codes. Excluded records retain their original fields and every applicable reason.


In [22]:
def classify_temperature_fields(
    records,
    year,
    station_id,
    source_quality_pairs,
):
    """Classify checked temperature records without removing source rows."""
    accepted_pairs = {
        (str(source).strip(), str(quality).strip())
        for source, quality in source_quality_pairs
    }
    reviewed_pairs = {("223", "1"), ("343", "5")}

    if not accepted_pairs or not accepted_pairs <= reviewed_pairs:
        raise ValueError(
            "Temperature source/quality pairs must be a nonempty subset "
            "of the reviewed pairs: 223/1 and 343/5."
        )

    # Normalize metadata for comparison while retaining the source fields.
    source = F.coalesce(
        F.trim(F.col("temperature_Source_Code")), F.lit("")
    )
    quality = F.coalesce(
        F.trim(F.col("temperature_Quality_Code")), F.lit("")
    )
    report = F.coalesce(
        F.trim(F.col("temperature_Report_Type")), F.lit("")
    )
    measurement = F.coalesce(
        F.trim(F.col("temperature_Measurement_Code")), F.lit("")
    )

    timestamp = F.col("timestamp_utc")
    value = F.col("temperature_value")
    finite = finite_number("temperature_value")

    matches_pair = F.lit(False)
    for pair_source, pair_quality in sorted(accepted_pairs):
        matches_pair = matches_pair | (
            (source == pair_source) & (quality == pair_quality)
        )

    # Interpret legacy quality codes within their source family.
    rejected_quality = (
        ((source == "343") & quality.isin("2", "3", "6", "7"))
        | ((source == "223") & quality.isin("2", "3", "5"))
    )

    rules = [
        ("missing_station", F.col("missing_station")),
        (
            "unexpected_station",
            ~F.col("missing_station")
            & (F.col("STATION") != F.lit(station_id)),
        ),
        ("missing_timestamp", F.col("missing_timestamp")),
        ("invalid_timestamp", F.col("invalid_timestamp")),
        (
            "outside_year",
            timestamp.isNotNull() & (F.year(timestamp) != year),
        ),
        ("missing_temperature", F.col("missing_temperature")),
        ("invalid_temperature", F.col("invalid_temperature")),
        ("nonfinite_temperature", F.col("nonfinite_temperature")),
        (
            "outside_temperature_range",
            finite & ~value.between(-100.0, 60.0),
        ),
        ("rejected_source_quality", rejected_quality),
        (
            "unsupported_source_quality",
            ~matches_pair & ~rejected_quality,
        ),
        (
            "unsupported_report_type",
            ~report.isin("FM12", "FM15", "FM16"),
        ),
        ("unsupported_measurement_code", measurement != ""),
    ]

    reasons = F.filter(
        F.array(*[
            F.when(condition, F.lit(reason))
            for reason, condition in rules
        ]),
        lambda reason: reason.isNotNull(),
    )

    return (
        records
        .withColumn("source_code", source)
        .withColumn("quality_code", quality)
        .withColumn("report_type", report)
        .withColumn("measurement_code", measurement)
        .withColumn("field_reasons", reasons)
        .withColumn("field_eligible", F.size("field_reasons") == 0)
        .withColumn(
            "field_disposition",
            F.when(F.col("field_eligible"), F.lit("eligible"))
            .otherwise(F.element_at("field_reasons", 1)),
        )
    )


### 6.3 Resolve temperature record keys

Exact source copies are counted. Reports at the same station and timestamp are compared using their temperature-related fields. Conflicting reports are quarantined; rows differing only in other weather fields use one consistently selected representative.


In [23]:
def resolve_temperature_keys(records, source_columns, temperature_columns):
    """Resolve source copies and temperature variants within report keys."""
    if (
        not temperature_columns
        or not set(temperature_columns) <= set(source_columns)
    ):
        raise ValueError(
            "Temperature comparison fields must be original source columns."
        )

    source_scope = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
    ]
    key_columns = ["STATION", "timestamp_utc"]

    usable_key = F.coalesce(
        ~F.col("missing_station")
        & F.col("timestamp_utc").isNotNull(),
        F.lit(False),
    )

    # Collapse complete copies, including their stable derived fields.
    distinct_records = records.groupBy(*records.columns).agg(
        F.count("*").alias("source_occurrences")
    )

    key_window = Window.partitionBy(*(source_scope + key_columns))
    temperature_window = key_window.orderBy(
        *[F.col(name) for name in temperature_columns]
    )

    source_json = F.to_json(
        F.struct(*[F.col(name) for name in sorted(source_columns)]),
        options={"ignoreNullFields": "false"},
    )
    representative_window = key_window.orderBy(
        F.col("source_record_sha256"),
        source_json,
    )

    ranked = (
        distinct_records
        .withColumn(
            "_temperature_rank",
            F.dense_rank().over(temperature_window),
        )
        .withColumn(
            "_source_rank",
            F.row_number().over(representative_window),
        )
    )

    resolved = (
        ranked
        .withColumn(
            "duplicate_extra_records",
            F.col("source_occurrences") - 1,
        )
        .withColumn(
            "key_variant_count",
            F.when(
                usable_key,
                F.max("_temperature_rank").over(key_window),
            ),
        )
        .withColumn(
            "key_source_records",
            F.when(
                usable_key,
                F.sum("source_occurrences").over(key_window),
            ),
        )
        .withColumn(
            "key_conflict",
            F.coalesce(
                F.col("key_variant_count") > 1,
                F.lit(False),
            ),
        )
        .withColumn(
            "redundant_temperature",
            usable_key
            & ~F.col("key_conflict")
            & F.col("field_eligible")
            & (F.col("_source_rank") > 1),
        )
        .withColumn(
            "record_disposition",
            F.when(
                F.col("key_conflict"),
                F.lit("conflicting_key"),
            )
            .when(
                F.col("redundant_temperature"),
                F.lit("redundant_temperature"),
            )
            .otherwise(F.col("field_disposition")),
        )
        .withColumn(
            "record_reasons",
            F.when(
                F.col("key_conflict"),
                F.concat(
                    F.array(F.lit("conflicting_key")),
                    F.col("field_reasons"),
                ),
            )
            .when(
                F.col("redundant_temperature"),
                F.concat(
                    F.array(F.lit("redundant_temperature")),
                    F.col("field_reasons"),
                ),
            )
            .otherwise(F.col("field_reasons")),
        )
        .withColumn(
            "record_eligible",
            F.col("record_disposition") == "eligible",
        )
        .drop("_temperature_rank", "_source_rank")
    )

    return resolved


### 6.4 Temperature observations and audit

The observation table contains eligible reports with their actual timestamps, values, metadata and source references. The audit records every distinct source row and the counts of extra identical copies.


In [24]:
def build_temperature_observations(records):
    """Build the eligible temperature report schema."""
    missing_original_station = missing_string(
        "temperature_Source_Station_ID"
    )

    quality_flags = F.filter(
        F.array(
            F.when(
                missing_original_station,
                F.lit("missing_source_station_id"),
            ),
        ),
        lambda flag: flag.isNotNull(),
    )

    lineage_columns = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
        "source_manifest_path",
        "source_record_hash_scheme",
        "source_record_sha256",
    ]

    return records.filter(F.col("record_eligible")).select(
        F.col("STATION").alias("station_id"),
        "timestamp_utc",
        F.col("temperature_value").alias("temperature_c"),
        F.col("temperature").alias("temperature_raw"),
        F.col("DATE").alias("timestamp_raw"),
        "source_code",
        "quality_code",
        "report_type",
        "measurement_code",
        F.when(
            ~missing_original_station,
            F.trim(F.col("temperature_Source_Station_ID")),
        ).alias("source_station_id"),
        F.col("temperature_Source_Code").alias("source_code_raw"),
        F.col("temperature_Quality_Code").alias("quality_code_raw"),
        F.col("temperature_Report_Type").alias("report_type_raw"),
        F.col("temperature_Measurement_Code").alias("measurement_code_raw"),
        F.col("temperature_Source_Station_ID").alias("source_station_id_raw"),
        quality_flags.alias("quality_flags"),
        "source_occurrences",
        F.col("key_source_records").alias("supporting_source_records"),
        *lineage_columns,
    )


def build_temperature_audit(records, source_columns, diagnostic_columns):
    """Build counted audit entries for source rows and duplicate copies."""
    lineage_columns = [
        "source_provider",
        "source_dataset",
        "source_version_sha256",
        "source_file_path",
        "source_file_sha256",
        "source_manifest_path",
        "source_record_hash_scheme",
        "source_record_sha256",
    ]

    base = (
        records
        .withColumn(
            "station_id",
            F.when(
                ~F.col("missing_station"),
                F.col("STATION"),
            ),
        )
        .withColumn(
            "raw_fields",
            named_column_map(source_columns),
        )
        .withColumn(
            "field_flags",
            named_column_map(diagnostic_columns),
        )
    )

    common_columns = [
        "station_id",
        "timestamp_utc",
        "raw_fields",
        "field_flags",
        "field_disposition",
        "field_reasons",
        "key_variant_count",
        "key_source_records",
        "key_conflict",
        "redundant_temperature",
        *lineage_columns,
    ]

    representatives = base.select(
        *common_columns,
        F.col("record_disposition").alias("disposition"),
        F.col("record_reasons").alias("reasons"),
        F.lit(1).cast("long").alias("record_count"),
    )

    duplicate_copies = (
        base
        .filter(F.col("duplicate_extra_records") > 0)
        .select(
            *common_columns,
            F.lit("exact_duplicate").alias("disposition"),
            F.concat(
                F.array(F.lit("exact_duplicate")),
                F.col("record_reasons"),
            ).alias("reasons"),
            F.col("duplicate_extra_records").alias("record_count"),
        )
    )

    return representatives.unionByName(duplicate_copies)


### 6.5 Temperature processing function

This function checks the station-year source against the profile and applies the parsing, quality and duplicate rules. It verifies that the audit accounts for the complete input and that eligible audit entries agree with the observation count.


In [25]:
def summarize_temperature_values(
    records,
    temperature_column,
    source_station_column,
):
    """Summarize temperature values, metadata completeness and time bounds."""
    temperature = F.col(temperature_column)
    temperature_is_finite = finite_number(temperature_column)

    return records.agg(
        F.count("*").alias("records"),
        F.count(
            F.when(~temperature_is_finite, 1)
        ).alias("missing_or_unusable_temperature"),
        F.count(
            F.when(
                temperature_is_finite & (temperature < 0),
                1,
            )
        ).alias("negative_temperatures"),
        F.count(
            F.when(missing_string(source_station_column), 1)
        ).alias("missing_source_station_id"),
        F.min(
            F.when(temperature_is_finite, temperature)
        ).alias("minimum_temperature_c"),
        F.max(
            F.when(temperature_is_finite, temperature)
        ).alias("maximum_temperature_c"),
        F.date_format(
            F.min("timestamp_utc"),
            "yyyy-MM-dd HH:mm:ss",
        ).alias("earliest_timestamp_utc"),
        F.date_format(
            F.max("timestamp_utc"),
            "yyyy-MM-dd HH:mm:ss",
        ).alias("latest_timestamp_utc"),
    ).first().asDict()


def process_temperature_source(source, profile, environment_root):
    """Build temperature observations, audit records and quality summaries."""
    year = profile["year"]
    station_id = profile["weather_mapping"]["noaa_station_id"]
    source_quality_pairs = (
        profile["temperature_policy"]["source_quality_pairs"]
    )

    if type(year) is not int:
        raise ValueError("The profile year must be an integer.")

    if (
        not isinstance(station_id, str)
        or not station_id
        or station_id != station_id.strip()
    ):
        raise ValueError(
            "The station identifier must be a nonempty, trimmed string."
        )

    request = source["request"]
    if (
        request.get("source"),
        request.get("dataset"),
        request.get("year"),
        request.get("station_id"),
    ) != ("noaa", "hourly_weather", year, station_id):
        raise ValueError(
            "The Bronze source does not match the temperature profile."
        )

    root = Path(environment_root).resolve(strict=True)
    data_path = Path(source["data_path"]).resolve(strict=True)
    manifest_path = Path(source["manifest_path"]).resolve(strict=True)

    for path in (data_path, manifest_path):
        if not path.is_relative_to(root):
            raise ValueError(
                f"Source path is outside the environment root: {path}"
            )

    manifest = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )

    if manifest.get("manifest_version") != 1:
        raise ValueError("Unsupported Bronze manifest version.")

    expected_identity = {
        "provider": "noaa",
        "dataset": "hourly_weather",
        "year": year,
        "station_id": station_id,
    }
    if any(
        manifest["source"].get(name) != value
        for name, value in expected_identity.items()
    ):
        raise ValueError(
            "The Bronze manifest does not match the temperature profile."
        )

    if (
        source["schema"]["status"] != "compatible"
        or manifest["schema"]["status"] != "compatible"
    ):
        raise ValueError(
            "The Bronze temperature schema is not compatible."
        )

    raw, structure = read_validated_delimited_file(
        data_path,
        delimiter="|",
    )

    temperature_columns = [
        "STATION",
        "DATE",
        "temperature",
        "temperature_Measurement_Code",
        "temperature_Quality_Code",
        "temperature_Report_Type",
        "temperature_Source_Code",
        "temperature_Source_Station_ID",
    ]

    missing_columns = set(temperature_columns) - set(raw.columns)
    if missing_columns:
        raise ValueError(
            f"Missing temperature columns: {sorted(missing_columns)}"
        )

    source_columns = raw.columns

    # Classify every source row so exclusions remain available for audit.
    typed = type_temperature_fields(raw)
    checked = typed.select(
        "*",
        *[
            condition.alias(name)
            for name, condition in temperature_field_checks.items()
        ],
    )

    classified = classify_temperature_fields(
        checked,
        year=year,
        station_id=station_id,
        source_quality_pairs=source_quality_pairs,
    )

    with_lineage = attach_bronze_lineage(
        classified,
        source=source,
        environment_root=root,
        source_columns=source_columns,
    )

    resolved = resolve_temperature_keys(
        with_lineage,
        source_columns=source_columns,
        temperature_columns=temperature_columns,
    )

    observations = build_temperature_observations(resolved)
    audit = build_temperature_audit(
        resolved,
        source_columns=source_columns,
        diagnostic_columns=list(temperature_field_checks),
    )

    before = summarize_temperature_values(
        checked,
        temperature_column="temperature_value",
        source_station_column="temperature_Source_Station_ID",
    )
    after = summarize_temperature_values(
        observations,
        temperature_column="temperature_c",
        source_station_column="source_station_id",
    )

    disposition_rows = (
        audit
        .groupBy("disposition")
        .agg(F.sum("record_count").alias("source_records"))
        .orderBy("disposition")
        .collect()
    )
    dispositions = {
        row["disposition"]: row["source_records"]
        for row in disposition_rows
    }

    if before["records"] != structure["records"]:
        raise ValueError(
            "Spark and structural temperature record counts differ."
        )

    if sum(dispositions.values()) != before["records"]:
        raise ValueError(
            "Temperature audit counts do not reconcile with the input."
        )

    if dispositions.get("eligible", 0) != after["records"]:
        raise ValueError(
            "Temperature observation and eligible audit counts differ."
        )

    if after["missing_or_unusable_temperature"] != 0:
        raise ValueError(
            "Eligible observations contain unusable temperatures."
        )

    flag_rows = (
        observations
        .select(F.explode("quality_flags").alias("quality_flag"))
        .groupBy("quality_flag")
        .agg(F.count("*").alias("observations"))
        .orderBy("quality_flag")
        .collect()
    )
    quality_flags = {
        row["quality_flag"]: row["observations"]
        for row in flag_rows
    }

    if (
        quality_flags.get("missing_source_station_id", 0)
        != after["missing_source_station_id"]
    ):
        raise ValueError(
            "Missing source-station flags and summary counts differ."
        )

    return {
        "observations": observations,
        "audit": audit,
        "summary": {
            "station_id": station_id,
            "year": year,
            "source_version_sha256": manifest["original_file"]["sha256"],
            "before": before,
            "after": after,
            "dispositions": dispositions,
            "quality_flags": quality_flags,
        },
    }


### 6.6 Process Alabama temperature

The selected station-year file is processed using the function above. The summary shows the effects of cleaning on record counts, temperature ranges and time bounds.


In [26]:
temperature_station_id = (
    alabama_profile["weather_mapping"]["noaa_station_id"]
)

temperature_candidates = [
    item for item in downloads
    if item["request"]["source"] == "noaa"
    and item["request"]["dataset"] == "hourly_weather"
    and item["request"].get("year") == alabama_profile["year"]
    and item["request"].get("station_id") == temperature_station_id
]

if len(temperature_candidates) != 1:
    raise ValueError(
        "Expected one weather input for the selected station and year."
    )

temperature_source = temperature_candidates[0]
if temperature_source["schema"]["status"] != "compatible":
    raise ValueError("The selected weather input has a blocked schema.")

temperature_result = process_temperature_source(
    source=temperature_source,
    profile=alabama_profile,
    environment_root=silver_context.environment_root,
)
temperature_observations = temperature_result["observations"]
temperature_audit = temperature_result["audit"]
temperature_result_summary = temperature_result["summary"]

print("Temperature source:", temperature_source["data_path"])
print("Source version:", temperature_result_summary["source_version_sha256"])
temperature_summary_metrics = [
    ("Records", "records"),
    ("Missing or unusable temperature", "missing_or_unusable_temperature"),
    ("Temperatures below zero (C)", "negative_temperatures"),
    ("Missing original source-station ID", "missing_source_station_id"),
    ("Minimum finite temperature (C)", "minimum_temperature_c"),
    ("Maximum finite temperature (C)", "maximum_temperature_c"),
    ("Earliest timestamp (UTC)", "earliest_timestamp_utc"),
    ("Latest timestamp (UTC)", "latest_timestamp_utc"),
]
show_source_summary(temperature_result_summary, temperature_summary_metrics)

if show_diagnostics:
    temperature_observations.select(
        "station_id", "timestamp_utc", "temperature_c",
        "source_code", "quality_code", "report_type", "quality_flags",
    ).orderBy("timestamp_utc").show(5, truncate=False)


26/11/09 23:13:03 WARN DAGScheduler: Broadcasting large task binary with size 1041.5 KiB
26/11/09 23:13:04 WARN DAGScheduler: Broadcasting large task binary with size 1094.9 KiB
26/11/09 23:13:05 WARN DAGScheduler: Broadcasting large task binary with size 1095.5 KiB
26/11/09 23:13:05 WARN DAGScheduler: Broadcasting large task binary with size 1091.6 KiB


Temperature source: /home/ubuntu/dat535-final-project-data/dev/data/bronze/noaa/hourly_weather_2025_USW00053864/2ccd886d5e6ba9b8c502058db047094dc8a76e50bcfd1c3d54f5c458ab97d308/GHCNh_USW00053864_2025.psv
Source version: 2ccd886d5e6ba9b8c502058db047094dc8a76e50bcfd1c3d54f5c458ab97d308
+----------------------------------+-------------------+-------------------+
|metric                            |before_cleaning    |after_cleaning     |
+----------------------------------+-------------------+-------------------+
|Records                           |11851              |11686              |
|Missing or unusable temperature   |29                 |0                  |
|Temperatures below zero (C)       |365                |365                |
|Missing original source-station ID|29                 |0                  |
|Minimum finite temperature (C)    |-10.0              |-10.0              |
|Maximum finite temperature (C)    |36.7               |36.7               |
|Earliest timestamp (U

### 6.7 Temperature results and table definition

Source version: `2ccd886d5e6ba9b8c502058db047094dc8a76e50bcfd1c3d54f5c458ab97d308`.

This input contained 11,851 records. Cleaning retained 11,686 reports and excluded 29 missing temperatures, 7 rejected source-quality combinations and 129 unsupported combinations. No exact duplicates, conflicting keys or redundant temperature reports were found. The audit accounted for every source record.

All 365 negative temperatures were retained. Values ranged from -10.0 to 36.7 degrees Celsius, with timestamps from 2025-01-01 00:53 UTC to 2025-12-31 23:53 UTC. These bounds do not imply complete hourly coverage. Every retained report had an original source-station identifier.

The observation table follows these rules:

- One eligible row per station and actual UTC report timestamp within a source file and version. The station and year match the profile.
- Temperatures are finite and within -100 to 60 degrees Celsius inclusive. This is a project plausibility rule; accepted values are unchanged.
- Source-quality pairs are selected from `223/1` and `343/5`. Supported reports are `FM12`, `FM15` and `FM16`, with a blank measurement code.
- Codes are trimmed for comparison, while their original values are retained. Missing original source-station identifiers are allowed and flagged.
- Exact copies collapse. Differences in original temperature-related fields, including formatting differences, cause variants at a usable key to be quarantined.
- Eligible rows differing only in other weather fields use a deterministic representative. Other variants remain in the audit.
- `source_occurrences` counts exact copies of the representative. `supporting_source_records` counts all records supporting the equivalent temperature report.

The audit preserves the original parsed weather fields, diagnostics, reasons and source references. Summing `record_count` reproduces the input count. Hourly temperature selection and the ozone-temperature join are separate from this source-level table.


## 7. Reference-data Silver

### 7.1 Select reference records

The reference files describe the EPA site and monitor and the NOAA weather station. The code checks each file and selects records matching the Alabama profile. The summary lists the number of rows and columns in each file and the number of matching records.


In [27]:
reference_site_keys = {
    "State Code": alabama_profile["ozone_monitor"]["state_code"],
    "County Code": alabama_profile["ozone_monitor"]["county_code"],
    "Site Number": alabama_profile["ozone_monitor"]["site_number"],
}

reference_specs = [
    (
        "epa_sites",
        "epa",
        "sites",
        reference_site_keys,
    ),
    (
        "epa_monitors",
        "epa",
        "monitors",
        {
            **reference_site_keys,
            "Parameter Code": (
                alabama_profile["ozone_monitor"]["parameter_code"]
            ),
            "POC": alabama_profile["ozone_monitor"]["poc"],
        },
    ),
    (
        "noaa_stations",
        "noaa",
        "stations",
        {
            "GHCN_ID": (
                alabama_profile["weather_mapping"]["noaa_station_id"]
            ),
        },
    ),
]

reference_sources = {}
reference_raw_tables = {}
reference_structures = {}
reference_profile_rows = {}
reference_profile_counts = {}
reference_input_summary = []

for (
    reference_name,
    reference_provider,
    reference_dataset,
    reference_keys,
) in reference_specs:
    reference_candidates = [
        item for item in downloads
        if item["request"]["source"] == reference_provider
        and item["request"]["dataset"] == reference_dataset
    ]

    if len(reference_candidates) != 1:
        raise ValueError(
            f"{reference_name}: expected one selected Bronze input."
        )

    reference_source = reference_candidates[0]

    if reference_source["schema"]["status"] != "compatible":
        raise ValueError(
            f"{reference_name}: the selected input has a blocked schema."
        )

    reference_records, reference_structure = (
        read_validated_delimited_file(
            reference_source["data_path"],
            delimiter=",",
        )
    )

    reference_missing_columns = (
        set(required_columns[(reference_provider, reference_dataset)])
        - set(reference_records.columns)
    )
    if reference_missing_columns:
        raise ValueError(
            f"{reference_name}: missing required columns "
            f"{sorted(reference_missing_columns)}"
        )

    reference_record_count = reference_records.count()

    if reference_record_count != reference_structure["records"]:
        raise ValueError(
            f"{reference_name}: structural and Spark record counts differ."
        )

    reference_match = F.lit(True)
    for source_column, expected_value in reference_keys.items():
        reference_match = reference_match & (
            F.col(source_column) == F.lit(expected_value)
        )

    reference_selected = reference_records.filter(reference_match)
    reference_selected_count = reference_selected.count()

    reference_sources[reference_name] = reference_source
    reference_raw_tables[reference_name] = reference_records
    reference_structures[reference_name] = reference_structure
    reference_profile_rows[reference_name] = reference_selected
    reference_profile_counts[reference_name] = reference_selected_count


    reference_input_summary.append((
        reference_name,
        reference_record_count,
        len(reference_records.columns),
        reference_selected_count,
    ))
    if show_diagnostics:
        print("Reference:", reference_name)
        print("File:", reference_source["data_path"])
        print("Manifest:", reference_source["manifest_path"])
        if reference_selected_count:
            reference_selected.orderBy(
                *reference_selected.columns
            ).show(3, truncate=False, vertical=True)

spark.createDataFrame(
    reference_input_summary,
    schema=(
        "reference STRING, source_records LONG, "
        "source_columns LONG, selected_records LONG"
    ),
).show(truncate=False)


+-------------+--------------+--------------+----------------+
|reference    |source_records|source_columns|selected_records|
+-------------+--------------+--------------+----------------+
|epa_sites    |20994         |28            |1               |
|epa_monitors |370090        |32            |1               |
|noaa_stations|38870         |11            |1               |
+-------------+--------------+--------------+----------------+



### 7.2 Parse reference fields

Coordinates and elevations become numeric values, and metadata dates become calendar dates. Original identifiers and fields are preserved. Checks identify missing values, failed conversions, nonfinite numbers, out-of-range coordinates and inconsistent date order.


In [28]:
reference_type_specs = {
    "epa_sites": {
        "latitude_deg": ("Latitude", "double"),
        "longitude_deg": ("Longitude", "double"),
        "elevation_m": ("Elevation", "double"),
        "site_established_date": ("Site Established Date", "date"),
        "site_closed_date": ("Site Closed Date", "date"),
        "extraction_date": ("Extraction Date", "date"),
    },
    "epa_monitors": {
        "latitude_deg": ("Latitude", "double"),
        "longitude_deg": ("Longitude", "double"),
        "first_year_of_data": ("First Year of Data", "int"),
        "last_sample_date": ("Last Sample Date", "date"),
        "extraction_date": ("Extraction Date", "date"),
    },
    "noaa_stations": {
        "latitude_deg": ("LATITUDE", "double"),
        "longitude_deg": ("LONGITUDE", "double"),
        "elevation_m": ("ELEVATION", "double"),
    },
}


def type_reference_fields(records, reference_name):
    """Add typed reference fields while preserving original source columns."""
    if reference_name not in reference_type_specs:
        raise ValueError(
            f"Unsupported reference type: {reference_name}"
        )

    field_specs = reference_type_specs[reference_name]

    missing_columns = {
        source_name
        for source_name, _ in field_specs.values()
    } - set(records.columns)

    if missing_columns:
        raise ValueError(
            f"{reference_name}: missing columns "
            f"{sorted(missing_columns)}"
        )

    existing_names = {name.casefold() for name in records.columns}
    collisions = [
        name for name in field_specs
        if name.casefold() in existing_names
    ]
    if collisions:
        raise ValueError(
            f"Typed fields would overwrite existing columns: {collisions}"
        )

    typed_expressions = []

    for output_name, (source_name, target_type) in field_specs.items():
        if target_type == "date":
            parsed_value = F.try_to_timestamp(
                F.col(source_name),
                F.lit("yyyy-MM-dd"),
            ).cast("date")
        elif target_type in {"double", "int"}:
            escaped_source_name = source_name.replace("`", "``")
            parsed_value = F.expr(
                f"try_cast(`{escaped_source_name}` AS {target_type})"
            )
        else:
            raise ValueError(
                f"Unsupported reference field type: {target_type}"
            )

        typed_expressions.append(
            parsed_value.alias(output_name)
        )

    return records.select("*", *typed_expressions)


def build_reference_field_checks(reference_name, key_columns):
    """Build reference diagnostics without scanning source records."""
    checks = {
        f"missing_identifier_{name.lower().replace(' ', '_')}": missing_string(name)
        for name in key_columns
    }

    # Compare original values with their typed counterparts.
    for typed_column, (source_column, data_type) in (
        reference_type_specs[reference_name].items()
    ):
        source_missing = missing_string(source_column)

        checks[f"missing_{typed_column}"] = source_missing
        checks[f"invalid_{typed_column}"] = (
            ~source_missing & F.col(typed_column).isNull()
        )

        if data_type == "double":
            checks[f"nonfinite_{typed_column}"] = (
                F.col(typed_column).isNotNull()
                & ~finite_number(typed_column)
            )

    checks["latitude_out_of_range"] = (
        finite_number("latitude_deg")
        & ~F.col("latitude_deg").between(-90.0, 90.0)
    )
    checks["longitude_out_of_range"] = (
        finite_number("longitude_deg")
        & ~F.col("longitude_deg").between(-180.0, 180.0)
    )

    if reference_name in {"epa_sites", "epa_monitors"}:
        checks["missing_datum"] = missing_string("Datum")

    # Compare metadata dates without restricting them to the analysis year.
    if reference_name == "epa_sites":
        checks["closure_before_establishment"] = (
            F.col("site_closed_date") < F.col("site_established_date")
        )
        checks["establishment_after_extraction"] = (
            F.col("site_established_date") > F.col("extraction_date")
        )
        checks["closure_after_extraction"] = (
            F.col("site_closed_date") > F.col("extraction_date")
        )

    elif reference_name == "epa_monitors":
        checks["first_year_after_last_sample"] = (
            F.col("first_year_of_data") > F.year("last_sample_date")
        )
        checks["last_sample_after_extraction"] = (
            F.col("last_sample_date") > F.col("extraction_date")
        )
    return checks


reference_field_checks = {
    reference_name: build_reference_field_checks(reference_name, reference_keys)
    for reference_name, _, _, reference_keys in reference_specs
}


### 7.3 Classify reference records

Missing identifiers and unusable coordinates exclude a record. Problems in optional elevation, date or datum fields produce flags. An absent site closure date is allowed without a quality flag.

NOAA's elevation value of `-999.9` means missing. EPA datums other than WGS84 are flagged for review before coordinate comparison.

These checks assess individual records. Entity-key uniqueness and publication of reference tables are not covered here.


In [29]:
def classify_reference_fields(records, reference_name):
    if reference_name not in reference_field_checks:
        raise ValueError(f"Unknown reference: {reference_name}")

    checks = dict(reference_field_checks[reference_name])

    # NOAA documents -999.9 as missing elevation.
    if reference_name == "noaa_stations":
        checks["missing_elevation_m"] = (
            missing_string("ELEVATION")
            | F.coalesce(
                F.col("elevation_m") == F.lit(-999.9),
                F.lit(False),
            )
        )

    # Preserve other datums and require review before coordinate comparison.
    if reference_name in {"epa_sites", "epa_monitors"}:
        checks["non_wgs84_datum"] = (
            ~missing_string("Datum")
            & (F.upper(F.trim(F.col("Datum"))) != F.lit("WGS84"))
        )

    if reference_name == "epa_monitors":
        checks["first_year_out_of_range"] = (
            F.col("first_year_of_data").isNotNull()
            & ~F.col("first_year_of_data").between(1, 9999)
        )
        checks["first_year_after_extraction"] = (
            F.col("first_year_of_data") > F.year("extraction_date")
        )

    exclusion_names = [
        name
        for name in checks
        if name.startswith("missing_identifier_")
    ] + [
        "missing_latitude_deg",
        "invalid_latitude_deg",
        "nonfinite_latitude_deg",
        "latitude_out_of_range",
        "missing_longitude_deg",
        "invalid_longitude_deg",
        "nonfinite_longitude_deg",
        "longitude_out_of_range",
    ]

    optional_flag_names = [
        name
        for name in checks
        if name not in exclusion_names
        and name != "missing_site_closed_date"
    ]

    def matching_checks(names):
        return F.filter(
            F.array(
                *[
                    F.when(
                        F.coalesce(checks[name], F.lit(False)),
                        F.lit(name),
                    )
                    for name in names
                ]
            ),
            lambda reason: reason.isNotNull(),
        )

    diagnostic_entries = []
    for name, condition in checks.items():
        diagnostic_entries.extend(
            [
                F.lit(name),
                F.coalesce(condition, F.lit(False)),
            ]
        )

    return (
        records
        .withColumn(
            "reference_field_flags",
            F.create_map(*diagnostic_entries),
        )
        .withColumn(
            "field_reasons",
            matching_checks(exclusion_names),
        )
        .withColumn(
            "field_eligible",
            F.size("field_reasons") == 0,
        )
        .withColumn(
            "field_disposition",
            F.when(
                F.col("field_eligible"),
                F.lit("eligible"),
            ).otherwise(F.element_at("field_reasons", 1)),
        )
        .withColumn(
            "reference_quality_flags",
            matching_checks(optional_flag_names),
        )
    )

reference_typed_tables = {
    reference_name: type_reference_fields(
        reference_profile_rows[reference_name],
        reference_name,
    )
    for reference_name, _, _, _ in reference_specs
}

reference_classified_tables = {}
reference_disposition_summary = None

for reference_name, _, _, _ in reference_specs:
    classified = classify_reference_fields(
        reference_typed_tables[reference_name],
        reference_name,
    )
    reference_classified_tables[reference_name] = classified

    summary = (
        classified
        .groupBy("field_disposition", "reference_quality_flags")
        .agg(F.count("*").alias("records"))
        .withColumn("reference", F.lit(reference_name))
        .select(
            "reference",
            "field_disposition",
            "records",
            "reference_quality_flags",
        )
    )

    if reference_disposition_summary is None:
        reference_disposition_summary = summary
    else:
        reference_disposition_summary = (
            reference_disposition_summary.unionByName(summary)
        )

reference_disposition_summary.orderBy(
    "reference",
    "field_disposition",
    "reference_quality_flags",
).show(truncate=False)

if show_diagnostics:
    for reference_name, _, _, reference_keys in reference_specs:
        print("Typed reference:", reference_name)
        reference_classified_tables[reference_name].select(
            *reference_keys,
            *reference_type_specs[reference_name],
            "reference_quality_flags",
        ).orderBy(*reference_keys).show(3, truncate=False, vertical=True)


+-------------+-----------------+-------+-----------------------+
|reference    |field_disposition|records|reference_quality_flags|
+-------------+-----------------+-------+-----------------------+
|epa_monitors |eligible         |1      |[]                     |
|epa_sites    |eligible         |1      |[]                     |
|noaa_stations|eligible         |1      |[]                     |
+-------------+-----------------+-------+-----------------------+



### 7.4 Reference lineage and key resolution

Attach Bronze file and record provenance to the selected reference records.
Represent exact duplicate rows once while retaining their occurrence counts.
Quarantine different rows that share an entity key within the same source
file and version. Reconcile the resulting dispositions with the selected
input records.

In [30]:
def resolve_reference_keys(records, key_columns):
    """Count exact copies and quarantine conflicting reference keys."""
    key_columns = list(key_columns)
    if not key_columns:
        raise ValueError("Reference keys must contain at least one column.")

    key_is_usable = F.lit(True)
    for column in key_columns:
        key_is_usable = key_is_usable & ~missing_string(column)

    # Represent diagnostic maps as sorted entries during grouping.
    groupable = (
        records
        .withColumn(
            "__reference_field_entries",
            F.sort_array(F.map_entries("reference_field_flags")),
        )
        .drop("reference_field_flags")
    )

    return (
        resolve_source_keys(
            groupable,
            key_columns=key_columns,
            key_is_usable=key_is_usable,
        )
        .withColumn(
            "reference_field_flags",
            F.map_from_entries("__reference_field_entries"),
        )
        .drop("__reference_field_entries")
    )


reference_resolved_tables = {}
reference_key_summary = []

for reference_name, _, _, reference_keys in reference_specs:
    reference_with_lineage = attach_bronze_lineage(
        records=reference_classified_tables[reference_name],
        source=reference_sources[reference_name],
        environment_root=silver_context.environment_root,
        source_columns=reference_raw_tables[reference_name].columns,
    )

    resolved = resolve_reference_keys(
        reference_with_lineage,
        key_columns=list(reference_keys),
    )
    reference_resolved_tables[reference_name] = resolved

    counts = resolved.agg(
        F.coalesce(F.sum("source_occurrences"), F.lit(0)).alias(
            "input_records"
        ),
        F.count("*").alias("distinct_source_rows"),
        F.coalesce(F.sum("duplicate_extra_records"), F.lit(0)).alias(
            "duplicate_extra_records"
        ),
        F.count(F.when(F.col("record_eligible"), 1)).alias(
            "eligible_records"
        ),
        F.count(F.when(F.col("key_conflict"), 1)).alias(
            "quarantined_records"
        ),
        F.count(
            F.when(
                ~F.col("key_conflict") & ~F.col("record_eligible"),
                1,
            )
        ).alias("field_excluded_records"),
    ).first().asDict()

    if (
        counts["input_records"] != reference_profile_counts[reference_name]
        or counts["input_records"] != (
            counts["distinct_source_rows"] + counts["duplicate_extra_records"]
        )
        or counts["distinct_source_rows"] != (
            counts["eligible_records"]
            + counts["quarantined_records"]
            + counts["field_excluded_records"]
        )
    ):
        raise ValueError(f"{reference_name}: reference counts do not reconcile.")

    reference_key_summary.append((
        reference_name,
        counts["input_records"],
        counts["distinct_source_rows"],
        counts["duplicate_extra_records"],
        counts["eligible_records"],
        counts["quarantined_records"],
        counts["field_excluded_records"],
    ))

spark.createDataFrame(
    reference_key_summary,
    schema=(
        "reference STRING, input_records LONG, distinct_source_rows LONG, "
        "duplicate_extra_records LONG, eligible_records LONG, "
        "quarantined_records LONG, field_excluded_records LONG"
    ),
).orderBy("reference").show(truncate=False)

+-------------+-------------+--------------------+-----------------------+----------------+-------------------+----------------------+
|reference    |input_records|distinct_source_rows|duplicate_extra_records|eligible_records|quarantined_records|field_excluded_records|
+-------------+-------------+--------------------+-----------------------+----------------+-------------------+----------------------+
|epa_monitors |1            |1                   |0                      |1               |0                  |0                     |
|epa_sites    |1            |1                   |0                      |1               |0                  |0                     |
|noaa_stations|1            |1                   |0                      |1               |0                  |0                     |
+-------------+-------------+--------------------+-----------------------+----------------+-------------------+----------------------+



### 7.5 Reference tables and audit records

Build typed tables for eligible sites, monitors and weather stations.
Preserve identifiers, relevant metadata, quality flags and source lineage.
Unusable optional values become null; valid values with consistency warnings
remain flagged.

The audit preserves every original source field and records the disposition
of each distinct row and any additional exact copies.

In [31]:
reference_output_specs = {
    "epa_sites": {
        "id_column": "site_id",
        "keys": {
            "State Code": "state_code", "County Code": "county_code",
            "Site Number": "site_number",
        },
        "text": {
            "Local Site Name": "site_name", "State Name": "state_name",
            "County Name": "county_name", "Land Use": "land_use",
            "Location Setting": "location_setting", "Datum": "datum",
        },
    },
    "epa_monitors": {
        "id_column": "monitor_id",
        "keys": {
            "State Code": "state_code", "County Code": "county_code",
            "Site Number": "site_number", "Parameter Code": "parameter_code",
            "POC": "poc",
        },
        "text": {
            "Parameter Name": "parameter_name", "Monitor Type": "monitor_type",
            "Measurement Scale": "measurement_scale", "Datum": "datum",
            "Last Method Code": "last_method_code", "Last Method": "last_method",
        },
    },
    "noaa_stations": {
        "id_column": "station_id",
        "keys": {"GHCN_ID": "station_id"},
        "text": {
            "NAME": "station_name", "STATE": "state_abbreviation",
            "ICAO": "icao_code", "ISO_CODE": "country_code",
        },
    },
}

reference_lineage_columns = [
    "source_provider", "source_dataset", "source_version_sha256",
    "source_file_path", "source_file_sha256", "source_manifest_path",
    "source_record_hash_scheme", "source_record_sha256",
]


def build_reference_outputs(records, reference_name, source_columns):
    """Build an eligible reference table and a counted source audit."""
    spec = reference_output_specs[reference_name]
    usable_key = F.lit(True)
    for column in spec["keys"]:
        usable_key = usable_key & ~missing_string(column)

    entity_id = F.when(
        usable_key,
        F.concat_ws("-", *[F.col(column) for column in spec["keys"]]),
    )

    identifiers = [
        F.col(original).alias(output)
        for original, output in spec["keys"].items()
        if output != spec["id_column"]
    ]
    identifiers.append(entity_id.alias(spec["id_column"]))

    typed_fields = []
    for column in reference_type_specs[reference_name]:
        value = F.col(column)
        if column not in {"latitude_deg", "longitude_deg"}:
            unusable = F.lit(False)
            for prefix in ("missing_", "invalid_", "nonfinite_"):
                unusable = unusable | F.coalesce(
                    F.col("reference_field_flags")[prefix + column],
                    F.lit(False),
                )
            if column == "first_year_of_data":
                unusable = unusable | F.coalesce(
                    F.col("reference_field_flags")["first_year_out_of_range"],
                    F.lit(False),
                )
            value = F.when(~unusable, value)
        typed_fields.append(value.alias(column))

    text_fields = []
    for original, output in spec["text"].items():
        value = (
            F.when(~missing_string(original), F.trim(F.col(original)))
            if original in records.columns
            else F.lit(None).cast("string")
        )
        text_fields.append(value.alias(output))

    reference = records.filter("record_eligible").select(
        *identifiers,
        *typed_fields,
        *text_fields,
        F.col("reference_quality_flags").alias("quality_flags"),
        "source_occurrences",
        *reference_lineage_columns,
    )

    audit_base = (
        records
        .withColumn("reference", F.lit(reference_name))
        .withColumn("entity_id", entity_id)
        .withColumn("raw_fields", named_column_map(source_columns))
        .withColumn("field_flags", F.col("reference_field_flags"))
    )
    common = [
        "reference", "entity_id", "raw_fields", "field_flags",
        "field_disposition", "field_reasons", "reference_quality_flags",
        "key_variant_count", "key_conflict", *reference_lineage_columns,
    ]
    representatives = audit_base.select(
        *common,
        F.col("record_disposition").alias("disposition"),
        F.col("record_reasons").alias("reasons"),
        F.lit(1).cast("long").alias("record_count"),
    )
    duplicate_copies = audit_base.filter("duplicate_extra_records > 0").select(
        *common,
        F.lit("exact_duplicate").alias("disposition"),
        F.concat(
            F.array(F.lit("exact_duplicate")), F.col("record_reasons"),
        ).alias("reasons"),
        F.col("duplicate_extra_records").alias("record_count"),
    )

    return {
        "reference": reference,
        "audit": representatives.unionByName(duplicate_copies),
    }

### 7.6 Process selected reference records

Process the reference records selected for the regional profile using shared
typing, classification, lineage, key-resolution and output functions.
Check that audit dispositions account for every selected input record and
that eligible audit counts match the reference-table counts.

The summaries report retained records, quality flags and source-record
dispositions. These outputs describe the selected reference snapshots;
they do not establish historical operation throughout the analysis year.

In [32]:
def process_reference_records(records, source, reference_name, environment_root):
    """Process selected raw reference records and reconcile their audit."""
    expected_provider, expected_dataset = reference_name.split("_", 1)
    manifest = json.loads(
        Path(source["manifest_path"]).read_text(encoding="utf-8")
    )
    if (
        source["request"].get("source") != expected_provider
        or source["request"].get("dataset") != expected_dataset
        or manifest["source"].get("provider") != expected_provider
        or manifest["source"].get("dataset") != expected_dataset
        or source["schema"]["status"] != "compatible"
        or manifest["schema"]["status"] != "compatible"
    ):
        raise ValueError(f"{reference_name}: incompatible Bronze source.")

    input_records = records.count()
    source_columns = records.columns
    typed = type_reference_fields(records, reference_name)
    classified = classify_reference_fields(typed, reference_name)
    with_lineage = attach_bronze_lineage(
        classified, source, environment_root, source_columns,
    )
    resolved = resolve_reference_keys(
        with_lineage,
        key_columns=list(reference_output_specs[reference_name]["keys"]),
    )
    result = build_reference_outputs(resolved, reference_name, source_columns)

    output_summary = result["reference"].agg(
        F.count("*").alias("output_records"),
        F.count(F.when(F.size("quality_flags") > 0, 1)).alias("flagged_records"),
    ).first().asDict()
    dispositions = {
        row["disposition"]: row["source_records"]
        for row in result["audit"].groupBy("disposition").agg(
            F.sum("record_count").alias("source_records")
        ).collect()
    }
    if sum(dispositions.values()) != input_records:
        raise ValueError(f"{reference_name}: audit does not account for all inputs.")
    if dispositions.get("eligible", 0) != output_summary["output_records"]:
        raise ValueError(f"{reference_name}: eligible audit and output counts differ.")

    result["summary"] = {
        "input_records": input_records,
        **output_summary,
        "dispositions": dispositions,
    }
    return result


reference_results = {}
reference_tables = {}
reference_audits = {}
reference_output_summary = []
reference_audit_summary = []

for reference_name, _, _, _ in reference_specs:
    result = process_reference_records(
        records=reference_profile_rows[reference_name],
        source=reference_sources[reference_name],
        reference_name=reference_name,
        environment_root=silver_context.environment_root,
    )
    reference_results[reference_name] = result
    reference_tables[reference_name] = result["reference"]
    reference_audits[reference_name] = result["audit"]
    summary = result["summary"]

    reference_output_summary.append((
        reference_name, summary["input_records"],
        summary["output_records"], summary["flagged_records"],
        sum(summary["dispositions"].values()),
    ))
    reference_audit_summary.extend(
        (reference_name, disposition, count)
        for disposition, count in sorted(summary["dispositions"].items())
    )

spark.createDataFrame(
    reference_output_summary,
    "reference STRING, input_records LONG, output_records LONG, "
    "flagged_records LONG, audited_source_records LONG",
).orderBy("reference").show(truncate=False)

spark.createDataFrame(
    reference_audit_summary,
    "reference STRING, disposition STRING, source_records LONG",
).orderBy("reference", "disposition").show(truncate=False)

if show_diagnostics:
    for reference_name, reference_table in reference_tables.items():
        print(reference_name)
        reference_table.select(
            reference_output_specs[reference_name]["id_column"],
            *reference_type_specs[reference_name],
            "quality_flags",
        ).show(truncate=False, vertical=True)

+-------------+-------------+--------------+---------------+----------------------+
|reference    |input_records|output_records|flagged_records|audited_source_records|
+-------------+-------------+--------------+---------------+----------------------+
|epa_monitors |1            |1             |0              |1                     |
|epa_sites    |1            |1             |0              |1                     |
|noaa_stations|1            |1             |0              |1                     |
+-------------+-------------+--------------+---------------+----------------------+

+-------------+-----------+--------------+
|reference    |disposition|source_records|
+-------------+-----------+--------------+
|epa_monitors |eligible   |1             |
|epa_sites    |eligible   |1             |
|noaa_stations|eligible   |1             |
+-------------+-----------+--------------+



## 8. Source Silver publication

### 8.1 Output contracts

Define the schemas, keys and required fields for five clean tables and three
audit tables. Temperature retains its original report timestamps. The reference
audits share one dataset, identified by reference type and source lineage.

Each selected reference must produce exactly one eligible entity. Audit keys
identify a source-record fingerprint and disposition; record counts include
additional exact copies.

In [33]:
def silver_column_types(data_type, names):
    return {name: data_type for name in names.split()}


silver_lineage_types = silver_column_types(
    "string",
    "source_provider source_dataset source_version_sha256 source_file_path "
    "source_file_sha256 source_manifest_path source_record_hash_scheme "
    "source_record_sha256",
)

silver_audit_types = {
    **silver_lineage_types,
    **silver_column_types("string", "field_disposition disposition"),
    **silver_column_types("array<string>", "field_reasons reasons"),
    "raw_fields": "map<string,string>",
    "field_flags": "map<string,boolean>",
    "key_variant_count": "bigint",
    "key_conflict": "boolean",
    "record_count": "bigint",
}

source_silver_schemas = {
    "ozone_observations": {
        **silver_lineage_types,
        **silver_column_types(
            "string",
            "state_code county_code site_number parameter_code poc monitor_id "
            "ozone_raw source_unit mdl_raw qualifier_code qualifier_raw "
            "method_type method_code method_name",
        ),
        "timestamp_utc": "timestamp",
        "ozone_ppm": "double",
        "mdl_ppm": "double",
        "quality_flags": "array<string>",
        "source_occurrences": "bigint",
    },
    "temperature_reports": {
        **silver_lineage_types,
        **silver_column_types(
            "string",
            "station_id temperature_raw timestamp_raw source_code quality_code "
            "report_type measurement_code source_station_id source_code_raw "
            "quality_code_raw report_type_raw measurement_code_raw "
            "source_station_id_raw",
        ),
        "timestamp_utc": "timestamp",
        "temperature_c": "double",
        "quality_flags": "array<string>",
        "source_occurrences": "bigint",
        "supporting_source_records": "bigint",
    },
    "ozone_audit": {
        **silver_audit_types,
        "monitor_id": "string",
        "timestamp_utc": "timestamp",
    },
    "temperature_audit": {
        **silver_audit_types,
        "station_id": "string",
        "timestamp_utc": "timestamp",
        "key_variant_count": "int",
        "key_source_records": "bigint",
        "redundant_temperature": "boolean",
    },
    "reference_audit": {
        **silver_audit_types,
        "reference": "string",
        "entity_id": "string",
        "reference_quality_flags": "array<string>",
    },
}

for reference_name, spec in reference_output_specs.items():
    source_silver_schemas[reference_name] = {
        **silver_lineage_types,
        **{name: "string" for name in spec["keys"].values()},
        spec["id_column"]: "string",
        **{
            name: data_type
            for name, (_, data_type) in reference_type_specs[reference_name].items()
        },
        **{name: "string" for name in spec["text"].values()},
        "quality_flags": "array<string>",
        "source_occurrences": "bigint",
    }

reference_audit_combined = reference_audits["epa_sites"].unionByName(
    reference_audits["epa_monitors"]
).unionByName(reference_audits["noaa_stations"])

source_silver_tables = {
    "ozone_observations": ozone_observations,
    "temperature_reports": temperature_observations,
    **reference_tables,
    "ozone_audit": ozone_audit,
    "temperature_audit": temperature_audit,
    "reference_audit": reference_audit_combined,
}

source_silver_rule_versions = {
    "ozone": "ozone-source-v1",
    "temperature": "temperature-source-v1",
    "references": "reference-source-v1",
}


def source_silver_contract(name, key, required, expected_rows=None):
    return {
        "schema_version": f"{name}-v1",
        "columns": source_silver_schemas[name],
        "key": key,
        "required": list(dict.fromkeys([*key, *required, *silver_lineage_types])),
        "expected_rows": expected_rows,
    }


source_silver_contracts = {
    "ozone_observations": source_silver_contract(
        "ozone_observations",
        ["monitor_id", "timestamp_utc"],
        "state_code county_code site_number parameter_code poc ozone_ppm "
        "source_unit quality_flags source_occurrences".split(),
        ozone_result_summary["after"]["records"],
    ),
    "temperature_reports": source_silver_contract(
        "temperature_reports",
        ["station_id", "timestamp_utc"],
        "temperature_c source_code quality_code report_type quality_flags "
        "source_occurrences supporting_source_records".split(),
        temperature_result_summary["after"]["records"],
    ),
}

for reference_name, spec in reference_output_specs.items():
    source_silver_contracts[reference_name] = source_silver_contract(
        reference_name,
        [spec["id_column"]],
        [
            *spec["keys"].values(), "latitude_deg", "longitude_deg",
            "quality_flags", "source_occurrences",
        ],
        expected_rows=1,
    )

silver_audit_key = [
    "source_provider", "source_dataset", "source_version_sha256",
    "source_file_path", "source_file_sha256", "source_record_sha256",
    "disposition",
]
for dataset in ("ozone_audit", "temperature_audit", "reference_audit"):
    required = [
        "raw_fields", "field_flags", "field_disposition", "field_reasons",
        "key_conflict", "reasons", "record_count",
    ]
    if dataset == "reference_audit":
        required += ["reference", "reference_quality_flags"]
    source_silver_contracts[dataset] = source_silver_contract(
        dataset, silver_audit_key.copy(), required,
    )

if set(source_silver_tables) != set(source_silver_contracts):
    raise ValueError("Silver tables and contracts do not match.")

for dataset, table in source_silver_tables.items():
    expected = source_silver_contracts[dataset]["columns"]
    if len(table.columns) != len(set(table.columns)) or dict(table.dtypes) != expected:
        raise ValueError(
            f"{dataset}: schema differs from its contract. "
            f"Expected {expected}; received {dict(table.dtypes)}"
        )

print(f"Source Silver schemas checked: {len(source_silver_tables)} datasets.")

Source Silver schemas checked: 8 datasets.


### 8.2 Stage Parquet outputs

Write Snappy-compressed Parquet into a new directory beneath the run's staging
root. Use one write partition per dataset for the current small outputs.

Temporarily cache each table for counting and writing, then release the cache
created by this function. Record schemas, row counts, file sizes and checksums
for read-back validation. Staged outputs remain unpublished.

In [34]:
def stage_source_silver(tables, contracts, context):
    """Write a new unpublished Parquet bundle and record its file inventory."""
    if set(tables) != set(contracts):
        raise ValueError("Tables and contracts must have the same dataset names.")

    root = Path(context.environment_root).resolve()
    parent = Path(context.staging_root).resolve()
    if parent == root / "staging" or not parent.is_relative_to(root / "staging"):
        raise ValueError("Staging must be inside this environment's run directory.")

    for name, table in tables.items():
        if (
            not name.isidentifier()
            or len(table.columns) != len(set(table.columns))
            or dict(table.dtypes) != contracts[name]["columns"]
        ):
            raise ValueError(f"{name}: invalid dataset name or schema.")

    stage_path = parent / f"source-silver-{time.time_ns()}"
    stage_path.mkdir(parents=True, exist_ok=False)
    started = time.perf_counter()
    created_at_utc = datetime.now(timezone.utc).isoformat()
    datasets = {}

    for name, table in tables.items():
        destination = stage_path / name
        owned_cache = not table.is_cached
        if owned_cache:
            table.cache()
        try:
            row_count = table.count()
            table.coalesce(1).write.parquet(
                str(destination), mode="errorifexists", compression="snappy",
            )
        finally:
            if owned_cache:
                table.unpersist(blocking=True)

        files = [
            {
                "path": path.relative_to(stage_path).as_posix(),
                "bytes": path.stat().st_size,
                "sha256": file_sha256(path),
            }
            for path in sorted(destination.glob("*.parquet"))
        ]
        if not files:
            raise ValueError(f"{name}: no Parquet files were produced.")

        datasets[name] = {
            "path": name,
            "row_count": row_count,
            "schema": table.schema.jsonValue(),
            "files": files,
            "bytes": sum(item["bytes"] for item in files),
        }
        print(f"Staged {name}: {row_count} rows, {len(files)} Parquet file(s).")

    contract_json = json.dumps(contracts, sort_keys=True, separators=(",", ":"))
    return {
        "status": "staged",
        "created_at_utc": created_at_utc,
        "run_id": context.run_id,
        "environment": context.environment,
        "environment_root": str(root),
        "bundle_path": stage_path.relative_to(root).as_posix(),
        "format": "parquet",
        "compression": "snappy",
        "write_partitions": 1,
        "partition_columns": [],
        "contract_sha256": hashlib.sha256(contract_json.encode("utf-8")).hexdigest(),
        "datasets": datasets,
        "elapsed_seconds": round(time.perf_counter() - started, 3),
    }


source_silver_stage = None
source_silver_stage = stage_source_silver(
    source_silver_tables, source_silver_contracts, silver_context,
)

print("Status:", source_silver_stage["status"])
print("Staging directory:", (
    silver_context.environment_root / source_silver_stage["bundle_path"]
))
print("Staging time (seconds):", source_silver_stage["elapsed_seconds"])

spark.createDataFrame(
    [
        (name, info["row_count"], len(info["files"]), info["bytes"])
        for name, info in source_silver_stage["datasets"].items()
    ],
    "dataset STRING, records LONG, parquet_files INT, parquet_bytes LONG",
).orderBy("dataset").show(truncate=False)

Staged ozone_observations: 5071 rows, 1 Parquet file(s).


Staged temperature_reports: 11686 rows, 1 Parquet file(s).
Staged epa_sites: 1 rows, 1 Parquet file(s).


Staged epa_monitors: 1 rows, 1 Parquet file(s).
Staged noaa_stations: 1 rows, 1 Parquet file(s).


Staged ozone_audit: 5166 rows, 1 Parquet file(s).


26/11/09 23:14:28 WARN DAGScheduler: Broadcasting large task binary with size 1038.9 KiB


Staged temperature_audit: 11851 rows, 1 Parquet file(s).


26/11/09 23:14:34 WARN DAGScheduler: Broadcasting large task binary with size 1057.9 KiB


Staged reference_audit: 3 rows, 1 Parquet file(s).
Status: staged
Staging directory: /home/ubuntu/dat535-final-project-data/dev/staging/26a8250b3c0a422dab4875f5e49d86d8/source-silver-1794266009649468161
Staging time (seconds): 64.559
+-------------------+-------+-------------+-------------+
|dataset            |records|parquet_files|parquet_bytes|
+-------------------+-------+-------------+-------------+
|epa_monitors       |1      |1            |10524        |
|epa_sites          |1      |1            |9970         |
|noaa_stations      |1      |1            |8094         |
|ozone_audit        |5166   |1            |444616       |
|ozone_observations |5071   |1            |385139       |
|reference_audit    |3      |1            |11385        |
|temperature_audit  |11851  |1            |3128825      |
|temperature_reports|11686  |1            |944457       |
+-------------------+-------+-------------+-------------+



### 8.3 Validate persisted source Silver

Read the staged Parquet files and verify their inventory, checksums, schemas,
row counts, required values and unique keys. Reconcile persisted audit
dispositions and quality flags with the processing summaries.

Check that source lineage matches the selected Bronze manifests and that each
retained record has a matching eligible audit entry. Successful validation
produces a separate report; the bundle remains unpublished.

In [35]:
def silver_json_sha256(value):
    encoded = json.dumps(value, sort_keys=True, separators=(",", ":")).encode("utf-8")
    return hashlib.sha256(encoded).hexdigest()


def selected_bronze_lineage(source, environment_root):
    """Read the expected file identity from the selected Bronze manifest."""
    root = Path(environment_root).resolve()
    manifest_path = Path(source["manifest_path"]).resolve(strict=True)
    data_path = Path(source["data_path"]).resolve(strict=True)
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest["manifest_version"] != 1 or manifest["schema"]["status"] != "compatible":
        raise ValueError("The selected Bronze manifest is incompatible.")

    relative_data_path = data_path.relative_to(root).as_posix()
    matches = [
        item for item in [manifest["original_file"], *manifest["extracted_files"]]
        if item["path"] == relative_data_path
    ]
    if len(matches) != 1:
        raise ValueError("The selected Bronze file must match one manifest entry.")

    return {
        "source_provider": manifest["source"]["provider"],
        "source_dataset": manifest["source"]["dataset"],
        "source_version_sha256": manifest["original_file"]["sha256"],
        "source_file_path": relative_data_path,
        "source_file_sha256": matches[0]["sha256"],
        "source_manifest_path": manifest_path.relative_to(root).as_posix(),
        "source_record_hash_scheme": "sha256-json-raw-fields-v1",
    }


def validate_source_silver(stage, contracts, context, inputs):
    """Validate staged files, table contracts, lineage and audit reconciliation."""
    root = Path(context.environment_root).resolve()
    if (
        stage["status"] != "staged"
        or Path(stage["environment_root"]).resolve() != root
        or stage["environment"] != context.environment
        or stage["run_id"] != context.run_id
        or stage["contract_sha256"] != silver_json_sha256(contracts)
        or set(stage["datasets"]) != set(contracts)
    ):
        raise ValueError("The staged bundle does not match this run and its contracts.")

    def contained_path(base, relative):
        relative = Path(relative)
        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(f"Expected a contained relative path: {relative}")
        path = (base / relative).resolve(strict=True)
        if path == base or not path.is_relative_to(base):
            raise ValueError(f"Path is outside the expected directory: {path}")
        return path

    bundle = contained_path(root, stage["bundle_path"])
    staging_root = Path(context.staging_root).resolve()
    if bundle == staging_root or not bundle.is_relative_to(staging_root):
        raise ValueError("The bundle is outside this run's staging directory.")

    readback, checks = {}, {}
    for name, contract in contracts.items():
        info = stage["datasets"][name]
        directory = contained_path(bundle, info["path"])
        paths = [contained_path(bundle, item["path"]) for item in info["files"]]
        if (
            not paths
            or len(paths) != len(set(paths))
            or any(not path.is_file() or not path.is_relative_to(directory) for path in paths)
            or set(paths) != {path.resolve() for path in directory.rglob("*.parquet")}
            or info["bytes"] != sum(item["bytes"] for item in info["files"])
        ):
            raise ValueError(f"{name}: the Parquet file inventory does not match.")

        for path, item in zip(paths, info["files"]):
            if path.stat().st_size != item["bytes"] or file_sha256(path) != item["sha256"]:
                raise ValueError(f"{name}: file size or checksum mismatch: {path.name}")

        table = (
            spark.read
            .option("mergeSchema", "true")
            .option("ignoreCorruptFiles", "false")
            .option("ignoreMissingFiles", "false")
            .parquet(*[str(path) for path in paths])
        )
        stored_columns = [field["name"] for field in info["schema"]["fields"]]
        if table.columns != stored_columns or dict(table.dtypes) != contract["columns"]:
            raise ValueError(f"{name}: persisted column names or types changed.")

        bad_required = F.lit(False)
        for column in contract["required"]:
            missing = F.col(column).isNull()
            if contract["columns"][column] == "string":
                missing = missing | (F.trim(F.col(column)) == "")
            bad_required = bad_required | missing

        bad_numeric, bad_array, bad_hash, bad_count = (F.lit(False) for _ in range(4))
        for column, data_type in contract["columns"].items():
            value = F.col(column)
            if data_type in {"float", "double"}:
                bad_numeric = bad_numeric | (
                    value.isNotNull() & ~finite_number(column)
                )
            if data_type == "array<string>":
                bad_array = bad_array | F.coalesce(
                    F.exists(value, lambda item: item.isNull()), F.lit(False),
                )
            if column.endswith("_sha256"):
                bad_hash = bad_hash | ~F.coalesce(
                    value.rlike("^[0-9a-f]{64}$"), F.lit(False),
                )
        for column in (
            "source_occurrences", "supporting_source_records",
            "record_count", "key_variant_count", "key_source_records",
        ):
            if column in table.columns:
                bad_count = bad_count | (
                    F.col(column).isNotNull() & (F.col(column) < 1)
                )
        if "record_count" in table.columns:
            bad_count = bad_count | (
                (F.col("disposition") != "exact_duplicate")
                & (F.col("record_count") != 1)
            )

        failures = {
            "missing_required": bad_required, "nonfinite_numbers": bad_numeric,
            "null_array_items": bad_array, "invalid_hashes": bad_hash,
            "invalid_counts": bad_count,
        }
        # Decode every column, including nested audit fields.
        decoded = F.to_json(
            F.struct(*[F.col(column) for column in table.columns]),
            options={"ignoreNullFields": "false"},
        )
        measured = table.agg(
            F.count("*").alias("records"),
            F.sum(F.length(decoded)).alias("decoded_characters"),
            *[
                F.count(F.when(condition, 1)).alias(label)
                for label, condition in failures.items()
            ],
        ).first().asDict()
        if any(measured[label] for label in failures):
            raise ValueError(f"{name}: persisted value checks failed: {measured}")
        if (
            measured["records"] != info["row_count"]
            or (
                contract["expected_rows"] is not None
                and measured["records"] != contract["expected_rows"]
            )
        ):
            raise ValueError(f"{name}: persisted row count does not match.")

        if table.groupBy(*contract["key"]).count().filter("count > 1").limit(1).count():
            raise ValueError(f"{name}: duplicate output keys.")

        readback[name] = table
        checks[name] = {
            "records": measured["records"], "parquet_files": len(paths),
            "status": "passed",
        }

    audit_names = {item["audit"] for item in inputs.values()}
    if (set(inputs) & audit_names) or (set(inputs) | audit_names) != set(readback):
        raise ValueError("Every persisted dataset must have one validation role.")

    audit_checks, accounted_audit_rows = {}, Counter()
    lineage_columns = [*silver_lineage_types]
    for name, expected in inputs.items():
        clean = readback[name]
        audit_name = expected["audit"]
        audit = readback[audit_name]
        if "reference" in expected:
            audit = audit.filter(F.col("reference") == F.lit(expected["reference"]))

        audit_rows = audit.groupBy("disposition").agg(
            F.sum("record_count").alias("source_records"),
            F.count("*").alias("audit_rows"),
        ).collect()
        dispositions = {row["disposition"]: row["source_records"] for row in audit_rows}
        accounted_audit_rows[audit_name] += sum(row["audit_rows"] for row in audit_rows)
        if (
            dispositions != expected["dispositions"]
            or sum(dispositions.values()) != expected["input_records"]
            or dispositions.get("eligible", 0) != checks[name]["records"]
        ):
            raise ValueError(f"{name}: persisted audit counts do not reconcile.")

        if "quality_flags" in expected:
            flags = {
                row["quality_flag"]: row["count"]
                for row in clean.select(
                    F.explode("quality_flags").alias("quality_flag")
                ).groupBy("quality_flag").count().collect()
            }
            if flags != expected["quality_flags"]:
                raise ValueError(f"{name}: persisted quality-flag counts differ.")
        if "flagged_records" in expected:
            if clean.filter(F.size("quality_flags") > 0).count() != expected["flagged_records"]:
                raise ValueError(f"{name}: persisted flagged-record count differs.")

        lineage = selected_bronze_lineage(expected["source"], root)
        mismatch = F.lit(False)
        for column, value in lineage.items():
            mismatch = mismatch | ~F.col(column).eqNullSafe(F.lit(value))
        combined_lineage = clean.select(*lineage).unionByName(audit.select(*lineage))
        if combined_lineage.filter(mismatch).limit(1).count():
            raise ValueError(f"{name}: lineage differs from the selected Bronze source.")

        accepted = audit.filter(F.col("disposition") == "eligible").select(*lineage_columns)
        observed = clean.select(*lineage_columns)
        differences = observed.exceptAll(accepted).unionByName(accepted.exceptAll(observed))
        if differences.limit(1).count():
            raise ValueError(f"{name}: output and eligible-audit lineage differ.")

        audit_checks[name] = {
            "input_records": expected["input_records"],
            "dispositions": dispositions,
            "lineage": lineage,
            "status": "passed",
        }

    if any(accounted_audit_rows[name] != checks[name]["records"] for name in audit_names):
        raise ValueError("Some audit rows are unaccounted for or counted more than once.")

    report = {
        "status": "validated",
        "validated_at_utc": datetime.now(timezone.utc).isoformat(),
        "run_id": context.run_id,
        "bundle_path": stage["bundle_path"],
        "stage_sha256": silver_json_sha256(stage),
        "contract_sha256": stage["contract_sha256"],
        "datasets": checks,
        "audits": audit_checks,
    }
    return report, readback


source_silver_validation_inputs = {
    "ozone_observations": {
        "source": ozone_source, "audit": "ozone_audit",
        "input_records": ozone_result_summary["before"]["records"],
        "dispositions": ozone_result_summary["dispositions"],
        "quality_flags": ozone_result_summary["quality_flags"],
    },
    "temperature_reports": {
        "source": temperature_source, "audit": "temperature_audit",
        "input_records": temperature_result_summary["before"]["records"],
        "dispositions": temperature_result_summary["dispositions"],
        "quality_flags": temperature_result_summary["quality_flags"],
    },
}
for reference_name, result in reference_results.items():
    source_silver_validation_inputs[reference_name] = {
        "source": reference_sources[reference_name],
        "audit": "reference_audit", "reference": reference_name,
        "input_records": result["summary"]["input_records"],
        "dispositions": result["summary"]["dispositions"],
        "flagged_records": result["summary"]["flagged_records"],
    }

source_silver_validation = None
source_silver_readback = None
source_silver_validation, source_silver_readback = validate_source_silver(
    source_silver_stage, source_silver_contracts, silver_context,
    source_silver_validation_inputs,
)

spark.createDataFrame(
    [
        (name, result["records"], result["parquet_files"], result["status"])
        for name, result in source_silver_validation["datasets"].items()
    ],
    "dataset STRING, records LONG, parquet_files INT, validation STRING",
).orderBy("dataset").show(truncate=False)

print("Audit and lineage reconciliations passed:", len(source_silver_validation["audits"]))
print("Validation status:", source_silver_validation["status"])
print("Publication status: unpublished")

+-------------------+-------+-------------+----------+
|dataset            |records|parquet_files|validation|
+-------------------+-------+-------------+----------+
|epa_monitors       |1      |1            |passed    |
|epa_sites          |1      |1            |passed    |
|noaa_stations      |1      |1            |passed    |
|ozone_audit        |5166   |1            |passed    |
|ozone_observations |5071   |1            |passed    |
|reference_audit    |3      |1            |passed    |
|temperature_audit  |11851  |1            |passed    |
|temperature_reports|11686  |1            |passed    |
+-------------------+-------+-------------+----------+

Audit and lineage reconciliations passed: 5
Validation status: validated
Publication status: unpublished


### 8.4 Define Silver publication

Publish validated source Silver tables as a versioned Parquet bundle. Recheck
the file inventory and checksums before moving the data, then write a manifest
containing the schemas, source lineage, processing rules and validation results.
The manifest marks the bundle as available for downstream processing.

In [36]:
def silver_child_path(base, relative, *, must_exist=False):
    """Resolve a relative path without leaving its base directory."""
    base = Path(base).resolve()
    relative = Path(relative)
    if relative.is_absolute() or ".." in relative.parts:
        raise ValueError(f"Expected a contained relative path: {relative}")
    result = (base / relative).resolve(strict=must_exist)
    if result == base or not result.is_relative_to(base):
        raise ValueError(f"Path is outside its base directory: {result}")
    return result


def verify_silver_file_inventory(bundle, datasets):
    """Check the recorded Parquet files and return their absolute paths."""
    bundle = Path(bundle).resolve(strict=True)
    verified = {}
    for name, info in datasets.items():
        if info["path"] != name:
            raise ValueError(f"{name}: unexpected dataset directory.")
        directory = silver_child_path(bundle, info["path"], must_exist=True)
        paths = [
            silver_child_path(bundle, item["path"], must_exist=True)
            for item in info["files"]
        ]
        if (
            not paths
            or len(paths) != len(set(paths))
            or any(
                not path.is_file() or not path.is_relative_to(directory)
                or path.suffix != ".parquet"
                for path in paths
            )
            or set(paths) != {
                path.resolve() for path in directory.rglob("*.parquet")
            }
            or info["bytes"] != sum(item["bytes"] for item in info["files"])
        ):
            raise ValueError(f"{name}: the Parquet file inventory differs.")

        for path, item in zip(paths, info["files"]):
            if (
                path.stat().st_size != item["bytes"]
                or file_sha256(path) != item["sha256"]
            ):
                raise ValueError(
                    f"{name}: file size or checksum differs: {path.name}"
                )
        verified[name] = [str(path) for path in paths]
    return verified


def publish_source_silver(
    stage, validation, contracts, context, *, profile, rule_versions, provenance,
):
    """Publish a validated bundle, using its manifest as the commit marker."""
    # Freeze the metadata and require JSON-compatible values.
    stage, validation, contracts, profile, rule_versions, provenance = json.loads(
        json.dumps(
            [stage, validation, contracts, profile, rule_versions, provenance],
            allow_nan=False,
        )
    )
    root = Path(context.environment_root).resolve()
    stage_hash = silver_json_sha256(stage)
    contract_hash = silver_json_sha256(contracts)
    if (
        stage["status"] != "staged"
        or Path(stage["environment_root"]).resolve() != root
        or stage["environment"] != context.environment
        or stage["run_id"] != context.run_id
        or stage["contract_sha256"] != contract_hash
        or validation["status"] != "validated"
        or validation["run_id"] != context.run_id
        or validation["bundle_path"] != stage["bundle_path"]
        or validation["stage_sha256"] != stage_hash
        or validation["contract_sha256"] != contract_hash
        or set(stage["datasets"]) != set(contracts)
        or set(validation["datasets"]) != set(contracts)
    ):
        raise ValueError(
            "Publication requires a matching stage, validation and contracts."
        )

    for name, info in stage["datasets"].items():
        check = validation["datasets"][name]
        if (
            check["status"] != "passed"
            or check["records"] != info["row_count"]
            or check["parquet_files"] != len(info["files"])
        ):
            raise ValueError(
                f"{name}: validation is incomplete or inconsistent."
            )

    clean_names = {
        "ozone_observations", "temperature_reports",
        "epa_sites", "epa_monitors", "noaa_stations",
    }
    if (
        set(validation["audits"]) != clean_names
        or any(
            item["status"] != "passed"
            for item in validation["audits"].values()
        )
    ):
        raise ValueError(
            "All five audit and lineage reconciliations must pass."
        )

    region = profile["region"].strip().lower().replace(" ", "_")
    year = profile["year"]
    if (
        not region
        or any(
            character not in "abcdefghijklmnopqrstuvwxyz0123456789_-"
            for character in region
        )
        or type(year) is not int
        or not 1 <= year <= 9999
        or not rule_versions
        or not provenance
    ):
        raise ValueError(
            "A valid region, year, rule versions and provenance are required."
        )

    staged_bundle = silver_child_path(root, stage["bundle_path"])
    staging_root = Path(context.staging_root).resolve()
    if (
        not staging_root.is_relative_to(root / "staging")
        or staged_bundle == staging_root
        or not staged_bundle.is_relative_to(staging_root)
    ):
        raise ValueError(
            "The bundle is outside this run's staging directory."
        )

    publication_id = stage_hash
    suffix = f"silver/source/{region}/{year}/{publication_id}"
    final_bundle = silver_child_path(root, f"data/{suffix}")
    manifest_path = silver_child_path(
        root, f"published/{suffix}/manifest.json"
    )
    publication_root = silver_child_path(root, "published/silver/source")

    identity = {
        "manifest_version": 1,
        "kind": "source_silver",
        "status": "published",
        "publication_id": publication_id,
        "run_id": context.run_id,
        "environment": context.environment,
        "bundle_path": final_bundle.relative_to(root).as_posix(),
        "stage_sha256": stage_hash,
        "contract_sha256": contract_hash,
        "profile": profile,
        "rule_versions": rule_versions,
        "provenance": provenance,
        "contracts": contracts,
        "storage": {
            key: stage[key]
            for key in (
                "format", "compression", "write_partitions",
                "partition_columns",
            )
        },
        "datasets": stage["datasets"],
    }

    publication_root.mkdir(parents=True, exist_ok=True)
    lock = publication_root / ".publish.lock"
    try:
        lock.mkdir()
    except FileExistsError as error:
        raise RuntimeError(
            "Publication lock already exists; inspect it before retrying: "
            f"{lock}"
        ) from error

    try:
        if manifest_path.exists():
            existing = json.loads(
                manifest_path.read_text(encoding="utf-8")
            )
            if {key: existing.get(key) for key in identity} != identity:
                raise ValueError(
                    "The existing publication has different metadata."
                )

            recorded_validation = existing.get("validation", {})
            if any(
                recorded_validation.get(key) != value
                for key, value in validation.items()
                if key != "validated_at_utc"
            ):
                raise ValueError(
                    "The existing publication has different validation results."
                )

            verify_silver_file_inventory(
                final_bundle, existing["datasets"]
            )
            return {
                "outcome": "reused",
                "manifest_path": str(manifest_path),
                "manifest": existing,
            }

        if final_bundle.exists() or final_bundle.is_symlink():
            raise FileExistsError(
                f"Data exists without a publication manifest: {final_bundle}. "
                "Inspect the incomplete attempt, then stage and validate "
                "a new attempt."
            )

        verify_silver_file_inventory(
            staged_bundle, stage["datasets"]
        )
        final_bundle.parent.mkdir(parents=True, exist_ok=True)
        manifest_path.parent.mkdir(parents=True, exist_ok=True)
        if staged_bundle.stat().st_dev != final_bundle.parent.stat().st_dev:
            raise ValueError(
                "Staging and final data must be on the same filesystem."
            )

        manifest = {
            **identity,
            "published_at_utc": datetime.now(timezone.utc).isoformat(),
            "validation": validation,
        }

        # Readers may use the bundle only after its manifest exists.
        staged_bundle.rename(final_bundle)
        write_json(manifest_path, manifest)

        return {
            "outcome": "published",
            "manifest_path": str(manifest_path),
            "manifest": manifest,
        }
    finally:
        lock.rmdir()


print("Source Silver publication helper ready. No data published.")

Source Silver publication helper ready. No data published.


### 8.5 Verification fixtures

Run the reader, measurement, reference and publication tests defined in
`silver_checks.ipynb` when `PIPELINE_RUN_FIXTURES=true`. Failed checks stop
execution. Record the results and the companion notebook's code hash with the
staged bundle.

In [37]:
def notebook_code_sha256(notebook):
    """Hash saved code cells without outputs or execution counts."""
    return silver_json_sha256([
        "".join(cell["source"])
        for cell in notebook["cells"]
        if cell["cell_type"] == "code"
    ])


source_silver_checks = None
fixture_results = {}
fixture_setting = os.environ.get(
    "PIPELINE_RUN_FIXTURES", "false"
).lower()
if fixture_setting not in {"true", "false"}:
    raise ValueError("PIPELINE_RUN_FIXTURES must be true or false.")

fixture_code_hash = None
if fixture_setting == "true":
    fixture_notebook_path = (
        project_dir / "notebooks" / "silver_checks.ipynb"
    )
    fixture_notebook = json.loads(
        fixture_notebook_path.read_text(encoding="utf-8")
    )
    fixture_code_hash = notebook_code_sha256(fixture_notebook)
    fixture_namespace = {"__name__": "silver_fixture_checks"}

    for index, cell in enumerate(fixture_notebook["cells"]):
        if cell["cell_type"] == "code":
            exec(
                compile(
                    "".join(cell["source"]),
                    f"{fixture_notebook_path}:cell-{index}",
                    "exec",
                ),
                fixture_namespace,
            )

    fixture_results = {
        "readers": fixture_namespace["verify_delimited_readers"](
            spark=spark,
            staging_root=silver_context.staging_root,
            read_validated_ozone_csv=read_validated_ozone_csv,
            read_validated_delimited_file=read_validated_delimited_file,
        ),
        "ozone": fixture_namespace["verify_ozone_rules"](
            spark=spark,
            type_ozone_fields=type_ozone_fields,
            ozone_field_checks=ozone_field_checks,
            classify_ozone_fields=classify_ozone_fields,
            resolve_source_keys=resolve_source_keys,
            ozone_key_columns=ozone_key_columns,
            build_ozone_observations=build_ozone_observations,
            build_ozone_audit=build_ozone_audit,
        ),
        "temperature": fixture_namespace["verify_temperature_rules"](
            spark=spark,
            profile=alabama_profile,
            temperature_columns=temperature_inspection_columns,
            type_temperature_fields=type_temperature_fields,
            temperature_field_checks=temperature_field_checks,
            classify_temperature_fields=classify_temperature_fields,
            resolve_temperature_keys=resolve_temperature_keys,
            build_temperature_observations=build_temperature_observations,
            build_temperature_audit=build_temperature_audit,
        ),
        "reference_fields": fixture_namespace["verify_reference_fields"](
            spark=spark,
            type_reference_fields=type_reference_fields,
            classify_reference_fields=classify_reference_fields,
        ),
        "reference_outputs": fixture_namespace["verify_reference_outputs"](
            spark=spark,
            type_reference_fields=type_reference_fields,
            classify_reference_fields=classify_reference_fields,
            resolve_reference_keys=resolve_reference_keys,
            build_reference_outputs=build_reference_outputs,
        ),
        "publication": fixture_namespace["verify_silver_publication"](
            stage=source_silver_stage,
            validation=source_silver_validation,
            contracts=source_silver_contracts,
            context=silver_context,
            profile=alabama_profile,
            rule_versions=source_silver_rule_versions,
            publish_source_silver=publish_source_silver,
            verify_silver_file_inventory=verify_silver_file_inventory,
            silver_json_sha256=silver_json_sha256,
        ),
    }
    print(json.dumps(fixture_results, indent=2))
else:
    print("Fixtures skipped: PIPELINE_RUN_FIXTURES=false.")

source_silver_checks = {
    "status": "passed" if fixture_setting == "true" else "skipped",
    "run_id": silver_context.run_id,
    "stage_sha256": silver_json_sha256(source_silver_stage),
    "companion_code_sha256": fixture_code_hash,
    "results": fixture_results,
}

valid: accepted; decoded values verified
empty_final_value: accepted; decoded values verified
quoted_comma: accepted; decoded values verified
escaped_quotes: accepted; decoded values verified
too_few_fields: rejected; reason verified
too_many_fields: rejected; reason verified
unclosed_quote: rejected; reason verified
Ozone CSV reader fixtures passed: 7 cases.
valid: accepted; decoded values verified
empty_final_value: accepted; decoded values verified
quoted_pipe: accepted; decoded values verified
escaped_quotes: accepted; decoded values verified
too_few_fields: rejected; reason verified
too_many_fields: rejected; reason verified
unclosed_quote: rejected; reason verified
multiline_record: rejected; reason verified
Temperature PSV reader fixtures passed: 8 cases.
Ozone field fixtures passed: 17
Ozone key fixtures passed: 12
Ozone audit fixtures passed: 24 audit entries account for 25 source records.
Ozone observation fixtures passed: 19 cases, 16 output records.
Temperature field fixtur

### 8.6 Publish source Silver

Record the saved code fingerprints, runtime and verification results, then
publish the validated Parquet bundle. The publication manifest identifies the
selected Bronze sources, processing rules, output contracts and files.

In [38]:
source_silver_publication = None
if (
    source_silver_checks is None
    or source_silver_checks["run_id"] != silver_context.run_id
    or source_silver_checks["stage_sha256"]
    != silver_json_sha256(source_silver_stage)
    or source_silver_checks["status"] not in {"passed", "skipped"}
):
    raise RuntimeError(
        "Run 8.5 for this staged bundle before publishing."
    )

saved_notebook_hashes = {}
notebook_names = ["pipeline.ipynb"]
if source_silver_checks["status"] == "passed":
    notebook_names.append("silver_checks.ipynb")

for name in notebook_names:
    path = project_dir / "notebooks" / name
    saved = json.loads(path.read_text(encoding="utf-8"))
    saved_notebook_hashes[name] = notebook_code_sha256(saved)

if (
    source_silver_checks["status"] == "passed"
    and saved_notebook_hashes["silver_checks.ipynb"]
    != source_silver_checks["companion_code_sha256"]
):
    raise RuntimeError(
        "Companion code changed after the checks. Rerun 8.5."
    )

source_silver_root = Path(
    silver_context.environment_root
).resolve()
supporting_paths = sorted(
    (project_dir / "src").rglob("*.py")
) + [
    project_dir / "config" / "spark.properties",
    environments_file,
]

source_silver_provenance = {
    **capture_provenance(silver_context),
    "notebook_code_hash_scheme": "sha256-json-code-cells-v1",
    "saved_notebook_code_sha256": saved_notebook_hashes,
    "supporting_files_sha256": {
        path.relative_to(project_dir).as_posix(): file_sha256(path)
        for path in supporting_paths
    },
    "bronze_manifests_sha256": {
        item["lineage"]["source_manifest_path"]: file_sha256(
            silver_child_path(
                source_silver_root,
                item["lineage"]["source_manifest_path"],
                must_exist=True,
            )
        )
        for item in source_silver_validation["audits"].values()
    },
    "runtime": {
        "python": sys.version.split()[0],
        "pyspark": pyspark.__version__,
        "spark": spark.version,
        "master": spark.sparkContext.master,
        "session_timezone": spark.conf.get(
            "spark.sql.session.timeZone"
        ),
    },
    "fixture_checks": source_silver_checks,
}

source_silver_publication = publish_source_silver(
    source_silver_stage,
    source_silver_validation,
    source_silver_contracts,
    silver_context,
    profile=alabama_profile,
    rule_versions=source_silver_rule_versions,
    provenance=source_silver_provenance,
)
source_silver_publication["manifest_sha256"] = file_sha256(
    Path(source_silver_publication["manifest_path"])
)

# The staging paths are no longer valid after publication.
source_silver_readback = None

print("Publication outcome:", source_silver_publication["outcome"])
print(
    "Publication status:",
    source_silver_publication["manifest"]["status"],
)
print(
    "Published datasets:",
    len(source_silver_publication["manifest"]["datasets"]),
)
print("Manifest:", source_silver_publication["manifest_path"])
print(
    "Manifest SHA-256:",
    source_silver_publication["manifest_sha256"],
)
print("Fixture status:", source_silver_checks["status"])

Publication outcome: published
Publication status: published
Published datasets: 8
Manifest: /home/ubuntu/dat535-final-project-data/dev/published/silver/source/alabama/2025/c9ca912a73aa65495afb629806252b65f5a718c43c514feb3c4c50eec3ff7cd2/manifest.json
Manifest SHA-256: 97f9aecb783b3c5b2619965dadcb2445f9a891ce749d6dcf42a16942b23bc976
Fixture status: passed


### 8.7 Load and verify published Silver

Load the eight source Silver tables using the published manifest. Verify the
manifest identity, file checksums, schemas and record counts, then save a
completion report. These published tables provide the inputs for hourly
alignment and integration.

In [39]:
def load_source_silver(
    manifest_path, context, *,
    expected_manifest_sha256, profile, rule_versions,
):
    """Load only the files listed in a validated source Silver manifest."""
    root = Path(context.environment_root).resolve()
    manifest_path = Path(manifest_path).resolve(strict=True)
    if (
        manifest_path.name != "manifest.json"
        or not manifest_path.is_relative_to(
            root / "published" / "silver" / "source"
        )
        or file_sha256(manifest_path) != expected_manifest_sha256
    ):
        raise ValueError(
            "The selected publication manifest path or checksum differs."
        )

    manifest = json.loads(
        manifest_path.read_text(encoding="utf-8")
    )
    validation = manifest["validation"]
    datasets = manifest["datasets"]
    contracts = manifest["contracts"]
    names = {
        "ozone_observations", "temperature_reports",
        "epa_sites", "epa_monitors", "noaa_stations",
        "ozone_audit", "temperature_audit", "reference_audit",
    }
    clean_names = names - {
        "ozone_audit", "temperature_audit", "reference_audit"
    }
    contract_hash = silver_json_sha256(contracts)
    if (
        manifest["manifest_version"] != 1
        or manifest["kind"] != "source_silver"
        or manifest["status"] != "published"
        or manifest["environment"] != context.environment
        or manifest["profile"] != profile
        or manifest["rule_versions"] != rule_versions
        or manifest["storage"]["format"] != "parquet"
        or manifest["publication_id"] != manifest["stage_sha256"]
        or validation["status"] != "validated"
        or validation["run_id"] != manifest["run_id"]
        or validation["stage_sha256"] != manifest["stage_sha256"]
        or manifest["contract_sha256"] != contract_hash
        or validation["contract_sha256"] != contract_hash
        or set(datasets) != names
        or set(contracts) != names
        or set(validation["datasets"]) != names
        or set(validation["audits"]) != clean_names
    ):
        raise ValueError(
            "The manifest does not match the requested source Silver bundle."
        )

    for name, check in validation["audits"].items():
        if (
            check["status"] != "passed"
            or sum(check["dispositions"].values())
            != check["input_records"]
            or check["dispositions"].get("eligible", 0)
            != datasets[name]["row_count"]
        ):
            raise ValueError(
                f"{name}: manifest audit counts do not reconcile."
            )

    region = profile["region"].strip().lower().replace(" ", "_")
    suffix = (
        f"silver/source/{region}/{profile['year']}/"
        f"{manifest['publication_id']}"
    )
    if (
        manifest["bundle_path"] != f"data/{suffix}"
        or manifest_path != silver_child_path(
            root, f"published/{suffix}/manifest.json"
        )
    ):
        raise ValueError(
            "The manifest and data locations do not match their publication."
        )

    bundle = silver_child_path(
        root, manifest["bundle_path"], must_exist=True
    )
    files = verify_silver_file_inventory(bundle, datasets)
    tables, counts = {}, {}

    for name in sorted(names):
        info, contract = datasets[name], contracts[name]
        check = validation["datasets"][name]
        if (
            contract["schema_version"] != f"{name}-v1"
            or check["status"] != "passed"
            or check["records"] != info["row_count"]
            or check["parquet_files"] != len(files[name])
        ):
            raise ValueError(
                f"{name}: unsupported schema or inconsistent validation."
            )

        table = (
            spark.read
            .option("mergeSchema", "true")
            .option("ignoreCorruptFiles", "false")
            .option("ignoreMissingFiles", "false")
            .parquet(*files[name])
        )
        if (
            table.columns != [
                field["name"] for field in info["schema"]["fields"]
            ]
            or dict(table.dtypes) != contract["columns"]
        ):
            raise ValueError(
                f"{name}: published schema differs from its contract."
            )

        count = table.count()
        if (
            count != info["row_count"]
            or (
                contract["expected_rows"] is not None
                and count != contract["expected_rows"]
            )
        ):
            raise ValueError(
                f"{name}: published row count differs."
            )
        tables[name], counts[name] = table, count

    return manifest, tables, counts


source_silver_loaded_manifest = None
source_silver_loaded_tables = None
source_silver_loaded_counts = None
(
    source_silver_loaded_manifest,
    source_silver_loaded_tables,
    source_silver_loaded_counts,
) = load_source_silver(
    source_silver_publication["manifest_path"],
    silver_context,
    expected_manifest_sha256=(
        source_silver_publication["manifest_sha256"]
    ),
    profile=alabama_profile,
    rule_versions=source_silver_rule_versions,
)

spark.createDataFrame(
    sorted(source_silver_loaded_counts.items()),
    "dataset STRING, records LONG",
).show(truncate=False)

source_silver_completion = {
    "status": "success",
    "stage": "source_silver",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_id": silver_context.run_id,
    "environment": silver_context.environment,
    "manifest_path": (
        Path(source_silver_publication["manifest_path"])
        .relative_to(silver_context.environment_root)
        .as_posix()
    ),
    "manifest_sha256": source_silver_publication["manifest_sha256"],
    "publication_outcome": source_silver_publication["outcome"],
    "datasets": source_silver_loaded_counts,
    "fixture_status": source_silver_checks["status"],
    "fixture_results": source_silver_checks["results"],
}
source_silver_completion_path = (
    silver_context.reports_dir / "source_silver_completion.json"
)
write_json(
    source_silver_completion_path, source_silver_completion
)

print(
    "Source Silver loaded and verified:",
    len(source_silver_loaded_tables),
    "datasets.",
)
print("Source Silver status:", source_silver_completion["status"])
print("Fixture status:", source_silver_checks["status"])
print(
    "Profile status:",
    source_silver_loaded_manifest["profile"]["status"],
)
print("Completion report:", source_silver_completion_path)

+-------------------+-------+
|dataset            |records|
+-------------------+-------+
|epa_monitors       |1      |
|epa_sites          |1      |
|noaa_stations      |1      |
|ozone_audit        |5166   |
|ozone_observations |5071   |
|reference_audit    |3      |
|temperature_audit  |11851  |
|temperature_reports|11686  |
+-------------------+-------+

Source Silver loaded and verified: 8 datasets.
Source Silver status: success
Fixture status: passed
Profile status: provisional
Completion report: /home/ubuntu/dat535-final-project-data/dev/runs/26a8250b3c0a422dab4875f5e49d86d8/reports/source_silver_completion.json


## 9. Hourly Silver integration and coverage

Combine eligible ozone observations with nearby temperature reports using the
selected yearly site–station mapping. Preserve missing temperature and report
coverage across the complete calendar year.

### 9.1 Published inputs and time convention

Use the source Silver tables loaded from their publication manifest. Work in
UTC with intervals that include the starting hour and exclude the next hour.

EPA ozone timestamps identify the start of sampling. The temperature policy
selects the latest eligible report within the same clock hour and preserves
its actual report timestamp. A report at 15:53 belongs to [15:00, 16:00).

In [40]:
hourly_manifest = source_silver_loaded_manifest
hourly_tables = source_silver_loaded_tables

if (
    hourly_manifest is None
    or hourly_tables is None
    or hourly_manifest["kind"] != "source_silver"
    or hourly_manifest["status"] != "published"
    or hourly_manifest["environment"] != silver_context.environment
):
    raise RuntimeError(
        "Load the published source Silver bundle in 8.7 first."
    )

if spark.conf.get("spark.sql.session.timeZone") != "UTC":
    raise RuntimeError(
        "Hourly integration requires the Spark session timezone UTC."
    )

hourly_profile = json.loads(json.dumps(hourly_manifest["profile"]))
hourly_year = hourly_profile["year"]
hourly_monitor = hourly_profile["ozone_monitor"]

hourly_monitor_id = "-".join(
    hourly_monitor[key]
    for key in (
        "state_code", "county_code", "site_number",
        "parameter_code", "poc",
    )
)
hourly_site_id = "-".join(
    hourly_monitor[key]
    for key in ("state_code", "county_code", "site_number")
)
hourly_station_id = (
    hourly_profile["weather_mapping"]["noaa_station_id"]
)

if hourly_profile["weather_mapping"]["epa_site_id"] != hourly_site_id:
    raise ValueError(
        "The profile's weather mapping belongs to a different EPA site."
    )

temperature_policy = hourly_profile["temperature_policy"]
if (
    temperature_policy["version"] != "temperature-hourly-v1"
    or temperature_policy["hourly_selection"]
    != "latest_within_utc_hour"
):
    raise ValueError(
        "The profile requests an unsupported hourly temperature policy."
    )

hourly_rules = {
    "version": "hourly-integration-v1",
    "timezone": "UTC",
    "hour_interval": "[hour_start_utc, hour_start_utc + 1 hour)",
    "ozone_timestamp_meaning": "sample_start",
    "temperature_policy_version": temperature_policy["version"],
    "temperature_selection": temperature_policy["hourly_selection"],
    "missing_temperature": "retain_ozone_with_null_temperature",
}

print("Source Silver publication:", hourly_manifest["publication_id"])
print("Region and year:", hourly_profile["region"], hourly_year)
print("Ozone monitor:", hourly_monitor_id)
print("Weather station:", hourly_station_id)
print(
    "Calendar hours:",
    (366 if calendar.isleap(hourly_year) else 365) * 24,
)
print(
    "Temperature selection:",
    hourly_rules["temperature_selection"],
)
print("Profile status:", hourly_profile["status"])

Source Silver publication: c9ca912a73aa65495afb629806252b65f5a718c43c514feb3c4c50eec3ff7cd2
Region and year: Alabama 2025
Ozone monitor: 01-073-1005-44201-1
Weather station: USW00053864
Calendar hours: 8760
Temperature selection: latest_within_utc_hour
Profile status: provisional


### 9.2 Inspect hourly alignment

Check that published observation keys are unique and belong to the selected
monitor or station and UTC year. Ozone observations must fall on hour boundaries;
temperature reports may occur between them.

Count the eligible temperature reports in each UTC hour. These counts describe
the candidates for hourly selection, with exact source copies already resolved
during source processing.

In [41]:
def inspect_hourly_inputs(tables, profile):
    """Check integration keys and summarize their UTC clock-hour distribution."""
    year = profile["year"]
    monitor = profile["ozone_monitor"]
    monitor_id = "-".join(
        monitor[key]
        for key in (
            "state_code", "county_code", "site_number",
            "parameter_code", "poc",
        )
    )
    station_id = profile["weather_mapping"]["noaa_station_id"]

    start = F.to_timestamp(F.lit(f"{year}-01-01 00:00:00"))
    end = F.to_timestamp(F.lit(f"{year + 1}-01-01 00:00:00"))
    rows = []

    for name, id_column, expected_id in (
        ("ozone_observations", "monitor_id", monitor_id),
        ("temperature_reports", "station_id", station_id),
    ):
        table = tables[name]
        timestamp = F.col("timestamp_utc")
        hour = F.date_trunc("hour", timestamp)

        invalid_scope = (
            ~F.col(id_column).eqNullSafe(F.lit(expected_id))
            | ~F.coalesce(
                (timestamp >= start) & (timestamp < end),
                F.lit(False),
            )
        )

        result = table.agg(
            F.count("*").alias("records"),
            F.countDistinct(
                id_column, "timestamp_utc"
            ).alias("distinct_keys"),
            F.countDistinct(
                F.col(id_column), hour
            ).alias("clock_hours"),
            F.count(
                F.when(timestamp != hour, 1)
            ).alias("off_hour_records"),
            F.count(
                F.when(invalid_scope, 1)
            ).alias("invalid_scope_records"),
            F.date_format(
                F.min(timestamp), "yyyy-MM-dd HH:mm:ss"
            ).alias("earliest_utc"),
            F.date_format(
                F.max(timestamp), "yyyy-MM-dd HH:mm:ss"
            ).alias("latest_utc"),
        ).first().asDict()

        if (
            result["invalid_scope_records"]
            or result["records"] != result["distinct_keys"]
            or (
                name == "ozone_observations"
                and result["off_hour_records"]
            )
        ):
            raise ValueError(
                f"{name}: hourly input checks failed: {result}"
            )

        rows.append((
            name,
            result["records"],
            result["distinct_keys"],
            result["clock_hours"],
            result["off_hour_records"],
            result["invalid_scope_records"],
            result["earliest_utc"],
            result["latest_utc"],
        ))

    summary = spark.createDataFrame(
        rows,
        "dataset STRING, records LONG, distinct_keys LONG, "
        "clock_hours LONG, off_hour_records LONG, "
        "invalid_scope_records LONG, "
        "earliest_utc STRING, latest_utc STRING",
    )

    temperature_hours = (
        tables["temperature_reports"]
        .withColumn(
            "hour_start_utc",
            F.date_trunc("hour", "timestamp_utc"),
        )
        .groupBy("station_id", "hour_start_utc")
        .agg(
            F.count("*").alias("qualifying_report_count")
        )
    )

    return summary, temperature_hours


hourly_input_summary, temperature_hour_counts = inspect_hourly_inputs(
    hourly_tables, hourly_profile,
)

hourly_input_summary.show(truncate=False)

print("Eligible temperature reports per UTC hour:")
temperature_hour_counts.groupBy(
    "qualifying_report_count"
).agg(
    F.count("*").alias("hours"),
).orderBy(
    "qualifying_report_count"
).show(50, truncate=False)

print("Hourly input checks passed.")

+-------------------+-------+-------------+-----------+----------------+---------------------+-------------------+-------------------+
|dataset            |records|distinct_keys|clock_hours|off_hour_records|invalid_scope_records|earliest_utc       |latest_utc         |
+-------------------+-------+-------------+-----------+----------------+---------------------+-------------------+-------------------+
|ozone_observations |5071   |5071         |5071       |0               |0                    |2025-04-01 06:00:00|2025-11-03 13:00:00|
|temperature_reports|11686  |11686        |8514       |11618           |0                    |2025-01-01 00:53:00|2025-12-31 23:53:00|
+-------------------+-------+-------------+-----------+----------------+---------------------+-------------------+-------------------+

Eligible temperature reports per UTC hour:
+-----------------------+-----+
|qualifying_report_count|hours|
+-----------------------+-----+
|1                      |6941 |
|2                

### 9.3 Validate the yearly site–station mapping

Require one matching EPA site, EPA monitor and NOAA station. Check identifier
consistency and usable coordinates, and calculate approximate distance from
the published coordinates.

Represent the configured pairing with a start-inclusive, end-exclusive yearly
interval. Preserve reference provenance and flag metadata concerns for review.
The mapping interval does not imply continuous operation or verified historical
station conditions.

In [42]:
def build_yearly_mapping(tables, profile, source_publication_id):
    """Check the selected entities and describe the fixed yearly mapping."""
    if spark.conf.get("spark.sql.session.timeZone") != "UTC":
        raise RuntimeError("Mapping intervals require UTC.")

    year = profile["year"]
    monitor = profile["ozone_monitor"]
    site_keys = ("state_code", "county_code", "site_number")
    monitor_keys = (*site_keys, "parameter_code", "poc")

    site_id = "-".join(monitor[key] for key in site_keys)
    monitor_id = "-".join(monitor[key] for key in monitor_keys)
    station_id = profile["weather_mapping"]["noaa_station_id"]

    if profile["weather_mapping"]["epa_site_id"] != site_id:
        raise ValueError(
            "The mapping and ozone monitor identify different EPA sites."
        )

    references = {}
    for name, id_column, expected_id in (
        ("epa_sites", "site_id", site_id),
        ("epa_monitors", "monitor_id", monitor_id),
        ("noaa_stations", "station_id", station_id),
    ):
        rows = tables[name].limit(2).collect()
        if len(rows) != 1 or rows[0][id_column] != expected_id:
            raise ValueError(
                f"{name}: expected exactly one matching reference record."
            )
        references[name] = rows[0].asDict(recursive=True)

    site, monitor_record, station = (
        references[name]
        for name in ("epa_sites", "epa_monitors", "noaa_stations")
    )

    if (
        any(site[key] != monitor[key] for key in site_keys)
        or any(
            monitor_record[key] != monitor[key]
            for key in monitor_keys
        )
    ):
        raise ValueError(
            "Reference identifier components disagree with the profile."
        )

    for name, record in references.items():
        lat, lon = record["latitude_deg"], record["longitude_deg"]
        if (
            lat is None or lon is None
            or not math.isfinite(lat) or not math.isfinite(lon)
            or not -90 <= lat <= 90
            or not -180 <= lon <= 180
        ):
            raise ValueError(f"{name}: coordinates are unusable.")

    review_flags = [
        f"{name}:{flag}"
        for name, record in references.items()
        for flag in record["quality_flags"]
    ]

    if (
        site["latitude_deg"],
        site["longitude_deg"],
        site["datum"],
    ) != (
        monitor_record["latitude_deg"],
        monitor_record["longitude_deg"],
        monitor_record["datum"],
    ):
        review_flags.append(
            "epa_site_monitor_location_metadata_differ"
        )

    # Approximate distance between the published coordinate pairs.
    distance_km = None
    if (site["datum"] or "").strip().upper() == "WGS84":
        lat1, lat2 = map(
            math.radians,
            (site["latitude_deg"], station["latitude_deg"]),
        )
        dlat = lat2 - lat1
        dlon = math.radians(
            station["longitude_deg"] - site["longitude_deg"]
        )
        a = math.sin(dlat / 2) ** 2 + (
            math.cos(lat1)
            * math.cos(lat2)
            * math.sin(dlon / 2) ** 2
        )
        distance_km = (
            2 * 6371.0
            * math.asin(math.sqrt(min(1.0, max(0.0, a))))
        )
    else:
        review_flags.append("distance_requires_epa_datum_review")

    first_day = datetime(year, 1, 1).date()
    last_day = datetime(year, 12, 31).date()

    if (
        site["site_established_date"]
        and site["site_established_date"] > first_day
    ):
        review_flags.append("site_established_after_year_start")

    if (
        site["site_closed_date"]
        and site["site_closed_date"] <= last_day
    ):
        review_flags.append("site_closure_recorded_by_year_end")

    if (
        monitor_record["first_year_of_data"] is not None
        and monitor_record["first_year_of_data"] > year
    ):
        review_flags.append("monitor_first_year_after_analysis_year")

    if (
        monitor_record["last_sample_date"]
        and monitor_record["last_sample_date"] < last_day
    ):
        review_flags.append("monitor_last_sample_before_year_end")

    for name in ("epa_sites", "epa_monitors"):
        extraction = references[name]["extraction_date"]
        if extraction and extraction < last_day:
            review_flags.append(f"{name}:snapshot_before_year_end")

    review_flags = sorted(set(review_flags))

    identity = {
        "mapping_version": "fixed-site-station-year-v1",
        "region": profile["region"],
        "year": year,
        "monitor_id": monitor_id,
        "site_id": site_id,
        "station_id": station_id,
        "valid_from_utc": f"{year}-01-01 00:00:00",
        "valid_until_utc": f"{year + 1}-01-01 00:00:00",
        "profile_sha256": silver_json_sha256(profile),
        "source_silver_publication_id": source_publication_id,
    }
    mapping_id = silver_json_sha256(identity)

    mapping = spark.createDataFrame(
        [(
            mapping_id,
            identity["mapping_version"],
            profile["region"],
            year,
            monitor_id,
            site_id,
            station_id,
            identity["valid_from_utc"],
            identity["valid_until_utc"],
            distance_km,
            review_flags,
            source_publication_id,
        )],
        "mapping_id STRING, mapping_version STRING, "
        "region STRING, year INT, monitor_id STRING, "
        "site_id STRING, station_id STRING, "
        "valid_from_utc STRING, valid_until_utc STRING, "
        "distance_km DOUBLE, mapping_review_flags ARRAY<STRING>, "
        "source_silver_publication_id STRING",
    ).withColumn(
        "valid_from_utc", F.to_timestamp("valid_from_utc"),
    ).withColumn(
        "valid_until_utc", F.to_timestamp("valid_until_utc"),
    )

    evidence = {
        **identity,
        "mapping_id": mapping_id,
        "structural_status": "passed",
        "approval_status": "pending",
        "distance_km": distance_km,
        "distance_method": "haversine; spherical Earth radius 6371 km",
        "review_flags": review_flags,
        "reference_records": json.loads(
            json.dumps(references, default=str, allow_nan=False)
        ),
        "limitations": [
            "The yearly interval is the configured pairing, not an operating schedule.",
            "Reference snapshots do not establish complete station or site history.",
            "Distance describes proximity, not equivalent local weather conditions.",
            "Distance uses published coordinates without a datum transformation or terrain model.",
        ],
    }
    return mapping, evidence


yearly_mapping = None
yearly_mapping_evidence = None
yearly_mapping, yearly_mapping_evidence = build_yearly_mapping(
    hourly_tables,
    hourly_profile,
    hourly_manifest["publication_id"],
)

yearly_mapping.select(
    "monitor_id",
    "site_id",
    "station_id",
    "valid_from_utc",
    "valid_until_utc",
    F.round("distance_km", 2).alias("distance_km"),
    "mapping_review_flags",
).show(truncate=False, vertical=True)

for name, record in yearly_mapping_evidence["reference_records"].items():
    print(name, json.dumps({
        key: record[key]
        for key in (
            "site_name", "land_use", "location_setting", "station_name",
            "site_established_date", "site_closed_date",
            "first_year_of_data", "last_sample_date", "extraction_date",
        )
        if key in record
    }))

print(
    "Mapping structure:",
    yearly_mapping_evidence["structural_status"],
)
print(
    "Mapping approval:",
    yearly_mapping_evidence["approval_status"],
)
print("Historical operating calendar: unknown")

-RECORD 0-----------------------------------
 monitor_id           | 01-073-1005-44201-1 
 site_id              | 01-073-1005         
 station_id           | USW00053864         
 valid_from_utc       | 2025-01-01 00:00:00 
 valid_until_utc      | 2026-01-01 00:00:00 
 distance_km          | 26.73               
 mapping_review_flags | []                  

epa_sites {"site_name": "McAdory", "land_use": "RESIDENTIAL", "location_setting": "SUBURBAN", "site_established_date": "1987-06-17", "site_closed_date": null, "extraction_date": "2026-06-26"}
epa_monitors {"first_year_of_data": 1987, "last_sample_date": "2026-03-31", "extraction_date": "2026-06-26"}
noaa_stations {"station_name": "ALABASTER SHELBY CO AP ASOS"}
Mapping structure: passed
Mapping approval: pending
Historical operating calendar: unknown


### 9.4 Select hourly temperature

Select the latest eligible report within each station's UTC clock hour.
Preserve its actual timestamp, temperature, source metadata, quality flags and
lineage. Record how many eligible reports were available in that hour.

Repeated station–timestamp keys stop processing before selection. Hours without
eligible reports have no hourly temperature row; calendar coverage identifies
those hours later. All eligible reports remain available in source Silver.

In [43]:
def select_hourly_temperature(reports, *, policy_version):
    """Select the latest eligible report per station and UTC clock hour."""
    if spark.conf.get("spark.sql.session.timeZone") != "UTC":
        raise RuntimeError("Hourly temperature selection requires UTC.")

    if policy_version != "temperature-hourly-v1":
        raise ValueError("Unsupported hourly temperature policy.")

    reserved = {
        "hour_start_utc",
        "report_timestamp_utc",
        "qualifying_report_count",
        "temperature_selection_rule",
        "__latest_report_timestamp",
    }
    if reserved & set(reports.columns):
        raise ValueError(
            "Input columns conflict with hourly output columns."
        )

    if reports.filter(
        F.col("station_id").isNull()
        | (F.trim("station_id") == "")
        | F.col("timestamp_utc").isNull()
    ).limit(1).count():
        raise ValueError(
            "Hourly selection requires usable station and timestamp keys."
        )

    if reports.groupBy(
        "station_id", "timestamp_utc"
    ).count().filter(
        F.col("count") > 1
    ).limit(1).count():
        raise ValueError(
            "Repeated report timestamps could create ambiguous hourly selection."
        )

    hour_window = Window.partitionBy(
        "station_id", "hour_start_utc"
    )

    candidates = (
        reports
        .withColumnRenamed(
            "timestamp_utc", "report_timestamp_utc"
        )
        .withColumn(
            "hour_start_utc",
            F.date_trunc("hour", "report_timestamp_utc"),
        )
        .withColumn(
            "qualifying_report_count",
            F.count("*").over(hour_window),
        )
        .withColumn(
            "__latest_report_timestamp",
            F.max("report_timestamp_utc").over(hour_window),
        )
    )

    retained_columns = [
        name for name in reports.columns
        if name not in {
            "station_id", "timestamp_utc", "temperature_c"
        }
    ]

    return (
        candidates
        .filter(
            F.col("report_timestamp_utc")
            == F.col("__latest_report_timestamp")
        )
        .withColumn(
            "temperature_selection_rule",
            F.lit(policy_version),
        )
        .select(
            "station_id",
            "hour_start_utc",
            "report_timestamp_utc",
            "temperature_c",
            "qualifying_report_count",
            "temperature_selection_rule",
            *retained_columns,
        )
    )


temperature_hourly = None
temperature_hourly = select_hourly_temperature(
    hourly_tables["temperature_reports"],
    policy_version=hourly_rules["temperature_policy_version"],
)

temperature_hourly_checks = temperature_hourly.agg(
    F.count("*").alias("hourly_records"),
    F.countDistinct(
        "station_id", "hour_start_utc"
    ).alias("distinct_hour_keys"),
    F.coalesce(
        F.sum("qualifying_report_count"), F.lit(0)
    ).alias("eligible_reports"),
    F.count(
        F.when(F.col("qualifying_report_count") > 1, 1)
    ).alias("hours_with_multiple_reports"),
    F.count(
        F.when(
            (
                F.col("report_timestamp_utc")
                < F.col("hour_start_utc")
            )
            | (
                F.col("report_timestamp_utc")
                >= F.expr("hour_start_utc + INTERVAL 1 HOUR")
            ),
            1,
        )
    ).alias("reports_outside_selected_hour"),
).first().asDict()

expected_temperature = hourly_input_summary.filter(
    F.col("dataset") == "temperature_reports"
).first().asDict()

if (
    temperature_hourly_checks["hourly_records"]
    != expected_temperature["clock_hours"]
    or temperature_hourly_checks["distinct_hour_keys"]
    != temperature_hourly_checks["hourly_records"]
    or temperature_hourly_checks["eligible_reports"]
    != expected_temperature["records"]
    or temperature_hourly_checks["reports_outside_selected_hour"]
):
    raise ValueError(
        f"Hourly temperature checks failed: {temperature_hourly_checks}"
    )

for metric, value in temperature_hourly_checks.items():
    print(f"{metric}: {value}")

print(
    "Other eligible reports retained in source Silver:",
    expected_temperature["records"]
    - temperature_hourly_checks["hourly_records"],
)
print(
    "Calendar hours without eligible temperature:",
    (366 if calendar.isleap(hourly_year) else 365) * 24
    - temperature_hourly_checks["hourly_records"],
)

temperature_hourly.filter(
    F.col("qualifying_report_count") > 1
).select(
    "station_id",
    "hour_start_utc",
    "report_timestamp_utc",
    "temperature_c",
    "qualifying_report_count",
    "report_type",
).orderBy(
    "station_id", "hour_start_utc"
).show(5, truncate=False)

print("Hourly temperature selection checks passed.")

hourly_records: 8514
distinct_hour_keys: 8514
eligible_reports: 11686
hours_with_multiple_reports: 1573
reports_outside_selected_hour: 0
Other eligible reports retained in source Silver: 3172
Calendar hours without eligible temperature: 246
+-----------+-------------------+--------------------+-------------+-----------------------+-----------+
|station_id |hour_start_utc     |report_timestamp_utc|temperature_c|qualifying_report_count|report_type|
+-----------+-------------------+--------------------+-------------+-----------------------+-----------+
|USW00053864|2025-01-05 15:00:00|2025-01-05 15:53:00 |8.9          |2                      |FM15       |
|USW00053864|2025-01-05 16:00:00|2025-01-05 16:53:00 |7.8          |2                      |FM15       |
|USW00053864|2025-01-05 17:00:00|2025-01-05 17:53:00 |7.8          |2                      |FM15       |
|USW00053864|2025-01-05 18:00:00|2025-01-05 18:53:00 |10.0         |3                      |FM15       |
|USW00053864|2025-01-05 

### 9.5 Enrich ozone with hourly temperature

Attach the selected temperature report using the yearly monitor–station mapping and UTC hour. Preserve every eligible ozone observation, retaining null temperature values and an integration flag when no eligible report is available. Keep ozone and temperature source lineage separate.

In [44]:
def check_hourly_keys(table, keys, label):
    """Require non-empty, unique keys and return the row count."""
    usable = F.lit(True)
    for name in keys:
        usable = usable & F.col(name).isNotNull() & (
            F.trim(F.col(name).cast("string")) != ""
        )

    counts = table.agg(
        F.count("*").alias("records"),
        F.countDistinct(*keys).alias("distinct_keys"),
        F.count(F.when(~usable, 1)).alias("unusable_keys"),
    ).first()

    if counts["unusable_keys"] or counts["records"] != counts["distinct_keys"]:
        raise ValueError(f"{label}: missing or repeated keys: {counts.asDict()}")
    return counts["records"]


def enrich_hourly_ozone(ozone, weather, mapping):
    """Attach the mapped hourly temperature without removing ozone rows."""
    if ozone.sparkSession.conf.get("spark.sql.session.timeZone") != "UTC":
        raise ValueError("Hourly enrichment requires UTC.")

    input_records = check_hourly_keys(
        ozone, ["monitor_id", "timestamp_utc"], "Ozone"
    )
    check_hourly_keys(
        weather, ["station_id", "hour_start_utc"], "Hourly temperature"
    )
    if mapping.count() != 1:
        raise ValueError("A region-year requires exactly one fixed mapping.")

    mapped = (
        ozone.withColumnRenamed("timestamp_utc", "hour_start_utc")
        .join(
            F.broadcast(mapping.withColumnRenamed(
                "station_id", "weather_station_id"
            )),
            on="monitor_id",
            how="left",
        )
    )
    valid_mapping = (
        F.col("mapping_id").isNotNull()
        & F.col("weather_station_id").isNotNull()
        & (F.trim("weather_station_id") != "")
        & (F.col("hour_start_utc") >= F.col("valid_from_utc"))
        & (F.col("hour_start_utc") < F.col("valid_until_utc"))
        & (F.col("hour_start_utc") == F.date_trunc("hour", "hour_start_utc"))
    )
    if mapped.filter(~F.coalesce(valid_mapping, F.lit(False))).limit(1).count():
        raise ValueError("An ozone hour has no usable mapping or is off-hour.")

    # Keep weather lineage separate from ozone lineage.
    weather_names = {
        name: (
            "weather_station_id" if name == "station_id"
            else name if name == "hour_start_utc" or name.startswith("temperature_")
            else f"temperature_{name}"
        )
        for name in weather.columns
    }
    weather_columns = weather.select(
        *[F.col(name).alias(target) for name, target in weather_names.items()]
    )
    enriched = (
        mapped.join(
            weather_columns,
            on=["weather_station_id", "hour_start_utc"],
            how="left",
        )
        .withColumn(
            "temperature_available",
            F.col("temperature_report_timestamp_utc").isNotNull(),
        )
        .withColumn(
            "integration_flags",
            F.when(
                ~F.col("temperature_available"),
                F.array(F.lit("no_eligible_temperature")),
            ).otherwise(F.array().cast("array<string>")),
        )
    )

    checks = enriched.agg(
        F.count("*").alias("enriched_records"),
        F.countDistinct("monitor_id", "hour_start_utc").alias("distinct_hour_keys"),
        F.count(F.when(F.col("temperature_available"), 1)).alias("matched_temperature"),
        F.count(F.when(~F.col("temperature_available"), 1)).alias("missing_temperature"),
    ).first().asDict()
    checks["input_ozone_records"] = input_records

    if (
        checks["enriched_records"] != input_records
        or checks["distinct_hour_keys"] != input_records
    ):
        raise ValueError(f"Ozone enrichment changed the row grain: {checks}")

    return enriched, checks


ozone_enriched, ozone_enrichment_checks = enrich_hourly_ozone(
    hourly_tables["ozone_observations"],
    temperature_hourly,
    yearly_mapping,
)

for metric, value in ozone_enrichment_checks.items():
    print(f"{metric}: {value}")

ozone_enriched.filter(~F.col("temperature_available")).select(
    "monitor_id", "hour_start_utc", "ozone_ppm",
    "temperature_c", "integration_flags",
).orderBy("hour_start_utc").show(5, truncate=False)

print("Ozone left-enrichment checks passed.")

enriched_records: 5071
distinct_hour_keys: 5071
matched_temperature: 4836
missing_temperature: 235
input_ozone_records: 5071
+-------------------+-------------------+---------+-------------+-------------------------+
|monitor_id         |hour_start_utc     |ozone_ppm|temperature_c|integration_flags        |
+-------------------+-------------------+---------+-------------+-------------------------+
|01-073-1005-44201-1|2025-06-01 14:00:00|0.04     |NULL         |[no_eligible_temperature]|
|01-073-1005-44201-1|2025-06-01 15:00:00|0.049    |NULL         |[no_eligible_temperature]|
|01-073-1005-44201-1|2025-06-01 16:00:00|0.047    |NULL         |[no_eligible_temperature]|
|01-073-1005-44201-1|2025-06-01 17:00:00|0.047    |NULL         |[no_eligible_temperature]|
|01-073-1005-44201-1|2025-06-01 18:00:00|0.047    |NULL         |[no_eligible_temperature]|
+-------------------+-------------------+---------+-------------+-------------------------+
only showing top 5 rows

Ozone left-enrichment 

### 9.6 Build calendar-hour coverage

Create one row for every UTC hour in the analysis year. Combine source audits with eligible observations to distinguish eligible, excluded and unobserved hours. Count source records separately from hours, report records that cannot be assigned to the calendar, and leave historical operating schedules unknown.

In [45]:
def build_calendar_coverage(tables, enriched, weather, profile):
    """Build one coverage row per monitor and UTC calendar hour."""
    session = enriched.sparkSession
    if session.conf.get("spark.sql.session.timeZone") != "UTC":
        raise ValueError("Calendar coverage requires UTC.")

    year = int(profile["year"])
    monitor = profile["ozone_monitor"]
    monitor_id = "-".join(monitor[key] for key in (
        "state_code", "county_code", "site_number", "parameter_code", "poc"
    ))
    station_id = profile["weather_mapping"]["noaa_station_id"]
    start = F.to_timestamp(F.lit(f"{year}-01-01 00:00:00"))
    end = F.to_timestamp(F.lit(f"{year + 1}-01-01 00:00:00"))

    calendar_hours = session.range(1).select(
        F.explode(F.sequence(
            start, end - F.expr("INTERVAL 1 HOUR"), F.expr("INTERVAL 1 HOUR")
        )).alias("hour_start_utc")
    ).select(
        F.lit(profile["region"]).alias("region"),
        F.lit(year).alias("year"),
        F.lit(monitor_id).alias("monitor_id"),
        F.lit(station_id).alias("station_id"),
        "hour_start_utc",
    )

    def audit_by_hour(audit, id_column, expected_id, prefix):
        assignable = (
            (F.col(id_column) == expected_id)
            & (F.col("timestamp_utc") >= start)
            & (F.col("timestamp_utc") < end)
        )
        if prefix == "ozone":
            assignable = assignable & (
                F.col("timestamp_utc") == F.date_trunc("hour", "timestamp_utc")
            )
        assignable = F.coalesce(assignable, F.lit(False))

        assignment = audit.agg(
            F.coalesce(F.sum("record_count"), F.lit(0)).alias("source_records"),
            F.coalesce(F.sum(F.when(
                ~assignable, F.col("record_count")
            ).otherwise(0)), F.lit(0)).alias("unassigned_source_records"),
        ).first().asDict()

        aggregates = [
            F.sum("record_count").alias(f"{prefix}_source_records")
        ]
        if prefix == "temperature":
            numeric = F.lit(True)
            for flag in (
                "missing_temperature", "invalid_temperature",
                "nonfinite_temperature",
            ):
                numeric = numeric & ~F.col("field_flags")[flag]
            aggregates.append(
                F.sum(F.when(
                    F.coalesce(numeric, F.lit(False)), F.col("record_count")
                ).otherwise(0)).alias("numeric_temperature_source_records")
            )

        grouped = (
            audit.filter(assignable)
            .withColumn("hour_start_utc", F.date_trunc("hour", "timestamp_utc"))
            .groupBy(id_column, "hour_start_utc")
            .agg(*aggregates)
        )
        return grouped, assignment

    ozone_hours, ozone_assignment = audit_by_hour(
        tables["ozone_audit"], "monitor_id", monitor_id, "ozone"
    )
    temperature_hours, temperature_assignment = audit_by_hour(
        tables["temperature_audit"], "station_id", station_id, "temperature"
    )

    coverage = (
        calendar_hours
        .join(ozone_hours, ["monitor_id", "hour_start_utc"], "left")
        .join(temperature_hours, ["station_id", "hour_start_utc"], "left")
        .join(
            enriched.select(
                "monitor_id", "hour_start_utc",
                F.lit(True).alias("ozone_available"),
            ),
            ["monitor_id", "hour_start_utc"], "left",
        )
        .join(
            weather.select(
                "station_id", "hour_start_utc", "qualifying_report_count"
            ),
            ["station_id", "hour_start_utc"], "left",
        )
        .fillna({
            "ozone_source_records": 0,
            "temperature_source_records": 0,
            "numeric_temperature_source_records": 0,
            "qualifying_report_count": 0,
            "ozone_available": False,
        })
        .withColumn("temperature_available", F.col("qualifying_report_count") > 0)
        .withColumn(
            "numeric_temperature_available",
            F.col("numeric_temperature_source_records") > 0,
        )
        .withColumn(
            "shared_available",
            F.col("ozone_available") & F.col("temperature_available"),
        )
        .withColumn("expected_ozone_operating", F.lit(None).cast("boolean"))
        .withColumn("expected_temperature_operating", F.lit(None).cast("boolean"))
    )
    for source in ("ozone", "temperature"):
        coverage = coverage.withColumn(
            f"{source}_status",
            F.when(F.col(f"{source}_available"), "eligible")
            .when(F.col(f"{source}_source_records") > 0, "excluded")
            .otherwise("not_observed"),
        )

    return coverage, {
        "ozone": ozone_assignment,
        "temperature": temperature_assignment,
    }


hourly_coverage, coverage_audit_assignment = build_calendar_coverage(
    hourly_tables, ozone_enriched, temperature_hourly, hourly_profile
)

coverage_metrics = {
    "observed_ozone_hours": F.col("ozone_source_records") > 0,
    "eligible_ozone_hours": F.col("ozone_available"),
    "excluded_ozone_hours": F.col("ozone_status") == "excluded",
    "observed_temperature_hours": F.col("temperature_source_records") > 0,
    "numeric_temperature_hours": F.col("numeric_temperature_available"),
    "eligible_temperature_hours": F.col("temperature_available"),
    "shared_hours": F.col("shared_available"),
}
hourly_coverage_checks = hourly_coverage.agg(
    F.count("*").alias("calendar_hours"),
    F.countDistinct("monitor_id", "hour_start_utc").alias("distinct_hour_keys"),
    F.sum("qualifying_report_count").alias("eligible_temperature_reports"),
    F.sum("ozone_source_records").alias("assigned_ozone_source_records"),
    F.sum("temperature_source_records").alias("assigned_temperature_source_records"),
    F.count(F.when(
        (F.col("ozone_available") & (F.col("ozone_source_records") == 0))
        | (F.col("temperature_available") & ~F.col("numeric_temperature_available")),
        1,
    )).alias("eligible_hours_without_source_evidence"),
    *[F.count(F.when(condition, 1)).alias(name)
      for name, condition in coverage_metrics.items()],
).first().asDict()

expected_hours = (366 if calendar.isleap(hourly_year) else 365) * 24
expected_coverage = {
    "calendar_hours": expected_hours,
    "distinct_hour_keys": expected_hours,
    "eligible_ozone_hours": ozone_enrichment_checks["enriched_records"],
    "eligible_temperature_hours": temperature_hourly_checks["hourly_records"],
    "eligible_temperature_reports": temperature_hourly_checks["eligible_reports"],
    "shared_hours": ozone_enrichment_checks["matched_temperature"],
    "eligible_hours_without_source_evidence": 0,
}
for metric, expected in expected_coverage.items():
    if hourly_coverage_checks[metric] != expected:
        raise ValueError(f"Calendar coverage mismatch: {metric}")

for source, dataset in (
    ("ozone", "ozone_observations"),
    ("temperature", "temperature_reports"),
):
    assignment = coverage_audit_assignment[source]
    expected = hourly_manifest["validation"]["audits"][dataset]["input_records"]
    assigned = hourly_coverage_checks[f"assigned_{source}_source_records"]
    if (
        assignment["source_records"] != expected
        or assigned + assignment["unassigned_source_records"] != expected
    ):
        raise ValueError(f"{source}: calendar and audit counts do not reconcile.")

for metric, value in hourly_coverage_checks.items():
    print(f"{metric}: {value}")
print("Audit assignment:", json.dumps(coverage_audit_assignment, sort_keys=True))
print("Historical operating calendars: unknown")

hourly_coverage.groupBy("ozone_status", "temperature_status").count().orderBy(
    "ozone_status", "temperature_status"
).show(truncate=False)

print("Calendar coverage checks passed.")

calendar_hours: 8760
distinct_hour_keys: 8760
eligible_temperature_reports: 11686
assigned_ozone_source_records: 5166
assigned_temperature_source_records: 11851
eligible_hours_without_source_evidence: 0
observed_ozone_hours: 5166
eligible_ozone_hours: 5071
excluded_ozone_hours: 95
observed_temperature_hours: 8662
numeric_temperature_hours: 8634
eligible_temperature_hours: 8514
shared_hours: 4836
Audit assignment: {"ozone": {"source_records": 5166, "unassigned_source_records": 0}, "temperature": {"source_records": 11851, "unassigned_source_records": 0}}
Historical operating calendars: unknown
+------------+------------------+-----+
|ozone_status|temperature_status|count|
+------------+------------------+-----+
|eligible    |eligible          |4836 |
|eligible    |excluded          |140  |
|eligible    |not_observed      |95   |
|excluded    |eligible          |95   |
|not_observed|eligible          |3583 |
|not_observed|excluded          |8    |
|not_observed|not_observed      |3    |
+

### 9.7 Summarise monthly coverage and gaps

Report coverage for all twelve months using explicit calendar-hour and eligible-ozone denominators. Leave percentages undefined when their denominator is zero, and retain unknown operating-calendar denominators. Identify continuous intervals without eligible ozone or temperature, including gaps that cross month boundaries. Gap intervals use an inclusive start and exclusive end.

In [46]:
def summarise_hourly_coverage(coverage):
    """Summarise calendar coverage by month and identify continuous gaps."""
    if coverage.sparkSession.conf.get("spark.sql.session.timeZone") != "UTC":
        raise ValueError("Coverage summaries require UTC.")

    keys = ["region", "year", "monitor_id", "station_id"]
    conditions = {
        "observed_ozone_hours": F.col("ozone_source_records") > 0,
        "eligible_ozone_hours": F.col("ozone_available"),
        "excluded_ozone_hours": F.col("ozone_status") == "excluded",
        "observed_temperature_hours": F.col("temperature_source_records") > 0,
        "numeric_temperature_hours": F.col("numeric_temperature_available"),
        "eligible_temperature_hours": F.col("temperature_available"),
        "excluded_temperature_hours": F.col("temperature_status") == "excluded",
        "shared_hours": F.col("shared_available"),
        "ozone_hours_without_temperature": (
            F.col("ozone_available") & ~F.col("temperature_available")
        ),
    }
    operating_counts = [
        F.when(
            F.count(f"expected_{source}_operating") == F.count("*"),
            F.sum(F.col(f"expected_{source}_operating").cast("long")),
        ).alias(f"documented_{source}_operating_hours")
        for source in ("ozone", "temperature")
    ]
    monthly = (
        coverage.withColumn("month_start_utc", F.date_trunc("month", "hour_start_utc"))
        .groupBy(*keys, "month_start_utc")
        .agg(
            F.count("*").alias("calendar_hours"),
            *[F.count(F.when(condition, 1)).alias(name)
              for name, condition in conditions.items()],
            *operating_counts,
        )
    )
    percentages = {
        "ozone_calendar_pct": ("eligible_ozone_hours", "calendar_hours"),
        "temperature_calendar_pct": ("eligible_temperature_hours", "calendar_hours"),
        "temperature_match_pct": ("shared_hours", "eligible_ozone_hours"),
    }
    for name, (numerator, denominator) in percentages.items():
        monthly = monthly.withColumn(
            name,
            F.when(
                F.col(denominator) > 0,
                F.round(100.0 * F.col(numerator) / F.col(denominator), 2),
            ),
        )
    monthly = (
        monthly.withColumn(
            "temperature_match_reason",
            F.when(F.col("eligible_ozone_hours") == 0, "no_eligible_ozone_hours"),
        )
        .withColumn(
            "operating_calendar_reason",
            F.when(
                F.col("documented_ozone_operating_hours").isNull()
                | F.col("documented_temperature_operating_hours").isNull(),
                "unknown_or_incomplete_operating_calendar",
            ),
        )
    )

    gap_conditions = {
        "no_eligible_ozone": ~F.col("ozone_available"),
        "no_eligible_temperature": ~F.col("temperature_available"),
        "ozone_without_temperature": (
            F.col("ozone_available") & ~F.col("temperature_available")
        ),
    }
    missing_hours = coverage.select(
        *keys, "hour_start_utc",
        F.explode(F.filter(
            F.array(*[
                F.when(condition, F.lit(name))
                for name, condition in gap_conditions.items()
            ]),
            lambda name: name.isNotNull(),
        )).alias("gap_kind"),
    )
    gap_order = Window.partitionBy(*keys, "gap_kind").orderBy("hour_start_utc")

    # Consecutive hours share the same timestamp-minus-position value.
    gaps = (
        missing_hours.withColumn(
            "__gap_group",
            F.unix_timestamp("hour_start_utc")
            - F.row_number().over(gap_order).cast("long") * 3600,
        )
        .groupBy(*keys, "gap_kind", "__gap_group")
        .agg(
            F.min("hour_start_utc").alias("gap_start_utc"),
            (F.max("hour_start_utc") + F.expr("INTERVAL 1 HOUR")).alias("gap_end_utc"),
            F.count("*").alias("gap_hours"),
        )
        .drop("__gap_group")
    )
    return monthly, gaps


monthly_coverage, hourly_gaps = summarise_hourly_coverage(hourly_coverage)

monthly_metrics = ["calendar_hours", *coverage_metrics]
monthly_totals = monthly_coverage.agg(
    F.count("*").alias("months"),
    *[F.sum(name).alias(name) for name in monthly_metrics],
).first().asDict()
if monthly_totals["months"] != 12:
    raise ValueError("Coverage must include all twelve months.")
for name in monthly_metrics:
    if monthly_totals[name] != hourly_coverage_checks[name]:
        raise ValueError(f"Monthly coverage does not reconcile: {name}")

gap_summary = hourly_gaps.groupBy("gap_kind").agg(
    F.count("*").alias("gap_count"),
    F.sum("gap_hours").alias("total_gap_hours"),
    F.max("gap_hours").alias("longest_gap_hours"),
    F.count(F.when(
        F.unix_timestamp("gap_end_utc") - F.unix_timestamp("gap_start_utc")
        != F.col("gap_hours") * 3600,
        1,
    )).alias("invalid_intervals"),
)
gap_totals = {row["gap_kind"]: row.asDict() for row in gap_summary.collect()}
expected_gap_hours = {
    "no_eligible_ozone": (
        hourly_coverage_checks["calendar_hours"]
        - hourly_coverage_checks["eligible_ozone_hours"]
    ),
    "no_eligible_temperature": (
        hourly_coverage_checks["calendar_hours"]
        - hourly_coverage_checks["eligible_temperature_hours"]
    ),
    "ozone_without_temperature": (
        hourly_coverage_checks["eligible_ozone_hours"]
        - hourly_coverage_checks["shared_hours"]
    ),
}
for kind, expected in expected_gap_hours.items():
    result = gap_totals.get(kind, {})
    if (
        result.get("total_gap_hours", 0) != expected
        or result.get("invalid_intervals", 0)
    ):
        raise ValueError(f"Gap intervals do not reconcile: {kind}")
    print(
        f"{kind}: gaps={result.get('gap_count', 0)}, "
        f"total_hours={result.get('total_gap_hours', 0)}, "
        f"longest_hours={result.get('longest_gap_hours', 0)}"
    )

monthly_coverage.orderBy("month_start_utc").select(
    F.date_format("month_start_utc", "yyyy-MM").alias("month"),
    "calendar_hours",
    F.col("eligible_ozone_hours").alias("ozone_hours"),
    F.col("eligible_temperature_hours").alias("temperature_hours"),
    "shared_hours", "ozone_calendar_pct", "temperature_calendar_pct",
    "temperature_match_pct", "temperature_match_reason",
).show(12, truncate=False)

hourly_gaps.orderBy(
    F.desc("gap_hours"), "gap_start_utc", "gap_kind"
).select(
    "gap_kind", "gap_start_utc", "gap_end_utc", "gap_hours"
).show(5, truncate=False)

print("Monthly coverage and gap checks passed.")

no_eligible_ozone: gaps=106, total_hours=3689, longest_hours=2166
no_eligible_temperature: gaps=19, total_hours=246, longest_hours=120
ozone_without_temperature: gaps=18, total_hours=235, longest_hours=80
+-------+--------------+-----------+-----------------+------------+------------------+------------------------+---------------------+------------------------+
|month  |calendar_hours|ozone_hours|temperature_hours|shared_hours|ozone_calendar_pct|temperature_calendar_pct|temperature_match_pct|temperature_match_reason|
+-------+--------------+-----------+-----------------+------------+------------------+------------------------+---------------------+------------------------+
|2025-01|744           |0          |738              |0           |0.0               |99.19                   |NULL                 |no_eligible_ozone_hours |
|2025-02|672           |0          |672              |0           |0.0               |100.0                   |NULL                 |no_eligible_ozone_hours |


### 9.8 Prepare and check hourly Silver

Run the hourly transformations for the selected profile and reconcile observation, audit, calendar, monthly and gap counts. Return the prepared tables, processing rules and validation evidence for publication.

In [47]:
def prepare_hourly_silver(tables, manifest, *, environment):
    """Prepare hourly Silver from a verified source Silver publication."""
    if (
        manifest["kind"] != "source_silver"
        or manifest["status"] != "published"
        or manifest["environment"] != environment
        or manifest["validation"]["status"] != "validated"
    ):
        raise ValueError("A validated source Silver publication is required.")
    if spark.conf.get("spark.sql.session.timeZone") != "UTC":
        raise ValueError("Hourly preparation requires UTC.")

    profile = json.loads(json.dumps(manifest["profile"]))
    policy = profile["temperature_policy"]
    if (
        policy["version"] != "temperature-hourly-v1"
        or policy["hourly_selection"] != "latest_within_utc_hour"
    ):
        raise ValueError("Unsupported hourly temperature policy.")

    def require_counts(actual, expected, label):
        differences = {
            name: {"actual": actual.get(name), "expected": value}
            for name, value in expected.items()
            if actual.get(name) != value
        }
        if differences:
            raise ValueError(f"{label}: {differences}")

    input_summary, _ = inspect_hourly_inputs(tables, profile)
    inputs = {row["dataset"]: row.asDict() for row in input_summary.collect()}
    mapping, mapping_evidence = build_yearly_mapping(
        tables, profile, manifest["publication_id"]
    )
    weather = select_hourly_temperature(
        tables["temperature_reports"], policy_version=policy["version"]
    )
    enriched, enrichment_checks = enrich_hourly_ozone(
        tables["ozone_observations"], weather, mapping
    )
    coverage, audit_assignment = build_calendar_coverage(
        tables, enriched, weather, profile
    )
    monthly, gaps = summarise_hourly_coverage(coverage)

    weather_checks = weather.agg(
        F.count("*").alias("hourly_records"),
        F.countDistinct("station_id", "hour_start_utc").alias("distinct_hour_keys"),
        F.coalesce(F.sum("qualifying_report_count"), F.lit(0)).alias("eligible_reports"),
        F.count(F.when(
            (F.col("report_timestamp_utc") < F.col("hour_start_utc"))
            | (F.col("report_timestamp_utc") >= F.expr("hour_start_utc + INTERVAL 1 HOUR")),
            1,
        )).alias("reports_outside_selected_hour"),
    ).first().asDict()
    require_counts(weather_checks, {
        "hourly_records": inputs["temperature_reports"]["clock_hours"],
        "distinct_hour_keys": inputs["temperature_reports"]["clock_hours"],
        "eligible_reports": inputs["temperature_reports"]["records"],
        "reports_outside_selected_hour": 0,
    }, "Hourly temperature")

    conditions = {
        "observed_ozone_hours": F.col("ozone_source_records") > 0,
        "eligible_ozone_hours": F.col("ozone_available"),
        "excluded_ozone_hours": F.col("ozone_status") == "excluded",
        "observed_temperature_hours": F.col("temperature_source_records") > 0,
        "numeric_temperature_hours": F.col("numeric_temperature_available"),
        "eligible_temperature_hours": F.col("temperature_available"),
        "shared_hours": F.col("shared_available"),
    }
    coverage_checks = coverage.agg(
        F.count("*").alias("calendar_hours"),
        F.countDistinct("monitor_id", "hour_start_utc").alias("distinct_hour_keys"),
        F.sum("qualifying_report_count").alias("eligible_temperature_reports"),
        F.sum("ozone_source_records").alias("assigned_ozone_source_records"),
        F.sum("temperature_source_records").alias("assigned_temperature_source_records"),
        F.count(F.when(
            (F.col("ozone_available") & (F.col("ozone_source_records") == 0))
            | (F.col("temperature_available") & ~F.col("numeric_temperature_available")),
            1,
        )).alias("eligible_hours_without_source_evidence"),
        *[F.count(F.when(condition, 1)).alias(name)
          for name, condition in conditions.items()],
    ).first().asDict()
    calendar_hours = (366 if calendar.isleap(profile["year"]) else 365) * 24
    require_counts(coverage_checks, {
        "calendar_hours": calendar_hours,
        "distinct_hour_keys": calendar_hours,
        "eligible_ozone_hours": inputs["ozone_observations"]["records"],
        "eligible_temperature_hours": weather_checks["hourly_records"],
        "eligible_temperature_reports": weather_checks["eligible_reports"],
        "shared_hours": enrichment_checks["matched_temperature"],
        "eligible_hours_without_source_evidence": 0,
    }, "Calendar coverage")

    for source, dataset in (
        ("ozone", "ozone_observations"),
        ("temperature", "temperature_reports"),
    ):
        expected = manifest["validation"]["audits"][dataset]["input_records"]
        assignment = audit_assignment[source]
        assigned = coverage_checks[f"assigned_{source}_source_records"]
        require_counts({
            "source_records": assignment["source_records"],
            "accounted_records": assigned + assignment["unassigned_source_records"],
        }, {
            "source_records": expected, "accounted_records": expected,
        }, f"{source} audit")

    monthly_metrics = ["calendar_hours", *conditions]
    monthly_checks = monthly.agg(
        F.count("*").alias("months"),
        *[F.sum(name).alias(name) for name in monthly_metrics],
    ).first().asDict()
    require_counts(monthly_checks, {
        "months": 12,
        **{name: coverage_checks[name] for name in monthly_metrics},
    }, "Monthly coverage")

    gap_rows = gaps.groupBy("gap_kind").agg(
        F.count("*").alias("gap_count"),
        F.sum("gap_hours").alias("total_gap_hours"),
        F.max("gap_hours").alias("longest_gap_hours"),
        F.count(F.when(
            F.unix_timestamp("gap_end_utc") - F.unix_timestamp("gap_start_utc")
            != F.col("gap_hours") * 3600,
            1,
        )).alias("invalid_intervals"),
    ).collect()
    gap_checks = {row["gap_kind"]: row.asDict() for row in gap_rows}
    expected_gaps = {
        "no_eligible_ozone": calendar_hours - coverage_checks["eligible_ozone_hours"],
        "no_eligible_temperature": calendar_hours - coverage_checks["eligible_temperature_hours"],
        "ozone_without_temperature": (
            coverage_checks["eligible_ozone_hours"] - coverage_checks["shared_hours"]
        ),
    }
    for kind, expected in expected_gaps.items():
        actual = gap_checks.get(kind, {
            "gap_count": 0, "total_gap_hours": 0,
            "longest_gap_hours": 0, "invalid_intervals": 0,
        })
        require_counts(actual, {
            "total_gap_hours": expected, "invalid_intervals": 0,
        }, f"Gap intervals: {kind}")
        gap_checks[kind] = actual

    return {
        "status": "prepared_unpublished",
        "profile": profile,
        "source_silver_publication_id": manifest["publication_id"],
        "rules": {
            "version": "hourly-integration-v1",
            "timezone": "UTC",
            "hour_interval": "[hour_start_utc, hour_start_utc + 1 hour)",
            "ozone_timestamp_meaning": "sample_start",
            "temperature_policy_version": policy["version"],
            "temperature_selection": policy["hourly_selection"],
            "missing_temperature": "retain_ozone_with_null_temperature",
        },
        "mapping_evidence": mapping_evidence,
        "tables": {
            "yearly_mapping": mapping,
            "temperature_hourly": weather,
            "ozone_enriched": enriched,
            "hourly_coverage": coverage,
            "monthly_coverage": monthly,
            "hourly_gaps": gaps,
        },
        "checks": {
            "scope": "in_memory_reconciliation",
            "status": "passed",
            "inputs": inputs,
            "temperature": weather_checks,
            "enrichment": enrichment_checks,
            "coverage": coverage_checks,
            "audit_assignment": audit_assignment,
            "monthly": monthly_checks,
            "gaps": gap_checks,
        },
    }


prepared_silver = None
prepared_silver = prepare_hourly_silver(
    hourly_tables, hourly_manifest,
    environment=silver_context.environment,
)

print("Preparation status:", prepared_silver["status"])
print("In-memory checks:", prepared_silver["checks"]["status"])
print("Region and year:", prepared_silver["profile"]["region"], prepared_silver["profile"]["year"])
print("Eligible ozone hours:", prepared_silver["checks"]["coverage"]["eligible_ozone_hours"])
print("Eligible temperature hours:", prepared_silver["checks"]["coverage"]["eligible_temperature_hours"])
print("Shared hours:", prepared_silver["checks"]["coverage"]["shared_hours"])
print("Calendar hours:", prepared_silver["checks"]["coverage"]["calendar_hours"])
print("Months:", prepared_silver["checks"]["monthly"]["months"])
print("Mapping approval:", prepared_silver["mapping_evidence"]["approval_status"])

Preparation status: prepared_unpublished
In-memory checks: passed
Region and year: Alabama 2025
Eligible ozone hours: 5071
Eligible temperature hours: 8514
Shared hours: 4836
Calendar hours: 8760
Months: 12
Mapping approval: pending


#### Verify hourly integration

Run the synthetic hourly integration checks and record the companion notebook's code fingerprint.

In [48]:
hourly_fixture_checks = None
hourly_fixture_setting = os.environ.get("PIPELINE_RUN_FIXTURES", "false").lower()
if hourly_fixture_setting not in {"true", "false"}:
    raise ValueError("PIPELINE_RUN_FIXTURES must be true or false.")

if hourly_fixture_setting == "true":
    hourly_fixture_path = project_dir / "notebooks" / "silver_checks.ipynb"
    hourly_fixture_notebook = json.loads(
        hourly_fixture_path.read_text(encoding="utf-8")
    )
    hourly_fixture_namespace = {"__name__": "hourly_fixture_checks"}
    for index, cell in enumerate(hourly_fixture_notebook["cells"]):
        if cell["cell_type"] == "code":
            exec(compile(
                "".join(cell["source"]),
                f"{hourly_fixture_path}:cell-{index}", "exec"
            ), hourly_fixture_namespace)

    hourly_fixture_result = hourly_fixture_namespace["verify_hourly_integration"](
        spark=spark,
        profile=hourly_profile,
        references=hourly_tables,
        build_yearly_mapping=build_yearly_mapping,
        select_hourly_temperature=select_hourly_temperature,
        enrich_hourly_ozone=enrich_hourly_ozone,
        prepare_hourly_silver=prepare_hourly_silver,
    )
    hourly_fixture_checks = {
        "status": "passed",
        "companion_code_sha256": notebook_code_sha256(hourly_fixture_notebook),
        "results": hourly_fixture_result,
    }
    print("Hourly integration fixtures passed:", hourly_fixture_result)
else:
    hourly_fixture_checks = {"status": "skipped", "results": {}}
    print("Hourly fixtures skipped: PIPELINE_RUN_FIXTURES=false.")

Hourly integration fixtures passed: {'checks': 29, 'calendar_hours': 8784, 'months': 12}


### 9.9 Check portability and record performance

#### Georgia portability

Apply the shared source-cleaning, reference and hourly-enrichment functions to the selected Georgia inputs. Inspect encountered temperature codes and reconcile output counts. This check produces unpublished results; full Georgia preparation and approval are assessed separately.

In [49]:
def portability_source(**identity):
    matches = [
        item for item in downloads
        if all(item["request"].get(key) == value for key, value in identity.items())
    ]
    if len(matches) != 1:
        raise ValueError(f"Expected one selected Bronze source: {identity}")
    return matches[0]


georgia_portability = None
ga_ozone = process_ozone_source(
    portability_source(
        source="epa", dataset="hourly_ozone", year=georgia_profile["year"]
    ),
    georgia_profile, silver_context.environment_root,
)
ga_temperature = process_temperature_source(
    portability_source(
        source="noaa", dataset="hourly_weather", year=georgia_profile["year"],
        station_id=georgia_profile["weather_mapping"]["noaa_station_id"],
    ),
    georgia_profile, silver_context.environment_root,
)
ga_monitor = georgia_profile["ozone_monitor"]
ga_site_keys = {
    "State Code": ga_monitor["state_code"],
    "County Code": ga_monitor["county_code"],
    "Site Number": ga_monitor["site_number"],
}
ga_reference_keys = {
    "epa_sites": ga_site_keys,
    "epa_monitors": {
        **ga_site_keys,
        "Parameter Code": ga_monitor["parameter_code"],
        "POC": ga_monitor["poc"],
    },
    "noaa_stations": {
        "GHCN_ID": georgia_profile["weather_mapping"]["noaa_station_id"],
    },
}
ga_tables = {
    "ozone_observations": ga_ozone["observations"],
    "temperature_reports": ga_temperature["observations"],
}
for name, keys in ga_reference_keys.items():
    selected = reference_raw_tables[name]
    for column, value in keys.items():
        selected = selected.filter(F.col(column) == F.lit(value))
    ga_tables[name] = process_reference_records(
        selected, reference_sources[name], name, silver_context.environment_root
    )["reference"]

ga_cached = []
try:
    for table in ga_tables.values():
        if not table.is_cached:
            table.cache()
            ga_cached.append(table)
        table.count()

    ga_inputs, _ = inspect_hourly_inputs(ga_tables, georgia_profile)
    ga_input_counts = {
        row["dataset"]: row.asDict() for row in ga_inputs.collect()
    }
    ga_mapping, ga_mapping_evidence = build_yearly_mapping(
        ga_tables, georgia_profile, source_publication_id=None
    )
    ga_hourly_temperature = select_hourly_temperature(
        ga_tables["temperature_reports"],
        policy_version=georgia_profile["temperature_policy"]["version"],
    )
    ga_enriched, ga_enrichment_checks = enrich_hourly_ozone(
        ga_tables["ozone_observations"], ga_hourly_temperature, ga_mapping
    )
    ga_weather_counts = ga_hourly_temperature.agg(
        F.count("*").alias("hours"),
        F.coalesce(F.sum("qualifying_report_count"), F.lit(0)).alias("reports"),
    ).first().asDict()
    if (
        ga_weather_counts["hours"]
        != ga_input_counts["temperature_reports"]["clock_hours"]
        or ga_weather_counts["reports"]
        != ga_temperature["summary"]["after"]["records"]
        or ga_enrichment_checks["enriched_records"]
        != ga_ozone["summary"]["after"]["records"]
        or ga_enrichment_checks["matched_temperature"] == 0
    ):
        raise ValueError(
            "Georgia portability counts failed; inspect the source audits."
        )

    georgia_portability = {
        "status": "checked_unpublished",
        "profile": json.loads(json.dumps(georgia_profile)),
        "source_silver_publication_id": None,
        "source_versions": {
            "ozone": ga_ozone["summary"]["source_version_sha256"],
            "temperature": ga_temperature["summary"]["source_version_sha256"],
        },
        "source_dispositions": {
            "ozone": ga_ozone["summary"]["dispositions"],
            "temperature": ga_temperature["summary"]["dispositions"],
        },
        "inputs": ga_input_counts,
        "hourly_temperature": ga_weather_counts,
        "enrichment": ga_enrichment_checks,
        "mapping_evidence": ga_mapping_evidence,
    }
    print("Georgia portability:", georgia_portability["status"])
    print("Ozone dispositions:",
          georgia_portability["source_dispositions"]["ozone"])
    print("Temperature dispositions:",
          georgia_portability["source_dispositions"]["temperature"])
    print("Hourly temperature:", ga_weather_counts)
    print("Enrichment:", ga_enrichment_checks)
    print("Mapping review flags:", ga_mapping_evidence["review_flags"])

    ga_temperature["audit"].groupBy(
        F.col("raw_fields")["temperature_Source_Code"].alias("source_code"),
        F.col("raw_fields")["temperature_Quality_Code"].alias("quality_code"),
        F.col("raw_fields")["temperature_Report_Type"].alias("report_type"),
        F.col("raw_fields")["temperature_Measurement_Code"].alias("measurement_code"),
        "disposition",
    ).agg(F.sum("record_count").alias("source_records")).orderBy(
        F.desc("source_records")
    ).show(100, truncate=False)
finally:
    for table in ga_cached:
        table.unpersist(blocking=True)

26/11/09 23:17:16 WARN DAGScheduler: Broadcasting large task binary with size 1041.5 KiB
26/11/09 23:17:17 WARN DAGScheduler: Broadcasting large task binary with size 1094.9 KiB
26/11/09 23:17:17 WARN DAGScheduler: Broadcasting large task binary with size 1095.5 KiB
26/11/09 23:17:17 WARN DAGScheduler: Broadcasting large task binary with size 1091.6 KiB


Georgia portability: checked_unpublished
Ozone dispositions: {'eligible': 8526, 'outside_year': 5}
Temperature dispositions: {'eligible': 13010, 'missing_temperature': 2, 'rejected_source_quality': 15, 'unsupported_source_quality': 122}
Hourly temperature: {'hours': 8636, 'reports': 13010}
Enrichment: {'enriched_records': 8526, 'distinct_hour_keys': 8526, 'matched_temperature': 8404, 'missing_temperature': 122, 'input_ozone_records': 8526}
Mapping review flags: ['distance_requires_epa_datum_review', 'epa_monitors:non_wgs84_datum', 'epa_sites:non_wgs84_datum']


26/11/09 23:18:10 WARN DAGScheduler: Broadcasting large task binary with size 1052.8 KiB


+-----------+------------+-----------+----------------+--------------------------+--------------+
|source_code|quality_code|report_type|measurement_code|disposition               |source_records|
+-----------+------------+-----------+----------------+--------------------------+--------------+
|343        |5           |FM15       |NULL            |eligible                  |5553          |
|223        |1           |FM15       |NULL            |eligible                  |3057          |
|223        |1           |FM12       |NULL            |eligible                  |2857          |
|343        |5           |FM16       |NULL            |eligible                  |1119          |
|223        |1           |FM16       |NULL            |eligible                  |424           |
|999        |NULL        |FM15       |NULL            |unsupported_source_quality|121           |
|343        |7           |FM15       |NULL            |rejected_source_quality   |15            |
|NULL       |NULL   

26/11/09 23:18:12 WARN DAGScheduler: Broadcasting large task binary with size 1103.3 KiB


#### Hourly preparation baseline

Measure three runs of Alabama hourly temperature selection, ozone enrichment and their checks using fixed published source inputs. Materialise every output field and verify identical schemas and complete results. Record timings, input volumes, Spark settings and physical plans. Spark SQL caches are cleared between runs; operating-system caches are uncontrolled.

In [50]:
from contextlib import redirect_stdout
from io import StringIO


def hourly_result_probe(table):
    return table.select(F.to_json(
        F.struct(*[F.col(name) for name in table.columns]),
        options={"ignoreNullFields": "false"},
    ).alias("record_json"))


def hourly_result_digest(rows):
    return hashlib.sha256(
        "\n".join(sorted(row["record_json"] for row in rows)).encode("utf-8")
    ).hexdigest()


baseline_expected_table = prepared_silver["tables"]["ozone_enriched"]
baseline_expected_schema = baseline_expected_table.schema.json()
baseline_expected_digest = hourly_result_digest(
    hourly_result_probe(baseline_expected_table).collect()
)
baseline_runs = []
baseline_plans = []

for repetition in range(1, 4):
    spark.catalog.clearCache()
    started = time.perf_counter()

    benchmark_weather = select_hourly_temperature(
        hourly_tables["temperature_reports"],
        policy_version=hourly_rules["temperature_policy_version"],
    )
    benchmark_enriched, benchmark_checks = enrich_hourly_ozone(
        hourly_tables["ozone_observations"],
        benchmark_weather,
        prepared_silver["tables"]["yearly_mapping"],
    )
    benchmark_probe = hourly_result_probe(benchmark_enriched)
    benchmark_rows = benchmark_probe.collect()
    elapsed = time.perf_counter() - started

    if (
        benchmark_enriched.schema.json() != baseline_expected_schema
        or hourly_result_digest(benchmark_rows) != baseline_expected_digest
    ):
        raise ValueError(
            "Benchmark output differs from the prepared ozone table."
        )

    plan_buffer = StringIO()
    with redirect_stdout(plan_buffer):
        benchmark_probe.explain(mode="formatted")
    baseline_plans.append(plan_buffer.getvalue())
    baseline_runs.append({
        "repetition": repetition,
        "seconds": elapsed,
        "rows": len(benchmark_rows),
    })
    print(
        f"Run {repetition}: {elapsed:.3f} seconds; "
        f"{len(benchmark_rows)} rows"
    )

saved_pipeline = json.loads(
    (project_dir / "notebooks" / "pipeline.ipynb").read_text(encoding="utf-8")
)
hourly_performance_baseline = {
    "status": "recorded",
    "scope": (
        "hourly temperature selection, ozone enrichment, "
        "checks and full JSON collection"
    ),
    "source_silver_publication_id": hourly_manifest["publication_id"],
    "saved_notebook_code_sha256": notebook_code_sha256(saved_pipeline),
    "profile_sha256": silver_json_sha256(hourly_profile),
    "rules": prepared_silver["rules"],
    "spark_version": spark.version,
    "master": spark.sparkContext.master,
    "spark_application_id": spark.sparkContext.applicationId,
    "spark_ui_url": spark.sparkContext.uiWebUrl,
    "runtime_settings": {
        name: spark.sparkContext.getConf().get(name, None)
        for name in (
            "spark.executor.cores", "spark.executor.memory",
            "spark.driver.memory", "spark.cores.max",
        )
    },
    "cache_condition": (
        "Spark SQL caches cleared before each run; "
        "OS cache unmanaged; JVM already running"
    ),
    "input_volumes": {
        name: {
            "records": hourly_manifest["datasets"][name]["row_count"],
            "stored_parquet_bytes": hourly_manifest["datasets"][name]["bytes"],
        }
        for name in ("ozone_observations", "temperature_reports")
    },
    "sql_settings": {
        name: spark.conf.get(name)
        for name in (
            "spark.sql.session.timeZone",
            "spark.sql.shuffle.partitions",
            "spark.sql.adaptive.enabled",
            "spark.sql.autoBroadcastJoinThreshold",
        )
    },
    "runs": baseline_runs,
    "median_seconds": sorted(run["seconds"] for run in baseline_runs)[1],
    "result_sha256": baseline_expected_digest,
    "schema_json": baseline_expected_schema,
    "physical_plans": baseline_plans,
}
hourly_baseline_path = (
    silver_context.reports_dir / "hourly_performance_baseline.json"
)
write_json(hourly_baseline_path, hourly_performance_baseline)

print(
    "Median seconds:",
    round(hourly_performance_baseline["median_seconds"], 3)
)
print("All three runs reproduced the complete prepared output.")
print("Baseline report:", hourly_baseline_path)

Run 1: 2.977 seconds; 5071 rows
Run 2: 2.906 seconds; 5071 rows
Run 3: 2.905 seconds; 5071 rows
Median seconds: 2.906
All three runs reproduced the complete prepared output.
Baseline report: /home/ubuntu/dat535-final-project-data/dev/runs/26a8250b3c0a422dab4875f5e49d86d8/reports/hourly_performance_baseline.json


### 9.10 Alabama suitability and completeness decision

The McAdory–Alabaster pairing is accepted for analysis of available 2025 observations, subject to successful publication validation. Eligible ozone covers 5,071 of 8,760 calendar hours (57.89%), and 4,836 eligible ozone hours have temperature (95.37%).

Coverage is reported for all twelve months. Missing temperature remains null, and eligible ozone is retained. Historical operating schedules are unknown, so gaps are not classified as seasonal closure or instrument downtime. The weather station is approximately 26.73 km from the ozone site; results describe this selected pair rather than statewide conditions.

The reviewed mapping has no metadata review flags. If flags appear on a later run, acceptance stops for review. The decision records the mapping flags, profile, processing rules, source publication and coverage counts.

Mapping, key, lineage, reconciliation and publication failures block publication. Incomplete observations are reported explicitly under the accepted completeness policy.

In [51]:
alabama_silver_acceptance = None

# Clear any previous approval before checking the current mapping.
decision_mapping = prepared_silver["mapping_evidence"]
decision_mapping["approval_status"] = "pending"
decision_mapping.pop("approval_decision_sha256", None)

if (
    hourly_profile["region"] != "Alabama"
    or hourly_profile["year"] != 2025
    or prepared_silver["mapping_evidence"]["monitor_id"]
    != "01-073-1005-44201-1"
    or prepared_silver["mapping_evidence"]["station_id"]
    != "USW00053864"
):
    raise ValueError(
        "This acceptance decision applies to the reviewed Alabama pairing."
    )

if prepared_silver["checks"]["status"] != "passed":
    raise ValueError("In-memory preparation checks must pass.")

if hourly_fixture_checks["status"] != "passed":
    raise ValueError(
        "Run the hourly integration fixtures before recording acceptance."
    )

decision_mapping_flags = sorted(set(decision_mapping["review_flags"]))
if decision_mapping_flags:
    raise ValueError(
        "Mapping metadata requires review before acceptance: "
        + ", ".join(decision_mapping_flags)
    )

decision_coverage = prepared_silver["checks"]["coverage"]
alabama_silver_acceptance = {
    "decision_version": "available-observations-v1",
    "status": "accepted_with_limitations",
    "region": hourly_profile["region"],
    "year": hourly_profile["year"],
    "profile_sha256": silver_json_sha256(prepared_silver["profile"]),
    "source_silver_publication_id": (
        prepared_silver["source_silver_publication_id"]
    ),
    "mapping_id": prepared_silver["mapping_evidence"]["mapping_id"],
    "mapping_review_flags": decision_mapping_flags,
    "rules_sha256": silver_json_sha256(prepared_silver["rules"]),
    "coverage": {
        name: decision_coverage[name]
        for name in (
            "calendar_hours",
            "eligible_ozone_hours",
            "eligible_temperature_hours",
            "shared_hours",
        )
    },
    "critical_failures": [
        "Incompatible or unverifiable source publication.",
        "Missing, ambiguous or invalid required mapping.",
        "Mapping metadata flags requiring review.",
        "Unusable or repeated output keys, or row multiplication.",
        "Unreconciled source audits, calendar counts or gap intervals.",
        "Failed integration fixtures or publication read-back checks.",
    ],
    "reportable_limitations": [
        "Missing and excluded observations remain visible across all twelve months.",
        "Missing temperature remains null; eligible ozone is retained.",
        "Historical operating schedules are unknown.",
        (
            "Monitor-station distance: "
            f"{prepared_silver['mapping_evidence']['distance_km']:.2f} km."
        ),
        "Results describe the selected monitor-station pair and available observations.",
        "Observed gaps do not establish seasonal operation or instrument downtime.",
    ],
    "publication_condition": (
        "All publication and read-back checks must pass."
    ),
}

alabama_acceptance_path = (
    silver_context.reports_dir / "alabama_silver_acceptance.json"
)
write_json(alabama_acceptance_path, alabama_silver_acceptance)

prepared_silver["mapping_evidence"] = {
    **prepared_silver["mapping_evidence"],
    "approval_status": "accepted_with_limitations",
    "approval_decision_sha256": silver_json_sha256(
        alabama_silver_acceptance
    ),
}

print("Alabama decision:", alabama_silver_acceptance["status"])
print("Eligible ozone hours:", decision_coverage["eligible_ozone_hours"])
print("Shared hours:", decision_coverage["shared_hours"])
print("Publication remains pending read-back validation.")
print("Decision report:", alabama_acceptance_path)

Alabama decision: accepted_with_limitations
Eligible ozone hours: 5071
Shared hours: 4836
Publication remains pending read-back validation.
Decision report: /home/ubuntu/dat535-final-project-data/dev/runs/26a8250b3c0a422dab4875f5e49d86d8/reports/alabama_silver_acceptance.json


### 9.11 Publish and reload hourly Silver

Publish the yearly mapping, hourly temperature, enriched ozone, calendar coverage, monthly coverage and gap intervals as one versioned bundle. The manifest records the source Silver publication, processing rules, acceptance decision, schemas and validation results.

#### 9.11a Stage hourly outputs

Check that acceptance matches the prepared profile, mapping, rules and source publication. Verify the source publication and fixture code, then write the six tables to staging with file checksums. Staged outputs remain unpublished.

In [52]:
hourly_output_keys = {
    "yearly_mapping": ["monitor_id", "valid_from_utc"],
    "temperature_hourly": ["station_id", "hour_start_utc"],
    "ozone_enriched": ["monitor_id", "hour_start_utc"],
    "hourly_coverage": ["monitor_id", "hour_start_utc"],
    "monthly_coverage": ["monitor_id", "month_start_utc"],
    "hourly_gaps": ["monitor_id", "gap_kind", "gap_start_utc"],
}


def hourly_code_hashes():
    return {
        name: notebook_code_sha256(json.loads(
            (project_dir / "notebooks" / name).read_text(encoding="utf-8")
        ))
        for name in ("pipeline.ipynb", "silver_checks.ipynb")
    }


def check_hourly_evidence(evidence):
    """Require acceptance bound to the prepared profile, rules and source."""
    acceptance = evidence["acceptance"]
    mapping = evidence["mapping"]
    profile = evidence["profile"]
    coverage = evidence["checks"]["coverage"]
    if (
        not isinstance(acceptance, dict)
        or acceptance.get("status") != "accepted_with_limitations"
        or evidence["checks"]["status"] != "passed"
        or evidence["fixtures"]["status"] != "passed"
        or evidence["rules"]["version"] != "hourly-integration-v1"
        or evidence["rules"]["timezone"] != "UTC"
        or mapping["structural_status"] != "passed"
        or mapping["approval_status"] != "accepted_with_limitations"
        or mapping["review_flags"] != []
        or acceptance["mapping_review_flags"] != []
        or acceptance["region"] != profile["region"]
        or acceptance["year"] != profile["year"]
        or acceptance["profile_sha256"] != silver_json_sha256(profile)
        or mapping["profile_sha256"] != silver_json_sha256(profile)
        or acceptance["rules_sha256"] != silver_json_sha256(evidence["rules"])
        or acceptance["mapping_id"] != mapping["mapping_id"]
        or mapping["approval_decision_sha256"] != silver_json_sha256(acceptance)
        or acceptance["source_silver_publication_id"]
        != evidence["source"]["publication_id"]
        or mapping["source_silver_publication_id"]
        != evidence["source"]["publication_id"]
        or acceptance["coverage"] != {
            name: coverage[name]
            for name in (
                "calendar_hours", "eligible_ozone_hours",
                "eligible_temperature_hours", "shared_hours",
            )
        }
    ):
        raise ValueError("Hourly publication evidence is incomplete or inconsistent.")


def hourly_expected_rows(evidence):
    checks = evidence["checks"]
    return {
        "yearly_mapping": 1,
        "temperature_hourly": checks["temperature"]["hourly_records"],
        "ozone_enriched": checks["enrichment"]["enriched_records"],
        "hourly_coverage": checks["coverage"]["calendar_hours"],
        "monthly_coverage": checks["monthly"]["months"],
        "hourly_gaps": sum(
            item["gap_count"] for item in checks["gaps"].values()
        ),
    }


hourly_publication = None
hourly_loaded_tables = None
hourly_validation = None
hourly_stage = None

if (
    prepared_silver["status"] != "prepared_unpublished"
    or prepared_silver["profile"] != hourly_manifest["profile"]
    or prepared_silver["source_silver_publication_id"]
    != hourly_manifest["publication_id"]
    or set(prepared_silver["tables"]) != set(hourly_output_keys)
):
    raise ValueError("Prepare hourly Silver from the selected source publication.")

hourly_saved_code = hourly_code_hashes()
if (
    hourly_fixture_checks["status"] != "passed"
    or hourly_saved_code["silver_checks.ipynb"]
    != hourly_fixture_checks["companion_code_sha256"]
):
    raise ValueError("Run the hourly fixtures against the saved companion code.")

hourly_evidence = json.loads(json.dumps({
    "profile": prepared_silver["profile"],
    "rules": prepared_silver["rules"],
    "mapping": prepared_silver["mapping_evidence"],
    "acceptance": alabama_silver_acceptance,
    "checks": prepared_silver["checks"],
    "fixtures": hourly_fixture_checks,
    "source": {
        "publication_id": hourly_manifest["publication_id"],
        "manifest_path": Path(source_silver_publication["manifest_path"])
        .relative_to(silver_context.environment_root).as_posix(),
        "manifest_sha256": source_silver_publication["manifest_sha256"],
        "rule_versions": hourly_manifest["rule_versions"],
    },
    "provenance": {
        **capture_provenance(silver_context),
        "saved_notebook_code_sha256": hourly_saved_code,
        "supporting_files_sha256": {
            path.relative_to(project_dir).as_posix(): file_sha256(path)
            for path in supporting_paths
        },
        "python": sys.version.split()[0],
        "spark": spark.version,
        "master": spark.sparkContext.master,
        "timezone": spark.conf.get("spark.sql.session.timeZone"),
    },
}, allow_nan=False))
check_hourly_evidence(hourly_evidence)

verified_parent, _, _ = load_source_silver(
    source_silver_publication["manifest_path"], silver_context,
    expected_manifest_sha256=hourly_evidence["source"]["manifest_sha256"],
    profile=hourly_evidence["profile"],
    rule_versions=hourly_evidence["source"]["rule_versions"],
)
if verified_parent != hourly_manifest:
    raise ValueError("The selected source publication changed.")

hourly_row_counts = hourly_expected_rows(hourly_evidence)
hourly_contracts = {
    name: {
        "schema_version": f"{name}-v1",
        "columns": dict(table.dtypes),
        "column_order": table.columns,
        "key": hourly_output_keys[name],
        "expected_rows": hourly_row_counts[name],
    }
    for name, table in prepared_silver["tables"].items()
}

hourly_stage = stage_source_silver(
    prepared_silver["tables"], hourly_contracts, silver_context,
)
print("Hourly status: staged, unpublished")

Staged yearly_mapping: 1 rows, 1 Parquet file(s).
Staged temperature_hourly: 8514 rows, 1 Parquet file(s).
Staged ozone_enriched: 5071 rows, 1 Parquet file(s).
Staged hourly_coverage: 8760 rows, 1 Parquet file(s).
Staged monthly_coverage: 12 rows, 1 Parquet file(s).
Staged hourly_gaps: 143 rows, 1 Parquet file(s).
Hourly status: staged, unpublished


#### 9.11b Validate staged hourly Silver

Verify the Parquet inventory, checksums, schemas, record counts and unique keys. Decode every output column and compare the persisted tables with the prepared tables, including null values, quality flags and source lineage.

Record successful validation against the staged bundle, contracts and acceptance evidence. The bundle remains unpublished.

In [53]:
def read_hourly_files(bundle, datasets, contracts, *, expected=None):
    """Verify every stored field, row count and key before returning tables."""
    if set(datasets) != set(contracts) or set(contracts) != set(hourly_output_keys):
        raise ValueError("The hourly bundle must contain all six datasets.")
    if spark.conf.get("spark.sql.session.timeZone") != "UTC":
        raise ValueError("Hourly Silver requires UTC.")

    paths = verify_silver_file_inventory(bundle, datasets)
    tables, checks = {}, {}

    for name in sorted(contracts):
        contract, info = contracts[name], datasets[name]
        table = (
            spark.read.option("mergeSchema", "true")
            .option("ignoreCorruptFiles", "false")
            .option("ignoreMissingFiles", "false")
            .parquet(*paths[name])
        )
        if (
            contract["schema_version"] != f"{name}-v1"
            or contract["key"] != hourly_output_keys[name]
            or table.columns != contract["column_order"]
            or table.columns != [field["name"] for field in info["schema"]["fields"]]
            or dict(table.dtypes) != contract["columns"]
        ):
            raise ValueError(f"{name}: schema or key contract differs.")

        table.cache()
        try:
            # Decode all columns; a count alone may read only Parquet metadata.
            probe = table.select(F.to_json(
                F.struct(*[F.col(column) for column in table.columns]),
                options={"ignoreNullFields": "false"},
            ).alias("record_json")).agg(
                F.count("*").alias("records"),
                F.sum(F.length("record_json")).alias("decoded_characters"),
            ).first()

            count = check_hourly_keys(table, contract["key"], name)
            if (
                count != probe["records"]
                or count != info["row_count"]
                or count != contract["expected_rows"]
            ):
                raise ValueError(f"{name}: record counts differ.")

            if expected is not None:
                reference = expected[name].select(*table.columns)
                if (
                    dict(reference.dtypes) != dict(table.dtypes)
                    or reference.exceptAll(table).limit(1).count()
                    or table.exceptAll(reference).limit(1).count()
                ):
                    raise ValueError(
                        f"{name}: persisted values differ from preparation."
                    )

            checks[name] = {
                "status": "passed",
                "records": count,
                "parquet_files": len(paths[name]),
                "complete_values_match": expected is not None,
            }
            tables[name] = table
            print(f"Verified {name}: {count} rows.")
        finally:
            table.unpersist(blocking=True)

    return tables, checks


hourly_validation = None
check_hourly_evidence(hourly_evidence)
if (
    hourly_stage["run_id"] != silver_context.run_id
    or hourly_stage["environment"] != silver_context.environment
    or hourly_stage["contract_sha256"] != silver_json_sha256(hourly_contracts)
    or {name: item["expected_rows"] for name, item in hourly_contracts.items()}
    != hourly_expected_rows(hourly_evidence)
):
    raise ValueError("Stage, contracts and preparation evidence do not match.")

hourly_readback, hourly_dataset_checks = read_hourly_files(
    silver_child_path(
        silver_context.environment_root,
        hourly_stage["bundle_path"],
        must_exist=True,
    ),
    hourly_stage["datasets"],
    hourly_contracts,
    expected=prepared_silver["tables"],
)

hourly_validation = {
    "status": "validated",
    "validated_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_id": silver_context.run_id,
    "stage_sha256": silver_json_sha256(hourly_stage),
    "contract_sha256": silver_json_sha256(hourly_contracts),
    "evidence_sha256": silver_json_sha256(hourly_evidence),
    "datasets": hourly_dataset_checks,
}
write_json(
    silver_context.reports_dir / "hourly_silver_validation.json",
    hourly_validation,
)
print("Hourly status: validated, unpublished")
print("Complete table comparisons passed:", len(hourly_dataset_checks))

Verified hourly_coverage: 8760 rows.
Verified hourly_gaps: 143 rows.
Verified monthly_coverage: 12 rows.


Verified ozone_enriched: 5071 rows.
Verified temperature_hourly: 8514 rows.
Verified yearly_mapping: 1 rows.
Hourly status: validated, unpublished
Complete table comparisons passed: 6


#### 9.11c Publish the hourly bundle

Require matching validation and acceptance evidence before publication. Recheck file checksums, move the staged bundle into its versioned location, and write the manifest last.

A publication lock prevents concurrent writes. Existing publications are preserved, and an identical attempt reuses its existing manifest. Data left by an incomplete attempt is not overwritten.

In [54]:
def publish_hourly_silver(stage, validation, contracts, evidence, context):
    """Commit a validated hourly bundle by writing its manifest last."""
    stage, validation, contracts, evidence = json.loads(json.dumps(
        [stage, validation, contracts, evidence], allow_nan=False,
    ))
    check_hourly_evidence(evidence)
    stage_hash = silver_json_sha256(stage)
    contract_hash = silver_json_sha256(contracts)
    evidence_hash = silver_json_sha256(evidence)
    root = Path(context.environment_root).resolve()

    if (
        stage["status"] != "staged"
        or stage["run_id"] != context.run_id
        or stage["environment"] != context.environment
        or Path(stage["environment_root"]).resolve() != root
        or stage["contract_sha256"] != contract_hash
        or validation["status"] != "validated"
        or validation["run_id"] != context.run_id
        or validation["stage_sha256"] != stage_hash
        or validation["contract_sha256"] != contract_hash
        or validation["evidence_sha256"] != evidence_hash
        or set(stage["datasets"]) != set(hourly_output_keys)
        or set(contracts) != set(hourly_output_keys)
        or set(validation["datasets"]) != set(hourly_output_keys)
        or {name: item["expected_rows"] for name, item in contracts.items()}
        != hourly_expected_rows(evidence)
    ):
        raise ValueError(
            "Publication requires matching stage, validation and evidence."
        )

    for name, info in stage["datasets"].items():
        check = validation["datasets"][name]
        if (
            check["status"] != "passed"
            or check["complete_values_match"] is not True
            or check["records"] != info["row_count"]
            or check["records"] != contracts[name]["expected_rows"]
            or check["parquet_files"] != len(info["files"])
        ):
            raise ValueError(
                f"{name}: complete read-back validation is required."
            )

    region = evidence["profile"]["region"].strip().lower().replace(" ", "_")
    year = evidence["profile"]["year"]
    if (
        not region
        or any(c not in "abcdefghijklmnopqrstuvwxyz0123456789_-" for c in region)
        or type(year) is not int or not 1 <= year <= 9999
    ):
        raise ValueError("Invalid region or year.")

    staged = silver_child_path(root, stage["bundle_path"])
    staging_root = Path(context.staging_root).resolve()
    if (
        not staging_root.is_relative_to(root / "staging")
        or staged == staging_root
        or not staged.is_relative_to(staging_root)
    ):
        raise ValueError(
            "The bundle must be inside this run's staging directory."
        )

    publication_id = silver_json_sha256({
        "stage_sha256": stage_hash,
        "evidence_sha256": evidence_hash,
    })
    suffix = f"silver/hourly/{region}/{year}/{publication_id}"
    final_bundle = silver_child_path(root, f"data/{suffix}")
    manifest_path = silver_child_path(root, f"published/{suffix}/manifest.json")
    publication_root = silver_child_path(root, "published/silver/hourly")

    identity = {
        "manifest_version": 1,
        "kind": "hourly_silver",
        "status": "published",
        "publication_id": publication_id,
        "run_id": context.run_id,
        "environment": context.environment,
        "bundle_path": final_bundle.relative_to(root).as_posix(),
        "stage_sha256": stage_hash,
        "contract_sha256": contract_hash,
        "evidence_sha256": evidence_hash,
        "contracts": contracts,
        "datasets": stage["datasets"],
        "evidence": evidence,
        "validation": validation,
        "storage": {
            key: stage[key]
            for key in (
                "format", "compression", "write_partitions",
                "partition_columns",
            )
        },
    }

    publication_root.mkdir(parents=True, exist_ok=True)
    lock = publication_root / ".publish.lock"
    try:
        lock.mkdir()
    except FileExistsError as error:
        raise RuntimeError(
            f"Publication lock already exists: {lock}"
        ) from error

    try:
        if manifest_path.exists():
            existing = json.loads(manifest_path.read_text(encoding="utf-8"))
            if {key: existing.get(key) for key in identity} != identity:
                raise ValueError(
                    "The existing publication has different metadata."
                )
            verify_silver_file_inventory(
                final_bundle, existing["datasets"]
            )
            outcome, manifest = "reused", existing
        else:
            if final_bundle.exists() or final_bundle.is_symlink():
                raise FileExistsError(
                    f"An incomplete publication exists at {final_bundle}. "
                    "Inspect it, then stage a new attempt."
                )
            verify_silver_file_inventory(staged, stage["datasets"])
            final_bundle.parent.mkdir(parents=True, exist_ok=True)
            if staged.stat().st_dev != final_bundle.parent.stat().st_dev:
                raise ValueError(
                    "Staging and publication require the same filesystem."
                )

            manifest = {
                **identity,
                "published_at_utc": datetime.now(timezone.utc).isoformat(),
            }
            staged.rename(final_bundle)
            write_json(manifest_path, manifest)
            outcome = "published"

        return {
            "outcome": outcome,
            "manifest_path": str(manifest_path),
            "manifest_sha256": file_sha256(manifest_path),
            "manifest": manifest,
        }
    finally:
        lock.rmdir()


hourly_publication = None
if (
    hourly_code_hashes()
    != hourly_evidence["provenance"]["saved_notebook_code_sha256"]
):
    raise ValueError(
        "Saved code changed after staging. Rerun 9.11a and 9.11b."
    )

# Failed validation must be rejected before publication.
try:
    publish_hourly_silver(
        hourly_stage,
        {**hourly_validation, "status": "failed"},
        hourly_contracts,
        hourly_evidence,
        silver_context,
    )
except ValueError:
    print("Failed-validation guard: passed")
else:
    raise AssertionError("Failed validation was incorrectly accepted.")

hourly_publication = publish_hourly_silver(
    hourly_stage, hourly_validation,
    hourly_contracts, hourly_evidence, silver_context,
)
hourly_retry = publish_hourly_silver(
    hourly_stage, hourly_validation,
    hourly_contracts, hourly_evidence, silver_context,
)
if (
    hourly_retry["outcome"] != "reused"
    or hourly_retry["manifest_sha256"]
    != hourly_publication["manifest_sha256"]
):
    raise AssertionError(
        "The identical publication attempt was not reused."
    )

hourly_readback = None
print("Publication outcome:", hourly_publication["outcome"])
print("Identical-attempt reuse: passed")
print("Manifest:", hourly_publication["manifest_path"])
print("Manifest SHA-256:", hourly_publication["manifest_sha256"])

Failed-validation guard: passed
Publication outcome: published
Identical-attempt reuse: passed
Manifest: /home/ubuntu/dat535-final-project-data/dev/published/silver/hourly/alabama/2025/774ac0c068e26fa310e48fd468c642884a588a569fa15f441f0f9cff9c8ba20a/manifest.json
Manifest SHA-256: ec77e802b1b916aabcd7b8e8ac6f6b6438ebb3618834ec2d41da20758a675a15


#### 9.11d Reload published hourly Silver

Load the explicitly selected publication using its manifest path and checksum. Verify its identity, acceptance evidence, source Silver dependency, file checksums, schemas, counts and keys.

Record successful preparation only after the published tables have been reloaded and checked. The completion report identifies the exact publication available to Gold.

In [55]:
def load_hourly_silver(
    manifest_path, context, *, expected_manifest_sha256, profile,
):
    """Load an explicitly selected hourly publication for downstream use."""
    root = Path(context.environment_root).resolve()
    path = Path(manifest_path).resolve(strict=True)
    if (
        path.name != "manifest.json"
        or not path.is_relative_to(root / "published/silver/hourly")
        or file_sha256(path) != expected_manifest_sha256
    ):
        raise ValueError("The hourly manifest path or checksum differs.")

    manifest = json.loads(path.read_text(encoding="utf-8"))
    evidence = manifest["evidence"]
    validation = manifest["validation"]
    contracts = manifest["contracts"]
    check_hourly_evidence(evidence)

    expected_id = silver_json_sha256({
        "stage_sha256": manifest["stage_sha256"],
        "evidence_sha256": silver_json_sha256(evidence),
    })
    if (
        manifest["manifest_version"] != 1
        or manifest["kind"] != "hourly_silver"
        or manifest["status"] != "published"
        or manifest["environment"] != context.environment
        or evidence["profile"] != profile
        or manifest["publication_id"] != expected_id
        or manifest["evidence_sha256"] != silver_json_sha256(evidence)
        or manifest["contract_sha256"] != silver_json_sha256(contracts)
        or manifest["storage"]["format"] != "parquet"
        or validation["status"] != "validated"
        or validation["run_id"] != manifest["run_id"]
        or validation["stage_sha256"] != manifest["stage_sha256"]
        or validation["evidence_sha256"] != manifest["evidence_sha256"]
        or validation["contract_sha256"] != manifest["contract_sha256"]
        or set(validation["datasets"]) != set(hourly_output_keys)
        or {name: item["expected_rows"] for name, item in contracts.items()}
        != hourly_expected_rows(evidence)
    ):
        raise ValueError(
            "The hourly publication metadata is inconsistent."
        )

    region = profile["region"].strip().lower().replace(" ", "_")
    suffix = f"silver/hourly/{region}/{profile['year']}/{expected_id}"
    if (
        manifest["bundle_path"] != f"data/{suffix}"
        or path != silver_child_path(
            root, f"published/{suffix}/manifest.json"
        )
    ):
        raise ValueError(
            "The publication is in an unexpected location."
        )

    source = evidence["source"]
    parent, _, _ = load_source_silver(
        silver_child_path(
            root, source["manifest_path"], must_exist=True
        ),
        context,
        expected_manifest_sha256=source["manifest_sha256"],
        profile=profile,
        rule_versions=source["rule_versions"],
    )
    if parent["publication_id"] != source["publication_id"]:
        raise ValueError(
            "The source Silver publication identity differs."
        )

    tables, checks = read_hourly_files(
        silver_child_path(
            root, manifest["bundle_path"], must_exist=True
        ),
        manifest["datasets"],
        contracts,
    )
    for name, actual in checks.items():
        recorded = validation["datasets"][name]
        if (
            recorded["status"] != "passed"
            or recorded["complete_values_match"] is not True
            or recorded["records"] != actual["records"]
            or recorded["parquet_files"] != actual["parquet_files"]
        ):
            raise ValueError(
                f"{name}: published validation evidence differs."
            )

    return manifest, tables, checks


hourly_loaded_manifest = None
hourly_loaded_tables = None
hourly_silver_completion = None

(
    hourly_loaded_manifest,
    hourly_loaded_tables,
    hourly_load_checks,
) = load_hourly_silver(
    hourly_publication["manifest_path"],
    silver_context,
    expected_manifest_sha256=hourly_publication["manifest_sha256"],
    profile=hourly_profile,
)

hourly_silver_completion = {
    "status": "success",
    "readiness": "prepared",
    "completed_at_utc": datetime.now(timezone.utc).isoformat(),
    "run_id": silver_context.run_id,
    "environment": silver_context.environment,
    "publication_id": hourly_loaded_manifest["publication_id"],
    "manifest_path": Path(hourly_publication["manifest_path"])
    .relative_to(silver_context.environment_root).as_posix(),
    "manifest_sha256": hourly_publication["manifest_sha256"],
    "source_silver_publication_id": (
        hourly_loaded_manifest["evidence"]["source"]["publication_id"]
    ),
    "datasets": {
        name: check["records"]
        for name, check in hourly_load_checks.items()
    },
}
hourly_completion_path = (
    silver_context.reports_dir / "hourly_silver_completion.json"
)
write_json(hourly_completion_path, hourly_silver_completion)

spark.createDataFrame(
    sorted(hourly_silver_completion["datasets"].items()),
    "dataset STRING, records LONG",
).show(truncate=False)

print("Hourly Silver status: prepared")
print("Published datasets reloaded:", len(hourly_loaded_tables))
print("Completion report:", hourly_completion_path)
print("Gold input manifest:", hourly_publication["manifest_path"])
print(
    "Gold input manifest SHA-256:",
    hourly_publication["manifest_sha256"],
)

Verified hourly_coverage: 8760 rows.
Verified hourly_gaps: 143 rows.
Verified monthly_coverage: 12 rows.
Verified ozone_enriched: 5071 rows.
Verified temperature_hourly: 8514 rows.
Verified yearly_mapping: 1 rows.
+------------------+-------+
|dataset           |records|
+------------------+-------+
|hourly_coverage   |8760   |
|hourly_gaps       |143    |
|monthly_coverage  |12     |
|ozone_enriched    |5071   |
|temperature_hourly|8514   |
|yearly_mapping    |1      |
+------------------+-------+

Hourly Silver status: prepared
Published datasets reloaded: 6
Completion report: /home/ubuntu/dat535-final-project-data/dev/runs/26a8250b3c0a422dab4875f5e49d86d8/reports/hourly_silver_completion.json
Gold input manifest: /home/ubuntu/dat535-final-project-data/dev/published/silver/hourly/alabama/2025/774ac0c068e26fa310e48fd468c642884a588a569fa15f441f0f9cff9c8ba20a/manifest.json
Gold input manifest SHA-256: ec77e802b1b916aabcd7b8e8ac6f6b6438ebb3618834ec2d41da20758a675a15
